# Math-solver benchmark (SymPy / SmartSolver / StepSolver) - re-runnable

Run this notebook top to bottom whenever a new version of any engine appears. It rebuilds isolated environments, runs the **same 72 test cases, the same judge, the same derivation rubric, the same engineering probes and the same weighted scoring**, and produces the same artefacts as the original benchmark (`summary.md`, `failure_analysis.md`, `raw_results.json/csv`, `scores.json`, `engineering_results.json`, `reference_validation.json`, `reproducibility_check.json`) plus `changes_vs_baseline.md`, then offers everything as one zip.

**How to use**
1. Edit the configuration cell (engine versions / git refs). Leave the defaults to benchmark the *latest* code; use the pinned values in the comment to reproduce the original run.
2. *Runtime -> Run all.* Typical time: 8-12 min (add ~5 min per engine if you enable their own test suites).
3. Read the summary printed at the end; download the zip from the last cell.

**What is and is not automatic**
- Everything is computed by code from raw engine output; the reports contain no hand-written narrative.
- Derivation scores = automated rubric + a *model-assigned* manual overlay written for the original outputs. The overlay is applied only to a derivation whose output is byte-identical to the one reviewed (fingerprint); changed outputs fall back to the automated score. Human audit is still recommended.
- Timings depend on the Colab machine. Compare latencies only between runs on the same runtime type.
- If an engine's API changed (method renamed/removed), the adapter reports *unsupported* (SmartSolver) or an infrastructure error (import failure) - it never substitutes another call. Update the adapter cell if you want to follow an API change.

In [ ]:
#@title 1. Configuration { display-mode: "form" }
# ---- Engine versions ------------------------------------------------------------------------------------------
# Original (baseline) run:  SYMPY_SPEC="sympy==1.14.0"
#   SmartSolver REF f0d46a1089366f8e14f73de9ba811d8e4c1471b0   StepSolver REF bd8ea4ffdbe49277d649017cb6a31bf05b376bd8
SYMPY_SPEC = "sympy"                      #@param {type:"string"}   # "sympy" = latest release, or e.g. "sympy==1.14.0"
PIN_SYMPY_IN_CANDIDATES = True            #@param {type:"boolean"}  # give SmartSolver/StepSolver the same SymPy as the standalone engine (falls back to their own resolution if they conflict)
SMARTSOLVER_URL = "https://github.com/mngadilinda/SmartSolver"      #@param {type:"string"}
SMARTSOLVER_REF = "HEAD"                  #@param {type:"string"}   # branch, tag or commit; HEAD = latest default branch
STEPSOLVER_URL = "https://github.com/DavideGalilei/stepsolver"      #@param {type:"string"}
STEPSOLVER_REF = "HEAD"                   #@param {type:"string"}

# ---- Judge (kept fixed so the same answers are judged the same way across engine versions) ---------------------
JUDGE_SYMPY_SPEC = "sympy==1.14.0"        #@param {type:"string"}

# ---- Protocol ---------------------------------------------------------------------------------------------------
REPEATS = 5                               #@param {type:"integer"}  # warm, cache-cleared repeats per answered case
TIMEOUT_S = 20                            #@param {type:"integer"}  # per-case timeout (worker is killed and restarted)
RUN_TWICE = True                          #@param {type:"boolean"}  # second full run -> reproducibility check
RUN_OWN_TEST_SUITES = False               #@param {type:"boolean"}  # also run each candidate's own pytest suite (slow; feeds the maintainability score)
COMPARE_WITH_BASELINE = True              #@param {type:"boolean"}
PYTHON_VERSION = "3.12"                   #@param {type:"string"}
BENCH_DIR = "/content/math_solver_benchmark"   #@param {type:"string"}

In [ ]:
#@title 2. Tooling (uv + git) { display-mode: "form" }
import base64, json, os, pathlib, platform, re, shutil, subprocess, sys, time

ROOT = pathlib.Path(BENCH_DIR)
ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)

def sh(cmd, check=True, timeout=None, cwd=None):
    r = subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True, text=True, timeout=timeout, cwd=cwd)
    if check and r.returncode:
        raise RuntimeError("command failed (%d): %s\n%s" % (r.returncode, cmd, (r.stdout + r.stderr)[-1500:]))
    return r

def run_stream(cmd, cwd=None, check=True):
    """Run a command and stream its output into the notebook."""
    p = subprocess.Popen(cmd, shell=isinstance(cmd, str), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=cwd)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    if check and p.returncode:
        raise RuntimeError("command failed (%d): %s" % (p.returncode, cmd))
    return p.returncode

def W(rel, text):
    p = ROOT / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(text, encoding="utf-8")

if not shutil.which("uv"):
    sh([sys.executable, "-m", "pip", "install", "-q", "uv"])
UV = shutil.which("uv") or "uv"
print("uv:", sh([UV, "--version"]).stdout.strip(), "| git:", sh("git --version").stdout.strip(), "| notebook python:", platform.python_version())

## Benchmark source
The next four cells write the benchmark code to disk (collapsed by default; click *Show code* to read or edit them - e.g. to update an adapter after an API change). Nothing is executed here.

In [ ]:
#@title 3a. Benchmark source: dataset generator + docs + data { display-mode: "form" }
W('tools/make_cases.py', r'''#!/usr/bin/env python3
"""Generate data/cases.json (the machine-readable benchmark dataset).

IDs and expected results are exactly those of the task specification.
Typographical omissions in the specification's *display* strings (e.g. "2xsin(x)")
are restored with explicit '*' in the machine-readable `expected.expr` fields; the
original specification text is preserved verbatim in `spec_expected`.

All comparison rules, tolerances and form checks below are declared HERE, before any
solver is executed.  Nothing in this file imports a solver.
"""
import json
import pathlib

CATS = {
    "A": "Arithmetic and expression evaluation",
    "B": "Simplification, expansion and factorisation",
    "C": "Linear and quadratic equations",
    "D": "Simultaneous equations and inequalities",
    "E": "Powers, roots, logarithms and trigonometry",
    "F": "Differentiation",
    "G": "Integration",
    "H": "Limits and continuity",
    "I": "Matrices and linear algebra",
    "J": "Differential equations and advanced calculus",
    "K": "Partial fractions, complex numbers and numerical solving",
    "L": "Domain restrictions and failure handling",
}

TOL_EXACT = {"abs": 1e-9, "rel": 1e-9}
TOL_NUM = {"abs": 1e-8, "rel": 1e-8}   # numeric-root cases (K03/K04)

# Default derivation milestones (regex alternatives, matched case-insensitively against the
# concatenated rule+description text of the solver's steps).  Heuristic -> flagged manual.
MS = {
    "eval": [r"order of operations|evaluat|arithmetic|compute|simplif|multipl|add|subtract|power|root|absolute"],
    "simplify": [r"combine|collect|like terms|distribut|cancel|simplif|factor|identity|rule"],
    "expand": [r"expand|distribut|multiply|foil"],
    "factor": [r"factor|common|difference of squares|product"],
    "lin": [r"inverse operation|both sides|isolate|subtract|add|divide|multiply|collect|rearrang|move|solve"],
    "quad": [r"coefficient|discriminant|standard form|quadratic equation",
             r"factor|quadratic formula|complete the square",
             r"zero.product|each factor|set each|separate",
             r"verif|check|substitut|extraneous|reject"],
    "system": [r"augmented|eliminat|substitut|row|rref|scale",
               r"back.substitut|solve for|solution"],
    "ineq": [r"isolate|divide|subtract|critical|sign|test|interval|factor|reverse|flip"],
    "power": [r"power|exponent|product rule|quotient rule|index|law|identity|rule"],
    "trig": [r"special angle|unit circle|exact value|identity|evaluate|table|known value"],
    "diff": [r"rule|derivative|differentiat|chain|product|power|sum"],
    "int": [r"rule|integral|antiderivative|substitut|power|by parts|standard|integrat"],
    "defint": [r"antiderivative|fundamental theorem|evaluate.*bound|upper.*lower|F\(b\)"],
    "limit": [r"substitut|indeterminate|l.?h.?pital|factor|cancel|divide|dominant|squeeze|standard limit|direct"],
    "matrix": [r"matrix|row|determinant|cofactor|adjugate|pivot|scale|elimination|formula"],
    "ode": [r"separat|characteristic|integrating factor|auxiliary|ansatz|integrate|solve"],
    "pf": [r"factor|decompos|partial fraction|cover.up|equate|solve for|coefficients"],
    "num": [r"newton|iterat|bisect|numeric|approximat|root|initial"],
    "dom": [r"domain|restrict|undefined|denominator|extraneous|reject|check|verify|must"],
}

cases = []


def add(cid, op, display, spec_expected, problem, expected, compare, *, domain="real",
        dsens=False, restrictions=None, ms=None, tol=None, notes="", sample=None, adapter_inputs=None):
    cases.append({
        "id": cid,
        "category": cid[0],
        "category_name": CATS[cid[0]],
        "operation": op,
        "display_input": display,
        "problem": problem,
        "domain": domain,                       # real | complex
        "domain_sensitive": dsens,
        "required_restrictions": restrictions or [],   # restriction(s) the answer must preserve/report
        "spec_expected": spec_expected,         # verbatim from the task specification
        "expected": expected,                   # machine-readable reference answer
        "comparison": compare,                  # declared judging method (see bench/judge.py)
        "tolerance": tol or TOL_EXACT,
        "reference_milestones": ms or [],
        "sample_points": sample,
        "notes": notes,
        "adapter_inputs": adapter_inputs or {},   # per-solver syntax overrides (counted as adaptation effort)
    })


# --------------------------------------------------------------------------- A
def A(cid, expr, exp, spec, form=None, ms="eval"):
    add(cid, "evaluate", expr, spec, {"expr": expr}, {"expr": exp},
        {"method": "value", "form": form}, ms=MS[ms])

A("A01", "2 + 3 * 4", "14", "14")
A("A02", "(2 + 3) * 4", "20", "20")
A("A03", "7/12 + 5/18", "31/36", "31/36")
A("A04", "sqrt(72)", "6*sqrt(2)", "6*sqrt(2), if simplified", form="simplified_radical")
A("A05", "(2^3)^2", "64", "64")
A("A06", "0.125 * 80", "10", "10")
A("A07", "(-3)^2 - 3^2", "0", "0")
A("A08", "abs(-7) + abs(3-9)", "13", "13")

# --------------------------------------------------------------------------- B
add("B01", "simplify", "3*x + 5*x - 2*x", "6*x", {"expr": "3*x + 5*x - 2*x"}, {"expr": "6*x"},
    {"method": "expr_equiv", "form": "collected"}, ms=MS["simplify"])
add("B02", "simplify", "2*(x + 3) - x", "x + 6", {"expr": "2*(x + 3) - x"}, {"expr": "x + 6"},
    {"method": "expr_equiv", "form": "no_unexpanded_product"}, ms=MS["simplify"])
add("B03", "expand", "(x + 2)*(x + 3) — expand", "x^2 + 5*x + 6", {"expr": "(x + 2)*(x + 3)"},
    {"expr": "x^2 + 5*x + 6"}, {"method": "expr_equiv", "form": "expanded"}, ms=MS["expand"])
add("B04", "factor", "x^2 + 5*x + 6 — factor", "(x + 2)*(x + 3)", {"expr": "x^2 + 5*x + 6"},
    {"expr": "(x + 2)*(x + 3)"}, {"method": "expr_equiv", "form": "factored"}, ms=MS["factor"])
add("B05", "factor", "x^2 - 9 — factor", "(x - 3)*(x + 3)", {"expr": "x^2 - 9"},
    {"expr": "(x - 3)*(x + 3)"}, {"method": "expr_equiv", "form": "factored"}, ms=MS["factor"])
add("B06", "simplify", "(x^2 - 1)/(x - 1) — simplify", "x + 1, preserving x != 1",
    {"expr": "(x^2 - 1)/(x - 1)"}, {"expr": "x + 1"}, {"method": "expr_equiv", "form": "no_fraction"},
    dsens=True, restrictions=["x != 1"], ms=MS["simplify"] + MS["dom"][:1], sample={"x": ["-3", "-1/2", "2", "7/3"]})
add("B07", "simplify", "x/x — simplify", "1, preserving x != 0", {"expr": "x/x"}, {"expr": "1"},
    {"method": "expr_equiv"}, dsens=True, restrictions=["x != 0"], ms=MS["simplify"] + MS["dom"][:1])
add("B08", "simplify", "(2*x^2*y)/(4*x*y^2) — simplify", "x/(2*y), preserving original denominator restrictions",
    {"expr": "(2*x^2*y)/(4*x*y^2)"}, {"expr": "x/(2*y)"}, {"method": "expr_equiv", "form": "reduced_fraction"},
    dsens=True, restrictions=["x != 0", "y != 0"], ms=MS["simplify"] + MS["dom"][:1],
    sample={"x": ["-3", "2", "5/3"], "y": ["-2", "3", "1/2"]})

# --------------------------------------------------------------------------- C
def C(cid, eq, sol, spec, **kw):
    add(cid, "solve", eq, spec, {"equation": eq, "var": "x"}, sol,
        {"method": "solution_set"}, ms=kw.pop("ms", MS["lin"]), **kw)

C("C01", "2*x + 3 = 11", {"kind": "finite", "values": ["4"]}, "x = 4")
C("C02", "2*(x + 3) = 14", {"kind": "finite", "values": ["4"]}, "x = 4")
C("C03", "3*x - 7 = x + 5", {"kind": "finite", "values": ["6"]}, "x = 6")
C("C04", "x/3 + 2 = 5", {"kind": "finite", "values": ["9"]}, "x = 9")
C("C05", "x^2 - 5*x + 6 = 0", {"kind": "finite", "values": ["2", "3"]}, "x = 2, 3", ms=MS["quad"])
C("C06", "x^2 - 4 = 0", {"kind": "finite", "values": ["-2", "2"]}, "x = -2, 2", ms=MS["quad"])
C("C07", "x^2 + 1 = 0", {"kind": "finite", "values": ["I", "-I"]},
  "Complex solutions x = i, -i; no real solutions", domain="complex", dsens=True,
  ms=MS["quad"][:2] + [r"complex|imaginary|no real|negative discriminant|\bi\b"],
  notes="Expected over C. A real-only answer (empty set) is recorded as 'real_domain_only', not correct.")
C("C08", "x^2 + 2*x + 1 = 0", {"kind": "finite", "values": ["-1"], "multiplicity": {"-1": 2}},
  "x = -1, repeated root", ms=MS["quad"],
  notes="Set {-1}. Multiplicity reporting is recorded as a secondary check, not part of correctness.")
C("C09", "2*x + 1 = 2*x + 3", {"kind": "empty"}, "No solution", dsens=True,
  ms=[r"no solution|contradiction|inconsistent|false|never|impossible|cancel"])
C("C10", "2*x + 1 = 2*x + 1", {"kind": "all"}, "All admissible x", dsens=True,
  ms=[r"identity|all (real )?(numbers|x)|infinitely many|always true|every|any x|all admissible"])

# --------------------------------------------------------------------------- D
def Dsys(cid, eqs, sol, spec, **kw):
    add(cid, "solve_system", "; ".join(eqs), spec, {"equations": eqs, "vars": ["x", "y"], "method": kw.pop("method", None)},
        sol, {"method": "system_solution"}, ms=MS["system"], **kw)

Dsys("D01", ["x + y = 7", "x - y = 1"], {"kind": "unique", "assignments": {"x": "4", "y": "3"}}, "x = 4, y = 3")
Dsys("D02", ["2*x + y = 8", "x - y = 1"], {"kind": "unique", "assignments": {"x": "3", "y": "2"}}, "x = 3, y = 2")
Dsys("D03", ["x + y = 2", "x + y = 5"], {"kind": "empty"}, "No solution", dsens=True)
for cid, ineq, sol, spec in [
    ("D04", "2*x + 3 < 11", ["-oo", "4"], "x < 4"),
    ("D05", "-2*x > 6", ["-oo", "-3"], "x < -3"),
]:
    add(cid, "solve_inequality", ineq, spec, {"inequality": ineq, "var": "x"},
        {"kind": "intervals", "intervals": [[sol[0], sol[1], True, True]]},   # (lo, hi, open_lo, open_hi)
        {"method": "inequality_set"}, ms=MS["ineq"], dsens=(cid == "D05"),
        notes="D05 requires the inequality direction to flip when dividing by a negative number." if cid == "D05" else "")
add("D06", "solve_inequality", "x^2 - 4 > 0", "x < -2 or x > 2", {"inequality": "x^2 - 4 > 0", "var": "x"},
    {"kind": "intervals", "intervals": [["-oo", "-2", True, True], ["2", "oo", True, True]]},
    {"method": "inequality_set"}, ms=MS["ineq"], dsens=True)

# --------------------------------------------------------------------------- E
def E(cid, expr, exp, spec, **kw):
    op = kw.pop("op", "simplify")
    add(cid, op, expr, spec, {"expr": expr}, {"expr": exp}, {"method": kw.pop("method", "expr_equiv"),
        "form": kw.pop("form", None)}, ms=kw.pop("ms", MS["power"]), **kw)

E("E01", "x^3 * x^4", "x^7", "x^7")
E("E02", "x^6 / x^2", "x^4", "x^4, preserving x != 0 from the original expression", dsens=True,
  restrictions=["x != 0"], ms=MS["power"] + MS["dom"][:1], sample={"x": ["-3", "-1/2", "2", "7/3"]})
E("E03", "sqrt(x^2)", "abs(x)", "abs(x) over the reals", dsens=True, sample={"x": ["-3", "-1/2", "2", "7/3"]},
  ms=[r"abs|absolute|\|x\||modulus|principal|real"], notes="Over the reals. Result must equal |x| at negative x.")
E("E04", "log(1000, 10)", "3", "3", op="evaluate", method="value", ms=MS["power"] + [r"log|power of 10|10\^3|exponent"])
E("E05", "ln(e^x)", "x", "x over the real domain", dsens=True, sample={"x": ["-3", "-1/2", "2", "7/3"]})
E("E06", "sin(pi/6)", "1/2", "1/2", op="evaluate", method="value", ms=MS["trig"])
E("E07", "cos(pi/3)", "1/2", "1/2", op="evaluate", method="value", ms=MS["trig"])
E("E08", "tan(pi/4)", "1", "1", op="evaluate", method="value", ms=MS["trig"])

# --------------------------------------------------------------------------- F
def F(cid, expr, exp, spec, **kw):
    add(cid, "diff", f"Differentiate {expr}", spec, {"expr": expr, "var": "x"}, {"expr": exp},
        {"method": "expr_equiv", "form": None}, ms=kw.pop("ms", MS["diff"]), **kw)

F("F01", "x^3", "3*x^2", "3*x^2")
F("F02", "5*x^4 - 2*x + 7", "20*x^3 - 2", "20*x^3 - 2")
F("F03", "x^2 * sin(x)", "2*x*sin(x) + x^2*cos(x)", "2xsin(x) + x^2*cos(x)", ms=MS["diff"] + [r"product"])
F("F04", "sin(x^2)", "2*x*cos(x^2)", "2xcos(x^2)", ms=MS["diff"] + [r"chain"])
F("F05", "ln(x)", "1/x", "1/x for x > 0", dsens=True, sample={"x": ["1/2", "13/10", "27/10", "71/10"]})
F("F06", "x^x", "x^x*(ln(x) + 1)", "x^x*(ln(x) + 1) for x > 0", dsens=True,
  sample={"x": ["1/2", "13/10", "27/10", "71/10"]}, ms=MS["diff"] + [r"logarithmic|ln|log"])

# --------------------------------------------------------------------------- G
def G(cid, expr, exp, spec, **kw):
    add(cid, "integrate", f"Integrate {expr}", spec, {"expr": expr, "var": "x"}, {"expr": exp},
        {"method": "antiderivative", "require_real_on": kw.pop("real_on", None)}, ms=kw.pop("ms", MS["int"]), **kw)

G("G01", "3*x^2", "x^3", "x^3 + C")
G("G02", "2*x + 5", "x^2 + 5*x", "x^2 + 5*x + C")
G("G03", "cos(x)", "sin(x)", "sin(x) + C")
G("G04", "1/x", "ln(abs(x))", "ln(abs(x)) + C, on intervals excluding zero", dsens=True,
  real_on=["-4", "-1/2", "1/2", "4"], ms=MS["int"] + [r"abs|absolute|\|x\|"],
  notes="Declared rule: F'(x)=1/x AND F real-valued on both x<0 and x>0 (log(x) alone fails for x<0).")
G("G05", "x*exp(x^2)", "exp(x^2)/2", "exp(x^2)/2 + C", ms=MS["int"] + [r"substitut|u\s*="])
add("G06", "integrate", "Integrate x^2 from 0 to 3", "9", {"expr": "x^2", "var": "x", "lower": "0", "upper": "3"},
    {"expr": "9"}, {"method": "value"}, ms=MS["defint"])

# --------------------------------------------------------------------------- H
def Hh(cid, expr, pt, exp, spec, disp):
    add(cid, "limit", disp, spec, {"expr": expr, "var": "x", "point": pt, "direction": "both"},
        {"expr": exp}, {"method": "value"}, ms=MS["limit"])

Hh("H01", "sin(x)/x", "0", "1", "1", "Limit as x approaches 0 of sin(x)/x")
Hh("H02", "(x^2 - 4)/(x - 2)", "2", "4", "4", "Limit as x approaches 2 of (x^2 - 4)/(x - 2)")
Hh("H03", "(3*x^2 + 1)/(x^2 - 5)", "oo", "3", "3", "Limit as x approaches infinity of (3*x^2 + 1)/(x^2 - 5)")
Hh("H04", "(1 - cos(x))/x^2", "0", "1/2", "1/2", "Limit as x approaches 0 of (1 - cos(x))/x^2")

# --------------------------------------------------------------------------- I
add("I01", "solve_system", "Solve x + y = 5; x - y = 1 by elimination", "x = 3, y = 2",
    {"equations": ["x + y = 5", "x - y = 1"], "vars": ["x", "y"], "method": "elimination"},
    {"kind": "unique", "assignments": {"x": "3", "y": "2"}}, {"method": "system_solution"}, ms=MS["system"])
M = [[1, 2], [3, 4]]
add("I02", "det", "Determinant of [[1,2],[3,4]]", "-2", {"matrix": M}, {"expr": "-2"}, {"method": "value"}, ms=MS["matrix"])
add("I03", "inverse", "Inverse of [[1,2],[3,4]]", "[[-2,1],[3/2,-1/2]]", {"matrix": M},
    {"matrix": [["-2", "1"], ["3/2", "-1/2"]]}, {"method": "matrix"}, ms=MS["matrix"])
add("I04", "rref", "RREF of [[1,2,3],[2,4,6]]", "[[1,2,3],[0,0,0]]", {"matrix": [[1, 2, 3], [2, 4, 6]]},
    {"matrix": [["1", "2", "3"], ["0", "0", "0"]]}, {"method": "matrix"}, ms=MS["matrix"])

# --------------------------------------------------------------------------- J
add("J01", "dsolve", "Solve y' = 3*y", "y = Cexp(3x)", {"ode": "Derivative(y(x), x) - 3*y(x)", "func": "y", "var": "x", "order": 1},
    {"family": "C*exp(3*x)", "n_constants": 1}, {"method": "ode_family"}, ms=MS["ode"],
    adapter_inputs={"stepsolver": "dsolve(diff(y(x), x) = 3*y(x), y(x))"})
add("J02", "dsolve", "Solve y'' + y = 0", "y = C1cos(x) + C2sin(x)", {"ode": "Derivative(y(x), (x, 2)) + y(x)", "func": "y", "var": "x", "order": 2},
    {"family": "C1*cos(x) + C2*sin(x)", "n_constants": 2}, {"method": "ode_family"}, ms=MS["ode"],
    adapter_inputs={"stepsolver": "dsolve(diff(y(x), x, 2) + y(x) = 0, y(x))"})
add("J03", "partial_derivative", "Differentiate exp(x*y) with respect to x, treating y as constant", "yexp(xy)",
    {"expr": "exp(x*y)", "var": "x", "vars": ["x", "y"]}, {"expr": "y*exp(x*y)"},
    {"method": "expr_equiv"}, ms=MS["diff"] + [r"constant|chain|partial"], sample={"x": ["-1", "1/2", "2"], "y": ["-3/2", "1", "2"]})
add("J04", "gradient", "Gradient of f(x,y) = x^2 + x*y + y^2", "(2x + y, x + 2y)",
    {"expr": "x^2 + x*y + y^2", "vars": ["x", "y"]}, {"vector": ["2*x + y", "x + 2*y"]},
    {"method": "gradient"}, ms=MS["diff"] + [r"partial|gradient"], sample={"x": ["-1", "1/2", "2"], "y": ["-3/2", "1", "2"]})

# --------------------------------------------------------------------------- K
add("K01", "partial_fractions", "Partial fractions of 1/(x^2 - 1)", "1/(2*(x - 1)) - 1/(2*(x + 1))",
    {"expr": "1/(x^2 - 1)", "var": "x"}, {"expr": "1/(2*(x - 1)) - 1/(2*(x + 1))"},
    {"method": "expr_equiv", "form": "partial_fractions"}, ms=MS["pf"], dsens=True,
    sample={"x": ["-3", "-1/2", "2", "7/3"]},
    notes="Declared rule: equivalent AND a sum of fractions with linear denominators (must actually be decomposed).")
add("K02", "solve", "Solve x^3 - 1 = 0 over complex numbers", "1, (-1 + isqrt(3))/2, (-1 - isqrt(3))/2",
    {"equation": "x^3 - 1 = 0", "var": "x"},
    {"kind": "finite", "values": ["1", "(-1 + I*sqrt(3))/2", "(-1 - I*sqrt(3))/2"]},
    {"method": "solution_set"}, domain="complex", dsens=True, ms=MS["quad"][:3] + [r"complex|cube root|unity|imaginary"])
add("K03", "nsolve", "Numerically solve cos(x) = x near 0.7", "x approximately 0.7390851332",
    {"equation": "cos(x) = x", "var": "x", "guess": "0.7"}, {"kind": "numeric_roots", "values": ["0.7390851332"], "require": "all_expected"},
    {"method": "numeric_roots"}, tol=TOL_NUM, ms=MS["num"],
    notes="Closest documented API per solver (SymPy: nsolve). Candidates without a numeric root-finder get the generic equation solver.")
add("K04", "nsolve", "Numerically solve x^5 - x - 1 = 0", "At least the real root approximately 1.167304015",
    {"equation": "x^5 - x - 1 = 0", "var": "x", "guess": None}, {"kind": "numeric_roots", "values": ["1.167303978261"], "require": "all_expected", "spec_printed_value": "1.167304015"},
    {"method": "numeric_roots"}, tol=TOL_NUM, ms=MS["num"],
    notes="SPEC DISCREPANCY (found by reference validation, declared before any solver ran): the specification prints "
          "'approximately 1.167304015' but the real root of x^5-x-1 is 1.1673039782614187 (mpmath 40 digits, bisection, polyroots). "
          "The printed value is off by 3.7e-8. spec_expected is preserved verbatim; the verified root is the reference; tolerance 1e-8. "
          "Extra (complex) roots are allowed. An exact CRootOf(...) is accepted if it evaluates to the real root within tolerance.")

# --------------------------------------------------------------------------- L
add("L01", "solve", "Solve (x^2 - 1)/(x - 1) = 0", "x = -1; x = 1 remains excluded from the original expression",
    {"equation": "(x^2 - 1)/(x - 1) = 0", "var": "x"}, {"kind": "finite", "values": ["-1"], "excluded": ["1"]},
    {"method": "solution_set"}, dsens=True, ms=MS["lin"] + MS["dom"],
    notes="x = 1 stays excluded: satisfied when the solution set is exactly {-1}; no separate restriction statement required.")
add("L02", "solve", "Solve sqrt(x + 1) = x - 1 over the reals", "x = 3 after domain and original-equation checks",
    {"equation": "sqrt(x + 1) = x - 1", "var": "x"}, {"kind": "finite", "values": ["3"], "extraneous": ["0"]},
    {"method": "solution_set"}, dsens=True, ms=[r"square|both sides|isolate|rearrang"] + MS["dom"])
add("L03", "simplify", "Simplify sqrt(x^2)", "abs(x) over the reals", {"expr": "sqrt(x^2)"}, {"expr": "abs(x)"},
    {"method": "expr_equiv"}, dsens=True, sample={"x": ["-3", "-1/2", "2", "7/3"]}, ms=[r"abs|absolute|\|x\||modulus|principal|real"])
add("L04", "solve", "Solve 1/(x - x) = 2", "Undefined expression or domain error; no fabricated solution",
    {"equation": "1/(x - x) = 2", "var": "x"}, {"kind": "undefined"}, {"method": "undefined"}, dsens=True,
    ms=[r"undefined|division by zero|denominator|domain"],
    notes="pass = explicit undefined/domain error; partial = empty set without flagging undefined; fail = any solution.")

assert len(cases) == 72, len(cases)
assert len({c["id"] for c in cases}) == 72

out = pathlib.Path(__file__).resolve().parent.parent / "data" / "cases.json"
out.write_text(json.dumps({"schema_version": 1, "count": len(cases), "categories": CATS, "cases": cases}, indent=1, ensure_ascii=False) + "\n")
print("wrote", out, "cases:", len(cases))
''')

W('docs/METHODOLOGY.md', r'''# Methodology and declared decisions

## Isolation
Three virtualenvs (`envs/sympy|smartsolver|stepsolver`), one worker process per solver (`bench/workers/*_worker.py`), launched from an empty temp directory with a minimal environment. The controller (`bench/runner.py`) talks JSON lines over pipes, enforces a 20 s per-case timeout (kills and restarts the worker), and preserves every raw reply in `results/raw/<solver>.json`. No application/production code exists in or is referenced by this workspace. SymPy runs through `bench/workers/sympy_worker.py`, which imports only `sympy`.

## Adapters (what each solver was asked)
Documented at the top of each worker file. Summary: SymPy bare API (simplify/expand/factor, `solveset` with explicit real/complex domain, `solve` for systems, `solve_univariate_inequality`, `diff`, `integrate`, `limit(dir='+-')`, Matrix ops, `dsolve`, `apart`, `nsolve`/`Poly.nroots`); SmartSolver one documented public method per operation family, **unsupported** where none exists (public-method list recorded in each reply); StepSolver documented ASCII operations, plus per-case syntax overrides in `cases.json:adapter_inputs` (ODEs), `ln->log`, and gradient as two native `diff` calls. Counts of adaptations are measured (`adapter_overrides`).

## Judge (`bench/judge.py`)
No string-equality-only comparison. Value: symbolic zero or 50-digit numeric agreement. Expression equivalence: symbolic zero and/or agreement at exact rational sample points (a complex/NaN value at a real sample point is a mismatch). Solution sets: exact set equality over the declared domain. Systems, inequality sets (via `as_set`), matrices, gradients, ODE general solutions (substitution residual + constant count + independent basis/Wronskian), numeric roots (tolerance 1e-8), antiderivatives (F'=f, plus real-valuedness on both sides of 0 for G04), undefined-expression handling (L04). Form checks and restriction-reporting rules are declared in `cases.json` and the module docstring. Judge parsing of solver text uses an allow-list and no builtins.

## Reference validation (`bench/reference_validation.py`)
Every reference answer is re-checked with stdlib `Fraction`/`cmath` and mpmath numerics (numerical differentiation/quadrature/root finding, exhaustive rational scans, truth tables for inequalities), without SymPy symbolic solvers. Result: 72/72 validated. It exposed the K04 specification typo (printed 1.167304015 vs true 1.1673039782614187); the verbatim spec text is kept in `spec_expected`, the verified root is the reference.

## Rule changes made after the first functional pass (disclosed)
Made after seeing raw outputs but before any verdicts or scores were produced or compared: (1) `ln->log` adapter normalisation for StepSolver (it silently treats `ln` as an unknown function); (2) L01 required no separate restriction statement (exclusion from the solution set suffices); (3) generic simplification-form rule (a simplify/evaluate answer must not be >1 operation more complex than the reference) after one solver echoed its input; (4) ODE input overrides for StepSolver. Made while reviewing derivations (applied uniformly): (5) raw rule-tree dumps are not counted as prose; workings from a code path separate from the answer are capped at correctness 2; (6) opaque-only derivations (no transformation shown) get correctness<=2, consistency 0, presentation 0; (7) the boilerplate label "Apply algebraic identities" does not count as a specific rule. (8) G1 filter bug fixed (a substring match wrongly treated L04's note "no fabricated solution" as critical). The sensitivity table in `summary.md` shows the ranking is unchanged under value-only/form-free correctness, availability-penalised derivations, automated-only derivation scores, and own-case latency.

## Manual review
In the baseline run, 39 of 113 scored derivations were reviewed by a language model (Claude) using the printed raw workings; each entry has adjustments, confidence and justification in `data/manual_reviews.json`; `raw_results.json` keeps automated and final scores. It is **not** human review and must be audited. Unreviewed derivations use automated scores. Mathematical correctness is not decided by this review.

## Reproducibility
`./run_all.sh` (clean machine with git, uv, Python 3.12). Candidate commits and dependency pins: `pins/`. Two full runs produced identical raw outputs, verdicts and derivation scores for all 216 rows (`results/reproducibility_check.json`); only latencies vary.

## Re-running on new engine versions (Colab notebook)
The notebook rebuilds isolated environments for the engine versions you configure, runs the identical dataset, judge, rubric, probes and scoring, and regenerates every report from data (no run-specific narrative). The manual derivation overlay carries a fingerprint (hash of the reviewed steps+answer): it is applied only when an engine's output is unchanged; otherwise the automated score is used and the report states how many reviews were not applied. The judge runs in a separate environment with a fixed SymPy version so the same answers are judged identically across engine versions. `data/baseline.json` stores the original run for `changes_vs_baseline.md`.
''')

W('docs/SCORING_RULES.md', r'''# Scoring rules (declared before the overall score was computed)

`Overall = 0.30*Correctness + 0.30*Derivations + 0.20*Coverage + 0.10*Reliability + 0.05*Integration + 0.05*Performance`

Applicable cases = all 72 for every solver (no case is excluded as "not applicable"; unsupported cases count against the solver).

| Component | Normalisation to 0-100 |
|---|---|
| Correctness | strictly-correct cases / 72 x 100. Strict = correct value (symbolic equivalence, set equality, substitution/ODE residual, numeric tolerance) AND required form AND (if the case requires it) the domain restriction reported. Unsupported, runtime errors and invalid answers all count as not correct but are reported separately. Value-only correctness is reported as a sensitivity. |
| Derivations | (mean final rubric score /10 over derivations that exist and whose answer was produced) x 10, as specified. Availability reported separately; an availability-penalised variant (unavailable = 0 over all 72) is in the sensitivity table. |
| Coverage | cases where the solver produced a result in the expected form / 72 x 100 (correctness not required). |
| Reliability | 100 x mean(R1..R5): R1 share of repeated identical runs (5 cache-cleared repeats + main pass) with identical canonical output; R2 mean engineering-probe score (1 / 0.5 / 0, n/a excluded); R3 = 1 - (runtime errors+timeouts+infra)/attempted (attempted = cases where an API exists); R4 strict-correct rate on the 21 domain-sensitive cases; R5 = 1 if the benign canary payload is rejected by the solver's own string interface, else 0. |
| Integration & maintainability | 14-point rubric computed from evidence: env compat 2 (<=4 third-party deps; 1 if <=30), install 2 (single command; 1 if it needed an unpinned-SymPy fallback), adaptation effort 2 (<=2% of cases need overrides; 1 if <=10%), testability/extensibility 2 (>=100 static test functions; 1 if >=1; -1 if the own suite was run and fails; SymPy fixed 0: no derivation framework), licence 2 (permissive; 1 unknown; 0 copyleft), maintenance 2 (>=2 years and >=5 authors; 1 if >=5 commits and active within a year; SymPy fixed 2), UI independence 1 and no-production-rewrite 1 (fixed assumptions) -> x 100/14. Evidence strings are in `scores.json`. |
| Performance | latency score S(m) = 100 if m<=10 ms; 0 if m>=2000 ms; else 100*log10(2000/m)/log10(200). Performance = 0.4*S(median) + 0.4*S(p95) + 0.2*100*(1-failure rate). Median/p95 are warm-process, SymPy-cache-cleared repeats over the cases ALL three solvers answered. Cold start and memory are reported, not scored. |

## Derivation rubric (10 points) - automated implementation
See the docstring of `bench/derivations.py`. Criteria: correctness 0-3, completeness 0-2, explanation 0-2, consistency 0-1, domain awareness 0-1, presentation 0-1. A manual overlay (`data/manual_reviews.json`) records per-criterion adjustments with confidence and justification; the overlay author is a language model and is flagged as requiring human audit. Mathematical correctness is never decided by a language model: it comes from `bench/judge.py` and `bench/reference_validation.py`.

## Hard gates
- G1 critical correctness defect: a confidently wrong answer (false "no solution", wrong value/set, fabricated solution). FAIL if >=1 confirmed.
- G2 invalid or self-contradicting derivation (automated invalid-step detection, `direct_evaluation` override, or manual finding of contradiction/unchanged-input final step). FAIL if >=2 cases, CONDITIONAL if 1.
- G3 benign canary `__import__('math').factorial(4)` evaluated by the solver's own string interface. FAIL if evaluated and no safe alternative interface; CONDITIONAL for SymPy (programmatic API + allow-list adapter demonstrated).
- G4 licence incompatible with commercial use: none found (BSD-3, MIT, MIT).
''')

W('data/manual_reviews.json', r'''{
 "stepsolver|A04": {
  "adjust": {
   "completeness": 1
  },
  "confidence": "high",
  "justification": "Single generic 'Apply algebraic identities' step: the specific identity (e.g. 72=36*2 / special-angle value / log definition) is never named or shown. Essential factoring of 72 skipped.",
  "fingerprint": "d2c5a1815a14a5f4"
 },
 "stepsolver|A08": {
  "adjust": {
   "completeness": 1
  },
  "confidence": "medium",
  "justification": "Single generic 'Apply algebraic identities' step: the specific identity (e.g. 72=36*2 / special-angle value / log definition) is never named or shown.",
  "fingerprint": "88d43fea6b54fe76"
 },
 "stepsolver|E04": {
  "adjust": {
   "completeness": 1
  },
  "confidence": "medium",
  "justification": "Single generic 'Apply algebraic identities' step: the specific identity (e.g. 72=36*2 / special-angle value / log definition) is never named or shown.",
  "fingerprint": "fc5c446851e8ede3"
 },
 "stepsolver|E06": {
  "adjust": {
   "completeness": 1
  },
  "confidence": "medium",
  "justification": "Single generic 'Apply algebraic identities' step: the specific identity (e.g. 72=36*2 / special-angle value / log definition) is never named or shown.",
  "fingerprint": "4a001801febd6348"
 },
 "stepsolver|E07": {
  "adjust": {
   "completeness": 1
  },
  "confidence": "medium",
  "justification": "Single generic 'Apply algebraic identities' step: the specific identity (e.g. 72=36*2 / special-angle value / log definition) is never named or shown.",
  "fingerprint": "7e4e2bf5616a7fa5"
 },
 "stepsolver|E08": {
  "adjust": {
   "completeness": 1
  },
  "confidence": "medium",
  "justification": "Single generic 'Apply algebraic identities' step: the specific identity (e.g. 72=36*2 / special-angle value / log definition) is never named or shown.",
  "fingerprint": "fc32ccf45b3afce8"
 },
 "stepsolver|A03": {
  "adjust": {
   "completeness": 1
  },
  "confidence": "medium",
  "justification": "Mentions a common denominator but never shows the LCD (36) or the rewritten numerators; result appears in one jump.",
  "fingerprint": "873b1d8cbf095f42"
 },
 "stepsolver|A07": {
  "adjust": {
   "correctness": 1,
   "consistency": 0,
   "presentation": 0
  },
  "confidence": "high",
  "justification": "Displayed first step is '-3 ^ 2 - 3 ^ 2' (LaTeX: -3^{2} - 3^{2}) i.e. parentheses around the negative base are lost; as displayed it equals -18, not the problem's value. Final answer 0 is right, but the shown expression misstates the problem. Reproduce: Solver().solve('simplify((-3)^2 - 3^2)').",
  "fingerprint": "15c441dccd74cb75"
 },
 "stepsolver|B06": {
  "adjust": {
   "completeness": 1
  },
  "confidence": "medium",
  "justification": "Explicit restriction 'x - 1 != 0' reported (good) but the factorisation (x-1)(x+1) that justifies the cancellation is described, not shown.",
  "fingerprint": "34a560a177eece67"
 },
 "stepsolver|C01": {
  "adjust": {},
  "confidence": "medium",
  "justification": "Clear, justified inverse-operation steps. Steps 2 and 5 display identical before/after text (cancellation only implied in ASCII) - minor. No change.",
  "fingerprint": "ba795e5b0bf49d85"
 },
 "stepsolver|C07": {
  "adjust": {
   "correctness": 1,
   "domain": 1
  },
  "confidence": "medium",
  "justification": "Derivation is valid and explicitly bounded to the real number system ('no roots in the real number system'), but stops there; the benchmark expects complex roots i,-i. Partial credit for valid workings; the final answer itself stays 'invalid_answer (real_domain_only)'.",
  "fingerprint": "a23a89b48f84ebbd"
 },
 "stepsolver|F03": {
  "adjust": {
   "completeness": 1
  },
  "confidence": "medium",
  "justification": "Product rule named and described, but u, v, u', v' and the u'v+uv' assembly are not displayed; result appears in one jump.",
  "fingerprint": "5dedf8f3621706a1"
 },
 "stepsolver|H03": {
  "adjust": {
   "completeness": 1
  },
  "confidence": "medium",
  "justification": "Key idea (dominant leading powers) stated clearly, but the division by x^2 step is not shown.",
  "fingerprint": "028f84a47983df3a"
 },
 "stepsolver|G05": {
  "adjust": {},
  "confidence": "high",
  "justification": "Clear, ordered, justified steps with correct rule names; domain restrictions handled where relevant (L01 discards x=1). No change.",
  "fingerprint": "91e8344a6ece8f9d"
 },
 "stepsolver|G06": {
  "adjust": {},
  "confidence": "high",
  "justification": "Clear, ordered, justified steps with correct rule names; domain restrictions handled where relevant (L01 discards x=1). No change.",
  "fingerprint": "25e50c9738d6c60e"
 },
 "stepsolver|L01": {
  "adjust": {},
  "confidence": "high",
  "justification": "Clear, ordered, justified steps with correct rule names; domain restrictions handled where relevant (L01 discards x=1). No change.",
  "fingerprint": "610c97357f665056"
 },
 "stepsolver|H01": {
  "adjust": {},
  "confidence": "medium",
  "justification": "Valid L'Hopital derivation with correct explanations. Pedagogical caveat: L'Hopital for sin(x)/x is circular in many curricula; not penalised.",
  "fingerprint": "a5b3abe1616a506c"
 },
 "stepsolver|D04": {
  "adjust": {},
  "confidence": "high",
  "justification": "Only the opaque 'Compute exact result ... with the symbolic backend' step: no workings at all (for L02 no squaring/domain/extraneous-root check although the spec requires them). Rubric rule 5/9: score stays at the minimum automated level.",
  "fingerprint": "f1a9ac3c3647ccc5"
 },
 "stepsolver|I03": {
  "adjust": {},
  "confidence": "high",
  "justification": "Only the opaque 'Compute exact result ... with the symbolic backend' step: no workings at all (for L02 no squaring/domain/extraneous-root check although the spec requires them). Rubric rule 5/9: score stays at the minimum automated level.",
  "fingerprint": "43692c4717f1260d"
 },
 "stepsolver|J01": {
  "adjust": {},
  "confidence": "high",
  "justification": "Only the opaque 'Compute exact result ... with the symbolic backend' step: no workings at all (for L02 no squaring/domain/extraneous-root check although the spec requires them). Rubric rule 5/9: score stays at the minimum automated level.",
  "fingerprint": "95bd57b17e94af84"
 },
 "stepsolver|L02": {
  "adjust": {},
  "confidence": "high",
  "justification": "Only the opaque 'Compute exact result ... with the symbolic backend' step: no workings at all (for L02 no squaring/domain/extraneous-root check although the spec requires them). Rubric rule 5/9: score stays at the minimum automated level.",
  "fingerprint": "508aa821c2a87214"
 },
 "smartsolver|C07": {
  "adjust": {
   "completeness": 1,
   "domain": 1
  },
  "confidence": "medium",
  "justification": "Correctly derives complex roots (discriminant -4) and names the complex-domain result, but never lists both roots in the steps (only '-I' shown) and has a pointless 'complete the square' display.",
  "fingerprint": "ccfd830bd1abdecc"
 },
 "smartsolver|F03": {
  "adjust": {
   "explanation": 2
  },
  "confidence": "medium",
  "justification": "Product rule named with u and v given and u'v+uv' assembled explicitly.",
  "fingerprint": "2ca7dccae09337e6"
 },
 "smartsolver|G05": {
  "adjust": {
   "explanation": 2,
   "presentation": 0
  },
  "confidence": "high",
  "justification": "Clear u-substitution steps, but an internal variable name leaks into the displayed workings ('exp(_u)/2').",
  "fingerprint": "371571bc499021b7"
 },
 "smartsolver|G06": {
  "adjust": {
   "completeness": 1
  },
  "confidence": "medium",
  "justification": "Power rule and FTC named, but F(3)-F(0) arithmetic (27/3 - 0) is skipped.",
  "fingerprint": "26488e1e5383791a"
 },
 "smartsolver|G04": {
  "adjust": {},
  "confidence": "high",
  "justification": "CONTRADICTION: step 2 states the antiderivative is ln|x|, then a 'Direct evaluation' step replaces it with log(x), which is not real-valued for x<0. steps_mode=direct_evaluation. Counts toward the fabricated/inconsistent-derivation hard gate.",
  "fingerprint": "e5af563d33f2f413"
 },
 "smartsolver|K01": {
  "adjust": {
   "correctness": 1,
   "completeness": 1
  },
  "confidence": "high",
  "justification": "Factorisation and cover-up coefficients (1/2, -1/2) are correct, but the final 'Partial fraction decomposition' step shows the ORIGINAL expression 1/(x**2-1) - the assembled decomposition is never produced.",
  "fingerprint": "b396ac1897bed41c"
 },
 "smartsolver|K02": {
  "adjust": {
   "completeness": 1
  },
  "confidence": "medium",
  "justification": "Factors x^3-1 correctly, then 'Solve factor x**2+x+1 = 0' without ever showing the complex roots.",
  "fingerprint": "cb9be6b4b35849ba"
 },
 "smartsolver|C01": {
  "adjust": {},
  "confidence": "medium",
  "justification": "Reviewed: automated score accepted. Ordered, correctly labelled steps.",
  "fingerprint": "7e11e87b61dc90ad"
 },
 "smartsolver|D01": {
  "adjust": {},
  "confidence": "medium",
  "justification": "Reviewed: automated score accepted. Ordered, correctly labelled steps.",
  "fingerprint": "5820ea7817edf2cb"
 },
 "smartsolver|I04": {
  "adjust": {},
  "confidence": "medium",
  "justification": "Reviewed: automated score accepted. Ordered, correctly labelled steps.",
  "fingerprint": "e6be2c5610c8d597"
 },
 "smartsolver|J03": {
  "adjust": {},
  "confidence": "medium",
  "justification": "Reviewed: automated score accepted. Ordered, correctly labelled steps.",
  "fingerprint": "84743dea511a1466"
 },
 "smartsolver|J04": {
  "adjust": {},
  "confidence": "medium",
  "justification": "Reviewed: automated score accepted. Ordered, correctly labelled steps.",
  "fingerprint": "ae2407ffb19ef063"
 },
 "smartsolver|F06": {
  "adjust": {},
  "confidence": "medium",
  "justification": "Reviewed: automated score accepted. Names logarithmic differentiation but shows none of the method.",
  "fingerprint": "f28642b8244cb20a"
 },
 "smartsolver|H01": {
  "adjust": {},
  "confidence": "medium",
  "justification": "Reviewed: automated score accepted. Only 'Setup / Limit setup / Final result' - no justification for the limit value.",
  "fingerprint": "d086deaaede64b49"
 },
 "smartsolver|L01": {
  "adjust": {},
  "confidence": "medium",
  "justification": "Reviewed: automated score accepted. Only 'Solved: x = -1'; no domain reasoning shown for the x=1 exclusion.",
  "fingerprint": "384d49a09d4c9ebc"
 },
 "smartsolver|L02": {
  "adjust": {},
  "confidence": "medium",
  "justification": "Reviewed: automated score accepted. Only 'Solved: x = 3'; no squaring/extraneous-root check shown.",
  "fingerprint": "45fb3c6eb5d03267"
 },
 "sympy-direct|G01": {
  "adjust": {},
  "confidence": "high",
  "justification": "SymPy's integral_steps returns a raw rule-tree repr (ConstantTimesRule(...)): rules are identifiable by name but unexplained and not student-readable; it comes from a separate code path from integrate(), so it does not prove how the answer was obtained.",
  "fingerprint": "2fbaf191d7867bf6"
 },
 "sympy-direct|G05": {
  "adjust": {},
  "confidence": "high",
  "justification": "SymPy's integral_steps returns a raw rule-tree repr (ConstantTimesRule(...)): rules are identifiable by name but unexplained and not student-readable; it comes from a separate code path from integrate(), so it does not prove how the answer was obtained.",
  "fingerprint": "60c0ed88db2fecc6"
 }
}''')

W('data/baseline.json', r'''{"label": "original benchmark run: SymPy 1.14.0, SmartSolver 0.4.1 @ f0d46a1, StepSolver 0.1.0 @ bd8ea4f (2026-10-23, 1-core sandbox)", "engines": {"sympy-direct": {"version": "1.14.0", "commit": null}, "smartsolver": {"version": "0.4.1", "commit": "f0d46a1089366f8e14f73de9ba811d8e4c1471b0"}, "stepsolver": {"version": "0.1.0", "commit": "bd8ea4ffdbe49277d649017cb6a31bf05b376bd8"}}, "overall": {"sympy-direct": 75.54963105541215, "smartsolver": 59.34776249462321, "stepsolver": 87.36096356205924}, "components": {"sympy-direct": {"correctness": 91.66666666666667, "derivations": 43.33333333333333, "coverage": 98.61111111111111, "reliability": 64.28571428571429, "integration": 85.71428571428571, "performance": 92.26246237808425}, "smartsolver": {"correctness": 48.611111111111114, "derivations": 67.94871794871794, "coverage": 51.388888888888886, "reliability": 57.11904761904761, "integration": 85.71428571428571, "performance": 82.08833902555335}, "stepsolver": {"correctness": 93.05555555555556, "derivations": 80.29411764705883, "coverage": 95.83333333333333, "reliability": 90.19047619047619, "integration": 78.57142857142857, "performance": 64.83551773978422}}, "cases": {"sympy-direct|A01": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|A02": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|A03": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|A04": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|A05": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|A06": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|A07": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|A08": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|B01": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|B02": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|B03": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|B04": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|B05": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|B06": {"status": "valid", "strict_correct": false, "derivation_final": null}, "sympy-direct|B07": {"status": "valid", "strict_correct": false, "derivation_final": null}, "sympy-direct|B08": {"status": "valid", "strict_correct": false, "derivation_final": null}, "sympy-direct|C01": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|C02": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|C03": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|C04": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|C05": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|C06": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|C07": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|C08": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|C09": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|C10": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|D01": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|D02": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|D03": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|D04": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|D05": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|D06": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|E01": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|E02": {"status": "valid", "strict_correct": false, "derivation_final": null}, "sympy-direct|E03": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|E04": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|E05": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|E06": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|E07": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|E08": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|F01": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|F02": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|F03": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|F04": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|F05": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|F06": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|G01": {"status": "valid", "strict_correct": true, "derivation_final": 5}, "sympy-direct|G02": {"status": "valid", "strict_correct": true, "derivation_final": 5}, "sympy-direct|G03": {"status": "valid", "strict_correct": true, "derivation_final": 5}, "sympy-direct|G04": {"status": "invalid_answer", "strict_correct": false, "derivation_final": 2}, "sympy-direct|G05": {"status": "valid", "strict_correct": true, "derivation_final": 5}, "sympy-direct|G06": {"status": "valid", "strict_correct": true, "derivation_final": 4}, "sympy-direct|H01": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|H02": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|H03": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|H04": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|I01": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|I02": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|I03": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|I04": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|J01": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|J02": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|J03": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|J04": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|K01": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|K02": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|K03": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|K04": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|L01": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|L02": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|L03": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|L04": {"status": "invalid_answer", "strict_correct": false, "derivation_final": null}, "smartsolver|A01": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|A02": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|A03": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|A04": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|A05": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|A06": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|A07": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|A08": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|B01": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|B02": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|B03": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|B04": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|B05": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|B06": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|B07": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|B08": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|C01": {"status": "valid", "strict_correct": true, "derivation_final": 8}, "smartsolver|C02": {"status": "valid", "strict_correct": true, "derivation_final": 8}, "smartsolver|C03": {"status": "valid", "strict_correct": true, "derivation_final": 8}, "smartsolver|C04": {"status": "valid", "strict_correct": true, "derivation_final": 8}, "smartsolver|C05": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "smartsolver|C06": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "smartsolver|C07": {"status": "valid", "strict_correct": true, "derivation_final": 8}, "smartsolver|C08": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "smartsolver|C09": {"status": "valid", "strict_correct": true, "derivation_final": 2}, "smartsolver|C10": {"status": "runtime_error", "strict_correct": false, "derivation_final": null}, "smartsolver|D01": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "smartsolver|D02": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "smartsolver|D03": {"status": "valid", "strict_correct": true, "derivation_final": 8}, "smartsolver|D04": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|D05": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|D06": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|E01": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|E02": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|E03": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|E04": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|E05": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|E06": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|E07": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|E08": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|F01": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "smartsolver|F02": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "smartsolver|F03": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "smartsolver|F04": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "smartsolver|F05": {"status": "valid", "strict_correct": true, "derivation_final": 8}, "smartsolver|F06": {"status": "valid", "strict_correct": true, "derivation_final": 7}, "smartsolver|G01": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "smartsolver|G02": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "smartsolver|G03": {"status": "valid", "strict_correct": true, "derivation_final": 8}, "smartsolver|G04": {"status": "invalid_answer", "strict_correct": false, "derivation_final": 3}, "smartsolver|G05": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "smartsolver|G06": {"status": "valid", "strict_correct": true, "derivation_final": 8}, "smartsolver|H01": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "smartsolver|H02": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "smartsolver|H03": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "smartsolver|H04": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "smartsolver|I01": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "smartsolver|I02": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|I03": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|I04": {"status": "valid", "strict_correct": true, "derivation_final": 8}, "smartsolver|J01": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|J02": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|J03": {"status": "valid", "strict_correct": true, "derivation_final": 8}, "smartsolver|J04": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "smartsolver|K01": {"status": "invalid_answer", "strict_correct": false, "derivation_final": 4}, "smartsolver|K02": {"status": "valid", "strict_correct": true, "derivation_final": 7}, "smartsolver|K03": {"status": "invalid_answer", "strict_correct": false, "derivation_final": 1}, "smartsolver|K04": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "smartsolver|L01": {"status": "valid", "strict_correct": true, "derivation_final": 2}, "smartsolver|L02": {"status": "valid", "strict_correct": true, "derivation_final": 2}, "smartsolver|L03": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|L04": {"status": "invalid_answer", "strict_correct": false, "derivation_final": 0}, "stepsolver|A01": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|A02": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|A03": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|A04": {"status": "valid", "strict_correct": true, "derivation_final": 8}, "stepsolver|A05": {"status": "valid", "strict_correct": true, "derivation_final": 7}, "stepsolver|A06": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|A07": {"status": "valid", "strict_correct": true, "derivation_final": 6}, "stepsolver|A08": {"status": "valid", "strict_correct": true, "derivation_final": 8}, "stepsolver|B01": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|B02": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|B03": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|B04": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|B05": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|B06": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|B07": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|B08": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|C01": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|C02": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|C03": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|C04": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|C05": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|C06": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|C07": {"status": "invalid_answer", "strict_correct": false, "derivation_final": 6}, "stepsolver|C08": {"status": "valid", "strict_correct": true, "derivation_final": 8}, "stepsolver|C09": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|C10": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "stepsolver|D01": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|D02": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|D03": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|D04": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "stepsolver|D05": {"status": "valid", "strict_correct": true, "derivation_final": 2}, "stepsolver|D06": {"status": "valid", "strict_correct": true, "derivation_final": 2}, "stepsolver|E01": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|E02": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|E03": {"status": "valid", "strict_correct": true, "derivation_final": 8}, "stepsolver|E04": {"status": "valid", "strict_correct": true, "derivation_final": 8}, "stepsolver|E05": {"status": "valid", "strict_correct": true, "derivation_final": 6}, "stepsolver|E06": {"status": "valid", "strict_correct": true, "derivation_final": 8}, "stepsolver|E07": {"status": "valid", "strict_correct": true, "derivation_final": 8}, "stepsolver|E08": {"status": "valid", "strict_correct": true, "derivation_final": 8}, "stepsolver|F01": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|F02": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|F03": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|F04": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|F05": {"status": "valid", "strict_correct": true, "derivation_final": 7}, "stepsolver|F06": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|G01": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|G02": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|G03": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|G04": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "stepsolver|G05": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|G06": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|H01": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|H02": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|H03": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|H04": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|I01": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|I02": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "stepsolver|I03": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "stepsolver|I04": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "stepsolver|J01": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "stepsolver|J02": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "stepsolver|J03": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|J04": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|K01": {"status": "valid", "strict_correct": true, "derivation_final": 7}, "stepsolver|K02": {"status": "invalid_answer", "strict_correct": false, "derivation_final": 4}, "stepsolver|K03": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "stepsolver|K04": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|L01": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|L02": {"status": "valid", "strict_correct": true, "derivation_final": 2}, "stepsolver|L03": {"status": "valid", "strict_correct": true, "derivation_final": 8}, "stepsolver|L04": {"status": "valid", "strict_correct": true, "derivation_final": null}}}''')
print('wrote', 5, 'files')

In [ ]:
#@title 3b. Benchmark source: worker protocol + adapters (SymPy, SmartSolver, StepSolver) { display-mode: "form" }
W('bench/__init__.py', r'''''')

W('bench/workers/__init__.py', r'''''')

W('bench/workers/common.py', r'''"""Shared worker protocol (stdlib only). One JSON request per line on stdin, one JSON reply per line on stdout.

A worker module defines:  NAME, info() -> dict, solve_case(case: dict) -> dict (see schema in docs/METHODOLOGY.md)
The worker measures ONLY the solver call (perf_counter around it); adapter (de)serialisation is excluded.
Solver stdout/stderr noise is redirected so it cannot corrupt the protocol.
"""
import contextlib
import io
import json
import os
import resource
import sys
import time
import traceback

PROTO_OUT = sys.stdout   # keep the real stdout for protocol messages


def timed(fn, *a, **kw):
    t = time.perf_counter()
    try:
        r = fn(*a, **kw)
        return r, (time.perf_counter() - t) * 1000.0, None
    except BaseException as e:   # noqa: BLE001  (RecursionError etc. must be captured, not crash the worker)
        if isinstance(e, (KeyboardInterrupt, SystemExit)):
            raise
        return None, (time.perf_counter() - t) * 1000.0, e


def err_dict(e):
    tb = traceback.format_exception(type(e), e, e.__traceback__)
    return {"type": type(e).__name__, "message": str(e)[:500], "traceback_tail": "".join(tb[-4:])[-1200:]}


def serve(mod):
    t_boot = time.perf_counter()
    for line in sys.stdin:
        line = line.strip()
        if not line:
            continue
        req = json.loads(line)
        cmd = req.get("cmd")
        if cmd == "shutdown":
            break
        sink = io.StringIO()
        try:
            with contextlib.redirect_stdout(sink), contextlib.redirect_stderr(sink):
                if cmd == "info":
                    out = mod.info()
                elif cmd == "ping":
                    out = {"pong": True}
                elif cmd == "clear_cache":
                    try:
                        from sympy.core.cache import clear_cache
                        clear_cache()
                        out = {"cleared": True}
                    except Exception as ce:   # noqa: BLE001
                        out = {"cleared": False, "error": str(ce)}
                elif cmd == "solve":
                    out = mod.solve_case(req["case"])
                elif cmd == "raw":     # engineering tests: solver-specific probe
                    out = mod.raw_probe(req["probe"])
                else:
                    out = {"error": {"type": "BadCommand", "message": str(cmd)}}
        except BaseException as e:   # noqa: BLE001
            out = {"outcome": "error", "error": err_dict(e), "worker_level_failure": True}
        out["req_id"] = req.get("req_id")
        out["maxrss_kb"] = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
        if sink.getvalue():
            out["captured_stdout_stderr"] = sink.getvalue()[:2000]
        PROTO_OUT.write(json.dumps(out, default=str, ensure_ascii=True) + "\n")
        PROTO_OUT.flush()
''')

W('bench/workers/safeinput.py', r'''"""Allow-list validation for benchmark-controlled math strings (never executes anything)."""
import re

_OK = re.compile(r"^[A-Za-z0-9_+\-*/^().,=<>!&| \[\]]*$")
_ATTR = re.compile(r"[A-Za-z_\)\]]\s*\.\s*[A-Za-z_]")


def check(s):
    if not isinstance(s, str) or len(s) > 400:
        raise ValueError("input rejected: not a short string")
    if "__" in s or not _OK.match(s) or _ATTR.search(s):
        raise ValueError("input rejected by allow-list: %r" % s[:80])
    return s
''')

W('bench/workers/sympy_worker.py', r'''"""Standalone SymPy adapter (configuration 'sympy-direct').

Design: SymPy is evaluated as a BARE ENGINE through its public API, exactly as an application developer
would call it, with NO added step generator and NO added domain-tracking layer.  Consequences (documented,
not hidden): SymPy provides no student-facing derivation (except sympy.integrals.manualintegrate.integral_steps
for integrals, captured as-is), and reports no domain restrictions unless the API returns them.

API choices (declared before execution):
  evaluate/simplify -> simplify      expand -> expand      factor -> factor
  solve             -> solveset(Eq, x, S.Reals | S.Complexes per case.domain)   [explicit domain]
  solve_system      -> solve(..., dict=True)          solve_inequality -> solve_univariate_inequality
  diff/partial/grad -> diff                           integrate -> integrate (indefinite & definite)
  limit             -> limit(dir='+-')                det/inverse/rref -> Matrix.det/.inv/.rref
  dsolve            -> dsolve                         partial_fractions -> apart
  nsolve            -> nsolve (K03, guess given) / Poly.nroots (K04)
Symbols are created real=True when case.domain == 'real' (explicit domain assumption), plain otherwise.
"""
import sys

import sympy as sp
from sympy.parsing.sympy_parser import convert_xor, parse_expr, standard_transformations

try:
    from . import common, safeinput
except ImportError:  # executed as a script
    import common
    import safeinput

NAME = "sympy-direct"
_TR = standard_transformations + (convert_xor,)
_ALLOWED = ["Integer", "Float", "Rational", "Symbol", "Function", "Mul", "Add", "Pow", "Derivative", "Eq",
            "sin", "cos", "tan", "exp", "log", "sqrt", "Abs", "pi", "E", "I", "oo"]


def _ns(extra=None):
    g = {k: getattr(sp, k) for k in _ALLOWED}
    g["__builtins__"] = {}
    g["abs"] = sp.Abs
    g["ln"] = sp.log
    g["e"] = sp.E      # Euler's number: SymPy's parser would otherwise read 'e' as a plain symbol
    g["i"] = sp.I
    if extra:
        g.update(extra)
    return g


_PARSE_MS = [0.0]


def parse(s, syms, extra=None):
    import time as _t
    _t0 = _t.perf_counter()
    try:
        return _parse(s, syms, extra)
    finally:
        _PARSE_MS[0] += (_t.perf_counter() - _t0) * 1000.0


def _parse(s, syms, extra=None):
    safeinput.check(s)
    loc = {k: v for k, v in syms.items() if not (extra and k in extra)}   # function names (y) must not be shadowed by Symbols
    return parse_expr(s, local_dict=loc, global_dict=_ns(extra), transformations=_TR, evaluate=True)


def info():
    return {"solver": NAME, "version": sp.__version__, "python": sys.version.split()[0],
            "commit": None, "package": "sympy", "api_methods": ["simplify", "expand", "factor", "solveset", "solve",
            "solve_univariate_inequality", "diff", "integrate", "limit", "Matrix.det/inv/rref", "dsolve", "apart",
            "nsolve", "Poly.nroots"]}


def _syms(case, names):
    real = case["domain"] == "real"
    return {n: sp.Symbol(n, real=True) if real else sp.Symbol(n) for n in names}


def _expr_answer(e):
    return {"kind": "expr", "text": str(e)}


def _fmt_set(s):
    """Convert a SymPy set to the canonical, string-free structure."""
    if s == sp.S.EmptySet:
        return {"kind": "empty"}
    if s in (sp.S.Reals, sp.S.Complexes):
        return {"kind": "all", "domain": str(s)}
    if isinstance(s, sp.FiniteSet):
        return {"kind": "solutions", "values": [str(v) for v in s]}
    if isinstance(s, sp.Interval):
        return {"kind": "intervals", "intervals": [[str(s.start), str(s.end), bool(s.left_open), bool(s.right_open)]]}
    if isinstance(s, sp.Union) and all(isinstance(a, sp.Interval) for a in s.args):
        return {"kind": "intervals", "intervals": [[str(a.start), str(a.end), bool(a.left_open), bool(a.right_open)] for a in s.args]}
    if isinstance(s, sp.ConditionSet):
        return {"kind": "condition_set", "text": str(s)}
    return {"kind": "other_set", "text": str(s)}


def solve_case(case):
    op, pb = case["operation"], case["problem"]
    _PARSE_MS[0] = 0.0
    res = {"solver": NAME, "restrictions_reported": [], "steps": [], "native_flags": {},
           "domain_assumption_passed": True, "input_overrides_used": False}
    symnames = ["x", "y"]
    S = _syms(case, symnames)
    x = S["x"]
    out = lambda **k: {**res, **k}

    def run(call_desc, fn):
        r, ms, e = common.timed(fn)
        res["api_call"] = call_desc
        res["time_ms"] = ms + _PARSE_MS[0]     # parse time included, as for the other two solvers
        return r, e

    if op in ("evaluate", "simplify"):
        expr = parse(pb["expr"], S)
        r, e = run("sympy.simplify(parse(expr))", lambda: sp.simplify(expr))
        if e: return out(outcome="error", error=common.err_dict(e))
        return out(outcome="answer", raw_output=str(r), answer=_expr_answer(r), input_sent=pb["expr"], native_status="ok")
    if op in ("expand", "factor"):
        expr = parse(pb["expr"], S)
        fn = sp.expand if op == "expand" else sp.factor
        r, e = run("sympy.%s(parse(expr))" % op, lambda: fn(expr))
        if e: return out(outcome="error", error=common.err_dict(e))
        return out(outcome="answer", raw_output=str(r), answer=_expr_answer(r), input_sent=pb["expr"], native_status="ok")
    if op == "solve":
        lhs, rhs = [parse(t, S) for t in pb["equation"].split("=", 1)]
        dom = sp.S.Reals if case["domain"] == "real" else sp.S.Complexes
        r, e = run("sympy.solveset(Eq(lhs, rhs), x, %s)" % dom, lambda: sp.solveset(sp.Eq(lhs, rhs), x, dom))
        if e: return out(outcome="error", error=common.err_dict(e))
        ans = _fmt_set(r)
        if ans["kind"] == "condition_set":
            return out(outcome="unsupported", unsupported_reason="solveset returned an unevaluated ConditionSet (SymPy's declared 'cannot solve')",
                       raw_output=str(r), answer=ans, input_sent=pb["equation"], native_status="ConditionSet")
        return out(outcome="answer", raw_output=str(r), answer=ans, input_sent=pb["equation"], native_status="ok")
    if op == "solve_system":
        eqs = []
        for t in pb["equations"]:
            l, rr = [parse(s, S) for s in t.split("=", 1)]
            eqs.append(sp.Eq(l, rr))
        r, e = run("sympy.solve(eqs, [x, y], dict=True)", lambda: sp.solve(eqs, [S["x"], S["y"]], dict=True))
        if e: return out(outcome="error", error=common.err_dict(e))
        if not r:
            ans = {"kind": "empty"}
        else:
            ans = {"kind": "system", "solutions": [{str(k): str(v) for k, v in d.items()} for d in r]}
        return out(outcome="answer", raw_output=str(r), answer=ans, input_sent=pb["equations"], native_status="ok")
    if op == "solve_inequality":
        s = pb["inequality"]
        for sym in (">", "<"):
            if sym in s:
                l, rr = [parse(t, S) for t in s.split(sym, 1)]
                rel = l > rr if sym == ">" else l < rr
        r, e = run("sympy.solve_univariate_inequality(rel, x, relational=False)",
                   lambda: sp.solve_univariate_inequality(rel, x, relational=False))
        if e: return out(outcome="error", error=common.err_dict(e))
        return out(outcome="answer", raw_output=str(r), answer=_fmt_set(r), input_sent=s, native_status="ok")
    if op in ("diff", "partial_derivative"):
        expr = parse(pb["expr"], S)
        v = S[pb["var"]]
        r, e = run("sympy.diff(expr, %s)" % pb["var"], lambda: sp.diff(expr, v))
        if e: return out(outcome="error", error=common.err_dict(e))
        return out(outcome="answer", raw_output=str(r), answer=_expr_answer(r), input_sent=pb["expr"], native_status="ok")
    if op == "gradient":
        expr = parse(pb["expr"], S)
        r, e = run("[sympy.diff(expr, v) for v in vars]", lambda: [sp.diff(expr, S[v]) for v in pb["vars"]])
        if e: return out(outcome="error", error=common.err_dict(e))
        return out(outcome="answer", raw_output=str(r), answer={"kind": "vector", "items": [str(i) for i in r]},
                   input_sent=pb["expr"], native_status="ok")
    if op == "integrate":
        expr = parse(pb["expr"], S)
        definite = "lower" in pb
        if definite:
            lo, hi = parse(pb["lower"], S), parse(pb["upper"], S)
            r, e = run("sympy.integrate(expr, (x, lo, hi))", lambda: sp.integrate(expr, (x, lo, hi)))
        else:
            r, e = run("sympy.integrate(expr, x)", lambda: sp.integrate(expr, x))
        if e: return out(outcome="error", error=common.err_dict(e))
        # SymPy's own documented step API (rule tree); captured as-is, NOT paraphrased or enriched.
        try:
            from sympy.integrals.manualintegrate import integral_steps
            tree = repr(integral_steps(expr, x))
            res["steps"] = [{"rule": "sympy.integral_steps (raw rule tree)", "description": tree[:4000], "before": str(expr),
                             "after": "", "notes": [], "verification": "", "constraints": []}]
            res["native_flags"]["workings_source"] = "sympy.integrals.manualintegrate.integral_steps"
            res["native_flags"]["workings_tied_to_answer"] = False   # integrate() and integral_steps() are separate code paths
        except Exception as ex:   # noqa: BLE001
            res["native_flags"]["workings_error"] = type(ex).__name__
        return out(outcome="answer", raw_output=str(r), answer=_expr_answer(r), input_sent=pb["expr"], native_status="ok")
    if op == "limit":
        expr = parse(pb["expr"], S)
        pt = parse(pb["point"], S)
        r, e = run("sympy.limit(expr, x, point, dir='+-')", lambda: sp.limit(expr, x, pt, dir="+-"))
        if e: return out(outcome="error", error=common.err_dict(e))
        return out(outcome="answer", raw_output=str(r), answer=_expr_answer(r), input_sent=pb["expr"], native_status="ok")
    if op in ("det", "inverse", "rref"):
        M = sp.Matrix(pb["matrix"])
        fn = {"det": lambda: M.det(), "inverse": lambda: M.inv(), "rref": lambda: M.rref()[0]}[op]
        r, e = run("sympy.Matrix(...).%s" % {"det": "det()", "inverse": "inv()", "rref": "rref()[0]"}[op], fn)
        if e: return out(outcome="error", error=common.err_dict(e))
        if op == "det":
            return out(outcome="answer", raw_output=str(r), answer=_expr_answer(r), input_sent=pb["matrix"], native_status="ok")
        return out(outcome="answer", raw_output=str(r), answer={"kind": "matrix", "rows": [[str(c) for c in r.row(i)] for i in range(r.rows)]},
                   input_sent=pb["matrix"], native_status="ok")
    if op == "dsolve":
        f = sp.Function(pb["func"])
        ode = parse(pb["ode"], S, extra={pb["func"]: f})
        r, e = run("sympy.dsolve(ode, y(x))", lambda: sp.dsolve(ode, f(x)))
        if e: return out(outcome="error", error=common.err_dict(e))
        return out(outcome="answer", raw_output=str(r), answer={"kind": "equation", "text": str(r)}, input_sent=pb["ode"], native_status="ok")
    if op == "partial_fractions":
        expr = parse(pb["expr"], S)
        r, e = run("sympy.apart(expr, x)", lambda: sp.apart(expr, x))
        if e: return out(outcome="error", error=common.err_dict(e))
        return out(outcome="answer", raw_output=str(r), answer=_expr_answer(r), input_sent=pb["expr"], native_status="ok")
    if op == "nsolve":
        l, rr = [parse(t, S) for t in pb["equation"].split("=", 1)]
        f = l - rr
        if pb.get("guess") is not None:
            r, e = run("sympy.nsolve(lhs-rhs, x, guess, prec=30)", lambda: sp.nsolve(f, x, sp.Float(pb["guess"]), prec=30))
            vals = [str(r)] if not e else []
        else:
            r, e = run("sympy.Poly(lhs-rhs, x).nroots(n=30)", lambda: sp.Poly(f, x).nroots(n=30))
            vals = [str(v) for v in r] if not e else []
        if e: return out(outcome="error", error=common.err_dict(e))
        return out(outcome="answer", raw_output=str(r), answer={"kind": "numeric", "values": vals}, input_sent=pb["equation"], native_status="ok")
    return out(outcome="unsupported", unsupported_reason="adapter has no mapping for operation %r" % op)


def raw_probe(p):
    """Engineering probes (see bench/engineering.py). Probe inputs are benchmark-authored strings."""
    kind = p["kind"]
    x = sp.Symbol("x")
    if kind == "solve_vs_solveset":
        eq = sp.sympify(p["lhs"], locals={"x": x}) - sp.sympify(p["rhs"], locals={"x": x})
        return {"outcome": "answer", "solve": str(sp.solve(eq, x)), "solveset": str(sp.solveset(eq, x, sp.S.Reals))}
    if kind == "parse":
        safeinput.check(p["text"])
        r, ms, e = common.timed(lambda: parse_expr(p["text"], local_dict={"x": x}, global_dict=_ns(), transformations=_TR))
        if e:
            return {"outcome": "error", "error": common.err_dict(e), "time_ms": ms}
        return {"outcome": "answer", "raw_output": str(r), "time_ms": ms}
    if kind == "eval":      # evaluate through the allow-listed adapter parse
        r, ms, e = common.timed(lambda: sp.simplify(parse(p["text"], {"x": x})))
        if e:
            return {"outcome": "error", "error": common.err_dict(e), "time_ms": ms}
        return {"outcome": "answer", "raw_output": str(r), "time_ms": ms}
    if kind == "solve_for":
        syms = {"x": x, "y": sp.Symbol("y"), "z": sp.Symbol("z")}
        l, rr = [parse(s, syms) for s in p["equation"].split("=", 1)]
        v = syms[p["var"]]
        r, ms, e = common.timed(lambda: sp.solveset(sp.Eq(l, rr), v, sp.S.Reals))
        if e:
            return {"outcome": "error", "error": common.err_dict(e), "time_ms": ms}
        return {"outcome": "answer", "raw_output": str(r), "answer_values": [str(i) for i in r] if isinstance(r, sp.FiniteSet) else None, "time_ms": ms}
    if kind == "canary":    # engine's OWN string interface, as shipped (sympy.sympify); benign payload: math.factorial(4)
        r, ms, e = common.timed(lambda: sp.sympify(p["text"]))
        if e:
            return {"outcome": "error", "error": common.err_dict(e), "time_ms": ms}
        return {"outcome": "answer", "raw_output": str(r), "time_ms": ms}
    return {"outcome": "error", "error": {"type": "UnknownProbe", "message": kind}}


if __name__ == "__main__":
    common.serve(sys.modules[__name__])
''')

W('bench/workers/smartsolver_worker.py', r'''"""SmartSolver adapter (package 'ed-master-smartsolver', import name 'smartsolver').

Mapping (declared before execution; one documented public method per operation family):
  solve (C, K02, K03, K04, L01, L02, L04) -> SmartSolver.solve_equation(equation, 'x')
  solve_system (D01-D03, I01)             -> SmartSolver.solve_linear_system(equations, variables)
  diff (F01-F06)                          -> SmartSolver.differentiate(expr, 'x')
  partial_derivative (J03)                -> SmartSolver.partial_derivative(expr, 'x', ['x','y'])
  gradient (J04)                          -> SmartSolver.gradient(expr, vars)
  integrate (G01-G06)                     -> SmartSolver.integrate(expr, 'x'[, lower, upper])
  limit (H01-H04)                         -> SmartSolver.limit(expr, 'x', point)      (documented default direction '+')
  rref (I04)                              -> SmartSolver.matrix_rref(matrix)
  partial_fractions (K01)                 -> SmartSolver.partial_fractions(expr)
Operations with NO public method (evaluate, simplify, expand, factor, solve_inequality, det, inverse, dsolve,
nsolve) are reported as UNSUPPORTED with the introspected method list as evidence; nothing is manufactured.
K03/K04 (numeric root finding) have no numeric API, so the generic equation solver is used (documented in cases.json).
SmartSolver result objects are converted with str(); steps with the library's own step_to_dict / StepRenderer.
No answer is modified.
"""
import re
import sys

import smartsolver
from smartsolver import SmartSolver, StepRenderer, step_to_dict

try:
    from . import common
except ImportError:
    import common

NAME = "smartsolver"
S = SmartSolver()
PUBLIC = sorted(m for m in dir(SmartSolver) if not m.startswith("_") and callable(getattr(SmartSolver, m)))


def info():
    import importlib.metadata as md
    dist = (md.packages_distributions().get("smartsolver") or ["ed-master-smartsolver"])[0]
    return {"solver": NAME, "version": md.version(dist), "package": dist,
            "module_version": getattr(smartsolver, "__version__", None), "python": sys.version.split()[0], "commit": None,
            "sympy_version": md.version("sympy"), "public_methods": PUBLIC}


def _steps(r):
    out = []
    for s in r.get("steps", []):
        d = step_to_dict(s)
        out.append({"rule": d.get("rule_applied", ""), "description": d.get("description", ""), "before": "",
                    "after": d.get("expression", ""), "notes": [], "verification": "", "constraints": [],
                    "substeps": d.get("substeps", [])})
    return out


def _restrictions(steps):
    out = []
    for s in steps:
        if s["rule"] in ("Domain", "Extraneous root") or re.search(r"reject|undefined|denominator|must be positive", s["description"], re.I):
            out.append(s["description"])
    return out


UNSUPPORTED = {"evaluate", "simplify", "expand", "factor", "solve_inequality", "det", "inverse", "dsolve", "nsolve_api"}


def solve_case(case):
    op, pb = case["operation"], case["problem"]
    base = {"solver": NAME, "domain_assumption_passed": False, "input_overrides_used": False}   # SmartSolver has no domain parameter

    def unsupported(why):
        return {**base, "outcome": "unsupported", "unsupported_reason": why, "api_call": None, "steps": [], "restrictions_reported": [],
                "native_flags": {"public_methods_checked": PUBLIC}}

    call = None
    if op == "solve":
        call = ("solve_equation(%r, 'x')" % pb["equation"], lambda: S.solve_equation(pb["equation"], "x"), "solutions")
    elif op == "nsolve":
        call = ("solve_equation(%r, 'x')  [no numeric API; generic equation solver]" % pb["equation"],
                lambda: S.solve_equation(pb["equation"], "x"), "solutions")
    elif op == "solve_system":
        call = ("solve_linear_system(%r, %r)" % (pb["equations"], pb["vars"]),
                lambda: S.solve_linear_system(pb["equations"], pb["vars"]), "system")
    elif op == "diff":
        call = ("differentiate(%r, %r)" % (pb["expr"], pb["var"]), lambda: S.differentiate(pb["expr"], pb["var"]), "derivative")
    elif op == "partial_derivative":
        call = ("partial_derivative(%r, %r, %r)" % (pb["expr"], pb["var"], pb["vars"]),
                lambda: S.partial_derivative(pb["expr"], pb["var"], pb["vars"]), "derivative")
    elif op == "gradient":
        call = ("gradient(%r, %r)" % (pb["expr"], pb["vars"]), lambda: S.gradient(pb["expr"], pb["vars"]), "gradient")
    elif op == "integrate":
        if "lower" in pb:
            call = ("integrate(%r, 'x', lower=%r, upper=%r)" % (pb["expr"], pb["lower"], pb["upper"]),
                    lambda: S.integrate(pb["expr"], "x", lower=pb["lower"], upper=pb["upper"]), "integral")
        else:
            call = ("integrate(%r, 'x')" % pb["expr"], lambda: S.integrate(pb["expr"], "x"), "integral")
    elif op == "limit":
        call = ("limit(%r, 'x', point=%r)  [documented default direction '+']" % (pb["expr"], pb["point"]),
                lambda: S.limit(pb["expr"], "x", point=pb["point"]), "limit")
    elif op == "rref":
        call = ("matrix_rref(%r)" % (pb["matrix"],), lambda: S.matrix_rref(pb["matrix"]), "matrix")
    elif op == "partial_fractions":
        call = ("partial_fractions(%r)" % pb["expr"], lambda: S.partial_fractions(pb["expr"]), "decomposition")
    else:
        return unsupported("SmartSolver has no public method for operation %r (public methods: %s)" % (op, ", ".join(PUBLIC)))

    desc, fn, kind = call
    method_name = desc.split("(")[0]
    if not hasattr(S, method_name):          # a newer/older version may have removed or renamed the method
        return unsupported("SmartSolver has no public method %r in this version (public methods: %s)" % (method_name, ", ".join(PUBLIC)))
    r, ms, e = common.timed(fn)
    res = {**base, "api_call": desc, "time_ms": ms, "input_sent": pb}
    if e is not None:
        return {**res, "outcome": "error", "error": common.err_dict(e), "steps": [], "restrictions_reported": []}
    steps = _steps(r)
    flags = {"steps_verified": r.get("steps_verified"), "steps_mode": r.get("steps_mode"),
             "consistent": r.get("consistent"), "methods": [str(m) for m in r.get("methods", [])],
             "result_keys": sorted(k for k in r if k != "steps")}
    raw = {k: str(v) for k, v in r.items() if k != "steps"}
    raw["steps_text"] = StepRenderer.to_text(r.get("steps", []))
    res.update(outcome="answer", steps=steps, restrictions_reported=_restrictions(steps), native_flags=flags, raw_output=raw, native_status="ok")
    if kind == "solutions":
        vals = [str(v) for v in r.get("solutions", [])]
        ans = {"kind": "solutions", "values": vals} if vals else {"kind": "empty"}
        if r.get("general_solutions") is not None:
            ans["general_solutions"] = str(r["general_solutions"])
    elif kind == "system":
        by = r.get("solutions_by_variable") or {}
        if r.get("consistent") is False:
            ans = {"kind": "empty"}
        elif by:
            ans = {"kind": "system", "solutions": [{k: str(v) for k, v in by.items()}]}
        else:
            ans = {"kind": "empty"} if not r.get("solutions") else {"kind": "other", "text": str(r.get("solutions"))}
    elif kind == "derivative":
        ans = {"kind": "expr", "text": str(r["derivative"])}
    elif kind == "gradient":
        g = r["gradient"]
        ans = {"kind": "vector", "items": [str(i) for i in list(g)]}
    elif kind == "integral":
        ans = {"kind": "expr", "text": str(r["integral"])}
    elif kind == "limit":
        ans = {"kind": "expr", "text": str(r["limit"])}
    elif kind == "matrix":
        m = r["matrix"]
        ans = {"kind": "matrix", "rows": [[str(m[i, j]) for j in range(m.cols)] for i in range(m.rows)]}
    elif kind == "decomposition":
        ans = {"kind": "expr", "text": str(r["decomposition"])}
    return {**res, "answer": ans}


def raw_probe(p):
    kind = p["kind"]
    if kind == "call":
        fn = getattr(S, p["method"])
        r, ms, e = common.timed(lambda: fn(*p.get("args", []), **p.get("kwargs", {})))
        if e:
            return {"outcome": "error", "error": common.err_dict(e), "time_ms": ms}
        return {"outcome": "answer", "raw_output": {k: str(v)[:300] for k, v in r.items() if k != "steps"}, "time_ms": ms,
                "nsteps": len(r.get("steps", []))}
    return {"outcome": "error", "error": {"type": "UnknownProbe", "message": kind}}


if __name__ == "__main__":
    common.serve(sys.modules[__name__])
''')

W('bench/workers/stepsolver_worker.py', r'''"""StepSolver adapter (package 'stepsolver', Solver().solve(<ASCII query>)).

Mapping (declared before execution) -- StepSolver's documented ASCII operations:
  evaluate/simplify -> simplify(e)    expand -> expand(e)     factor -> factor(e)
  solve / K03 / K04 -> solve(eq, x)   (K03/K04: no numeric root-finder exists; generic solver used)
  solve_system      -> solve([eq1, eq2], [x, y])    solve_inequality -> solve_inequality(ineq, x)
  diff/partial      -> diff(e, x)     gradient -> TWO native diff(e, v) calls (no gradient op; composition is
                       definitional: gradient == vector of partials; flagged composed_from)
  integrate         -> integrate(e, x) / integrate(e, x, a, b)      limit -> limit(e, x, p)
  det/inverse/rref  -> det(matrix(..)) / inverse(matrix(..)) / rref(matrix(..))
  dsolve            -> dsolve(<ode in StepSolver syntax from cases.json adapter_inputs>, y(x))
  partial_fractions -> apart(e, x)
Serialisation: answers are read from StepSolver's typed MathValue tree with its own format_expression(); steps from
SolutionStep fields (rule, explanation, before, after, notes, verification, introduced_constraints).  Nothing is
rewritten.  Parse errors are 'input_incompatible'; UnsolvedResult is 'unsupported' (the solver's own declaration).
"""
import sys

import stepsolver
from stepsolver import (DivergentResult, ExactResult, MappingValue, MatrixValue, NoSolutionValue, ScalarValue,
                        SequenceValue, BooleanValue, Solver, UndefinedResult, UnsolvedResult, format_ascii,
                        format_expression, ParseError, StepSolverError)

try:
    from . import common
except ImportError:
    import common

NAME = "stepsolver"
SOLVER = Solver()


def info():
    import importlib.metadata as md
    dist = (md.packages_distributions().get("stepsolver") or ["stepsolver"])[0]
    return {"solver": NAME, "version": md.version(dist), "package": dist, "python": sys.version.split()[0],
            "commit": None, "sympy_version": md.version("sympy"), "operations": [o.value for o in stepsolver.Operation]}


def _mat(rows):
    return "matrix([" + ", ".join("[" + ", ".join(str(c) for c in r) + "]" for r in rows) + "])"


def _ln(q, on):
    """StepSolver has no 'ln' alias: its natural log token is 'log' (an unknown 'ln' is silently accepted as an opaque
    function).  Adapter normalisation ln( -> log( is applied (documented) and the as-specified query is ALSO run and
    recorded as 'as_specified_variant' so the silent-misparse hazard stays visible."""
    return q.replace("ln(", "log(") if on else q


def build_queries(case, normalise_ln=True):
    qs, ov = _build(case)
    return [_ln(q, normalise_ln) for q in qs] if qs else qs, ov


def _build(case):
    op, pb = case["operation"], case["problem"]
    ov = case.get("adapter_inputs", {}).get("stepsolver")
    if ov:
        return [ov], True
    if op in ("evaluate", "simplify"):
        return ["simplify(%s)" % pb["expr"]], False
    if op in ("expand", "factor"):
        return ["%s(%s)" % (op, pb["expr"])], False
    if op in ("solve", "nsolve"):
        return ["solve(%s, x)" % pb["equation"]], False
    if op == "solve_system":
        return ["solve([%s], [%s])" % (", ".join(pb["equations"]), ", ".join(pb["vars"]))], False
    if op == "solve_inequality":
        return ["solve_inequality(%s, x)" % pb["inequality"]], False
    if op in ("diff", "partial_derivative"):
        return ["diff(%s, %s)" % (pb["expr"], pb["var"])], False
    if op == "gradient":
        return ["diff(%s, %s)" % (pb["expr"], v) for v in pb["vars"]], False
    if op == "integrate":
        if "lower" in pb:
            return ["integrate(%s, x, %s, %s)" % (pb["expr"], pb["lower"], pb["upper"])], False
        return ["integrate(%s, x)" % pb["expr"]], False
    if op == "limit":
        return ["limit(%s, x, %s)" % (pb["expr"], pb["point"])], False
    if op in ("det", "inverse", "rref"):
        return ["%s(%s)" % (op, _mat(pb["matrix"]))], False
    if op == "partial_fractions":
        return ["apart(%s, x)" % pb["expr"]], False
    return None, False


def _val(v, op):
    """Typed MathValue -> canonical answer dict (no string re-interpretation)."""
    if isinstance(v, NoSolutionValue):
        return {"kind": "empty"}
    if isinstance(v, BooleanValue):
        return {"kind": "bool", "value": v.value}
    if isinstance(v, MatrixValue):
        return {"kind": "matrix", "rows": [[format_expression(c) for c in r] for r in v.rows]}
    if isinstance(v, ScalarValue):
        return {"kind": "expr", "text": format_expression(v.expression)}
    if isinstance(v, MappingValue):
        return {"kind": "mapping", "entries": {format_expression(e.key): _val(e.value, op) for e in v.entries}}
    if isinstance(v, SequenceValue):
        return {"kind": "sequence", "items": [_val(i, op) for i in v.items]}
    return {"kind": "unknown", "text": repr(v)[:300]}


def _canon(av, op):
    """Flatten sequence/mapping answers into the benchmark's canonical kinds, documenting every rule."""
    if op in ("solve", "nsolve") and av["kind"] == "sequence":
        vals = []
        for it in av["items"]:
            if it["kind"] == "mapping" and len(it["entries"]) == 1:
                e = list(it["entries"].values())[0]
                vals.append(e.get("text", str(e)))
            else:
                return {"kind": "other", "detail": av}
        return {"kind": "solutions", "values": vals} if vals else {"kind": "empty"}
    if op == "solve_system" and av["kind"] == "sequence":
        sols = []
        for it in av["items"]:
            if it["kind"] == "mapping":
                sols.append({k: v.get("text", str(v)) for k, v in it["entries"].items()})
            else:
                return {"kind": "other", "detail": av}
        return {"kind": "system", "solutions": sols} if sols else {"kind": "empty"}
    if op == "rref" and av["kind"] == "sequence" and av["items"] and av["items"][0]["kind"] == "matrix":
        return av["items"][0]      # (matrix, pivots) -> matrix ; pivots dropped (documented)
    if op == "solve_inequality" and av["kind"] == "expr":
        return {"kind": "relation", "text": av["text"]}
    if op == "dsolve" and av["kind"] == "expr":
        return {"kind": "equation", "text": av["text"]}
    return av


def _steps(res):
    out = []
    for s in res.steps:
        out.append({"rule": s.rule, "description": s.explanation, "before": format_expression(s.before),
                    "after": format_expression(s.after),
                    "notes": [[n.label, format_expression(n.expression)] for n in s.notes],
                    "verification": "%s: %s" % (s.verification.method.value, s.verification.detail),
                    "constraints": [[c.explanation, format_expression(c.expression)] for c in s.introduced_constraints]})
    return out


def solve_case(case):
    queries, overridden = build_queries(case)
    raw_queries, _ = build_queries(case, normalise_ln=False)
    ln_normalised = bool(queries) and queries != raw_queries
    overridden = overridden or ln_normalised
    base = {"solver": NAME, "domain_assumption_passed": False, "input_overrides_used": bool(overridden)}
    if queries is None:
        return {**base, "outcome": "unsupported", "unsupported_reason": "adapter has no mapping for operation %r" % case["operation"],
                "steps": [], "restrictions_reported": []}
    op = case["operation"]
    results, total_ms, raws = [], 0.0, []
    for q in queries:
        r, ms, e = common.timed(lambda q=q: SOLVER.solve(q))
        total_ms += ms
        if e is not None:
            kind = "input_incompatible" if isinstance(e, ParseError) else "error"
            err = common.err_dict(e)
            err["contract_exception"] = isinstance(e, StepSolverError)
            return {**base, "outcome": kind, "error": err, "api_call": "Solver().solve(%r)" % q, "input_sent": queries, "time_ms": total_ms,
                    "steps": [], "restrictions_reported": [], "raw_output": raws or None}
        results.append(r)
        raws.append(format_ascii(r))
    steps, restr, answers = [], [], []
    native = {"result_types": [type(r).__name__ for r in results], "composed_from": queries if len(queries) > 1 else None}
    for q, r in zip(queries, results):
        st = _steps(r)
        steps += st
        for s in st:
            restr += [c[0] + " | " + c[1] for c in s["constraints"]]
        if isinstance(r, UndefinedResult):
            restr.append("undefined: " + r.reason)
        native.setdefault("opaque_steps", 0)
        native["opaque_steps"] += sum(1 for s in st if s["rule"].strip().lower() == "compute exact result")
    if ln_normalised:
        native["input_normalisation"] = "ln( -> log(  (StepSolver natural-log token is 'log')"
        v, _, ve = common.timed(lambda: SOLVER.solve(raw_queries[0]))
        native["as_specified_variant"] = {"query": raw_queries[0], "result_type": type(v).__name__ if ve is None else type(ve).__name__,
                                          "raw_output": format_ascii(v) if ve is None else str(ve)[:200]}
    res = {**base, "api_call": " ; ".join("Solver().solve(%r)" % q for q in queries), "input_sent": queries, "time_ms": total_ms,
           "steps": steps, "restrictions_reported": restr, "native_flags": native, "raw_output": raws if len(raws) > 1 else raws[0]}
    r0 = results[0]
    for r in results:
        if isinstance(r, UnsolvedResult):
            return {**res, "outcome": "unsupported", "unsupported_reason": "UnsolvedResult: " + r.reason[:300], "native_status": "UnsolvedResult"}
    if len(results) > 1 and all(isinstance(r, ExactResult) for r in results):   # gradient composition
        items = [_val(r.value, op) for r in results]
        return {**res, "outcome": "answer", "native_status": "ExactResult", "answer": {"kind": "vector", "items": [i.get("text", str(i)) for i in items]}}
    if isinstance(r0, ExactResult):
        return {**res, "outcome": "answer", "native_status": "ExactResult", "answer": _canon(_val(r0.value, op), op)}
    if isinstance(r0, UndefinedResult):
        return {**res, "outcome": "answer", "native_status": "UndefinedResult", "answer": {"kind": "undefined", "reason": r0.reason}}
    if isinstance(r0, DivergentResult):
        return {**res, "outcome": "answer", "native_status": "DivergentResult", "answer": {"kind": "divergent", "reason": r0.reason, "divergence": r0.kind.value}}
    return {**res, "outcome": "error", "error": {"type": "UnexpectedResultType", "message": type(r0).__name__}}


def raw_probe(p):
    kind = p["kind"]
    if kind == "query":
        r, ms, e = common.timed(lambda: SOLVER.solve(p["query"]))
        if e:
            return {"outcome": "error", "error": {**common.err_dict(e), "contract_exception": isinstance(e, StepSolverError)}, "time_ms": ms}
        return {"outcome": "answer", "native_status": type(r).__name__, "raw_output": format_ascii(r), "time_ms": ms,
                "nsteps": len(r.steps)}
    return {"outcome": "error", "error": {"type": "UnknownProbe", "message": kind}}


if __name__ == "__main__":
    common.serve(sys.modules[__name__])
''')
print('wrote', 7, 'files')

In [ ]:
#@title 3c. Benchmark source: runner, judge, derivation rubric { display-mode: "form" }
W('bench/runner.py', r'''#!/usr/bin/env python3
"""Run the dataset against each solver (each in its own venv + worker process). Raw replies are preserved verbatim.

  python -m bench.runner [--solver sympy-direct|smartsolver|stepsolver ...] [--category A] [--case C05]
                         [--repeats 5] [--timeout 20] [--no-timing] [--out results/raw]

Protocol per solver:
  1. cold-start trials  : N fresh processes; time to ready (spawn->first reply to 'ping') and first solve of C01.
  2. functional pass    : each selected case once, in dataset order, per-case timeout (process killed + restarted on timeout).
  3. warm repeats       : for each case that produced an answer: `repeats` extra runs, each preceded by a SymPy cache clear
                          (untimed), same process -> warm-process / cold-cache latency; canonical outputs hashed (repeatability).
Same cases, same timeout, same machine, same protocol for every solver. No solver gets special pre-processing.
"""
import argparse
import hashlib
import json
import os
import pathlib
import select
import statistics
import subprocess
import sys
import tempfile
import time

ROOT = pathlib.Path(__file__).resolve().parent.parent
WORKERS = {
    "sympy-direct": ("sympy", ROOT / "bench/workers/sympy_worker.py"),
    "smartsolver": ("smartsolver", ROOT / "bench/workers/smartsolver_worker.py"),
    "stepsolver": ("stepsolver", ROOT / "bench/workers/stepsolver_worker.py"),
}


class Worker:
    def __init__(self, name):
        env_name, script = WORKERS[name]
        self.name = name
        self.py = str(ROOT / "envs" / env_name / "bin" / "python")
        self.script = str(script)
        self.proc = None
        self.spawn_to_ready_ms = None
        self.start()

    def start(self):
        self.cwd = tempfile.mkdtemp(prefix="bench_wd_")       # neutral working directory (no repo files visible)
        env = {"PATH": "/usr/bin:/bin", "PYTHONHASHSEED": "0", "PYTHONNOUSERSITE": "1", "PYTHONDONTWRITEBYTECODE": "1",
               "HOME": self.cwd, "LC_ALL": "C.UTF-8"}
        t = time.perf_counter()
        self.proc = subprocess.Popen([self.py, "-s", self.script], stdin=subprocess.PIPE, stdout=subprocess.PIPE,
                                     stderr=subprocess.PIPE, text=True, bufsize=1, cwd=self.cwd, env=env)
        r = self.call({"cmd": "ping"}, timeout=120)
        self.spawn_to_ready_ms = (time.perf_counter() - t) * 1000.0
        return r

    def call(self, req, timeout):
        """Returns (reply|None, status) where status in ok|timeout|crash."""
        try:
            self.proc.stdin.write(json.dumps(req) + "\n")
            self.proc.stdin.flush()
        except BrokenPipeError:
            return None, "crash"
        end = time.time() + timeout
        buf = ""
        while time.time() < end:
            rem = max(0.0, end - time.time())
            r, _, _ = select.select([self.proc.stdout], [], [], min(rem, 0.5))
            if r:
                line = self.proc.stdout.readline()
                if not line:
                    return None, "crash"
                try:
                    return json.loads(line), "ok"
                except json.JSONDecodeError:
                    buf += line
            elif self.proc.poll() is not None:
                return None, "crash"
        return None, "timeout"

    def kill(self):
        if self.proc and self.proc.poll() is None:
            self.proc.kill()
            try:
                self.proc.wait(5)
            except Exception:   # noqa: BLE001
                pass

    def stderr_tail(self):
        try:
            if self.proc.poll() is not None:
                return self.proc.stderr.read()[-1500:]
        except Exception:   # noqa: BLE001
            pass
        return ""

    def close(self):
        try:
            self.proc.stdin.write(json.dumps({"cmd": "shutdown"}) + "\n")
            self.proc.stdin.flush()
            self.proc.wait(5)
        except Exception:   # noqa: BLE001
            self.kill()


def canon_hash(reply):
    """Hash of the semantically relevant output (answer, steps, outcome) -- excludes timings/memory."""
    keep = {k: reply.get(k) for k in ("outcome", "answer", "steps", "restrictions_reported", "unsupported_reason", "raw_output")}
    err = reply.get("error")
    if err:
        keep["error"] = {k: err.get(k) for k in ("type", "message")}
    return hashlib.sha256(json.dumps(keep, sort_keys=True, default=str).encode()).hexdigest()[:16]


def run_solver(name, cases, repeats, timeout, timing, cold_trials=5):
    out = {"solver": name, "cold_start": [], "rows": {}, "info": None, "maxrss_kb": 0, "timeouts": 0, "crashes": 0}
    w = Worker(name)
    reply, _ = w.call({"cmd": "info"}, 60)
    out["info"] = reply
    c01 = next(c for c in json.load(open(ROOT / "data/cases.json"))["cases"] if c["id"] == "C01")
    if timing:
        for _ in range(cold_trials):                                  # cold-start trials: fresh process each time
            cw = Worker(name)
            t = time.perf_counter()
            r, st = cw.call({"cmd": "solve", "case": c01, "req_id": "cold"}, timeout)
            out["cold_start"].append({"spawn_to_ready_ms": cw.spawn_to_ready_ms, "first_solve_wall_ms": (time.perf_counter() - t) * 1000.0,
                                      "first_solve_inner_ms": (r or {}).get("time_ms"), "status": st})
            cw.close()
    for c in cases:
        t = time.perf_counter()
        r, st = w.call({"cmd": "solve", "case": c, "req_id": c["id"]}, timeout)
        wall = (time.perf_counter() - t) * 1000.0
        row = {"case_id": c["id"], "wall_ms": wall, "transport": st}
        if st == "ok":
            row["reply"] = r
            row["hash"] = canon_hash(r)
            out["maxrss_kb"] = max(out["maxrss_kb"], r.get("maxrss_kb", 0))
        else:
            row["reply"] = {"outcome": "timeout" if st == "timeout" else "infra_error", "solver": name,
                            "error": {"type": "WorkerTimeout" if st == "timeout" else "WorkerCrash",
                                      "message": "no reply within %ss" % timeout if st == "timeout" else w.stderr_tail()}}
            out["timeouts" if st == "timeout" else "crashes"] += 1
            w.kill()
            w = Worker(name)      # restart for the remaining cases (state of a killed process is not trusted)
        out["rows"][c["id"]] = row
    if timing and repeats:
        for c in cases:
            row = out["rows"][c["id"]]
            if row["transport"] != "ok" or row["reply"].get("outcome") != "answer":
                continue
            times, hashes = [], []
            for _ in range(repeats):
                w.call({"cmd": "clear_cache"}, 30)
                r, st = w.call({"cmd": "solve", "case": c, "req_id": c["id"] + "-rep"}, timeout)
                if st != "ok":
                    row.setdefault("repeat_failures", []).append(st)
                    if st != "ok":
                        w.kill(); w = Worker(name)
                    continue
                times.append(r.get("time_ms"))
                hashes.append(canon_hash(r))
            row["warm_times_ms"] = times
            row["repeat_hashes"] = hashes
            row["repeatable"] = len(set(hashes + [row["hash"]])) == 1 if hashes else None
    w.close()
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--solver", nargs="*", default=list(WORKERS))
    ap.add_argument("--category")
    ap.add_argument("--case", nargs="*")
    ap.add_argument("--repeats", type=int, default=5)
    ap.add_argument("--timeout", type=float, default=20.0)
    ap.add_argument("--no-timing", action="store_true")
    ap.add_argument("--out", default=str(ROOT / "results/raw"))
    a = ap.parse_args()
    data = json.load(open(ROOT / "data/cases.json"))
    cases = data["cases"]
    if a.category:
        cases = [c for c in cases if c["category"] == a.category.upper()]
    if a.case:
        cases = [c for c in cases if c["id"] in set(a.case)]
    pathlib.Path(a.out).mkdir(parents=True, exist_ok=True)
    print("cases loaded: %d (dataset total %d)" % (len(cases), data["count"]))
    for s in a.solver:
        t = time.time()
        res = run_solver(s, cases, a.repeats, a.timeout, not a.no_timing)
        res["protocol"] = {"repeats": a.repeats, "timeout_s": a.timeout, "cases_run": [c["id"] for c in cases],
                           "cpu_count": os.cpu_count(), "started_unix": t}
        (pathlib.Path(a.out) / ("%s.json" % s)).write_text(json.dumps(res, indent=1, default=str))
        oc = {}
        for r in res["rows"].values():
            o = r["reply"].get("outcome")
            oc[o] = oc.get(o, 0) + 1
        print("%-13s %5.1fs  outcomes=%s  maxrss=%.0fMB timeouts=%d crashes=%d" % (s, time.time() - t, oc, res["maxrss_kb"] / 1024, res["timeouts"], res["crashes"]))


if __name__ == "__main__":
    main()
''')

W('bench/judge.py', r'''"""Mathematical judge (runs in the controller env: SymPy used ONLY as a checker, never to produce a solver's answer).

Rules (declared in cases.json 'comparison' + here, applied identically to every solver):
  value        scalar equality: symbolic simplify(a-b)==0 OR high-precision numeric |a-b|<=tol  (numeric alone is flagged)
  expr_equiv   equivalence as functions: symbolic zero OR agreement at >=3 exact sample points (50-digit) [flagged if numeric only]
               + form check (below).  Complex/NaN at a real sample point is a MISMATCH (catches log(x) vs ln|x|).
  solution_set exact set equality over the case's declared domain (real|complex); multiplicity not part of correctness
  system_solution / inequality_set / matrix / gradient / ode_family / numeric_roots / antiderivative / undefined: see functions.
Form rules (uniform): simplify/evaluate -> structure of the RAW answer text must not be more than 1 operation more complex than the
  reference's raw structure (rejects echoing the input); named checks: expanded, factored, simplified_radical, no_fraction,
  reduced_fraction, partial_fractions.  The generic simplification-form rule was added after the first functional pass showed one solver
  echoing its input for E05; it is applied to all solvers, and results with/without form rules are both reported (sensitivity).
Domain rule: cases with required_restrictions need the solver to REPORT each restriction (constraint list or step text);
  strict_correct = value_correct AND form_ok AND domain_ok.  value-only correctness is reported alongside.
"""
import json
import re

import sympy as sp
from sympy.parsing.sympy_parser import convert_xor, parse_expr, standard_transformations

try:
    from .workers import safeinput
except ImportError:
    from workers import safeinput

TR = standard_transformations + (convert_xor,)
X, Y = sp.Symbol("x"), sp.Symbol("y")
XR, YR = sp.Symbol("x", real=True), sp.Symbol("y", real=True)
_ALLOWED = ["Integer", "Float", "Rational", "Symbol", "Function", "Mul", "Add", "Pow", "Derivative", "Eq", "sin", "cos", "tan", "exp", "log",
            "sqrt", "Abs", "pi", "E", "I", "oo", "zoo", "CRootOf", "Matrix", "Or", "And", "Not"]


def ns(extra=None):
    g = {k: getattr(sp, k) for k in _ALLOWED}
    g.update({"__builtins__": {}, "abs": sp.Abs, "ln": sp.log, "e": sp.E, "i": sp.I, "C": sp.Symbol("C")})
    if extra:
        g.update(extra)
    return g


def P(text, evaluate=True, extra=None, syms=None):
    """Restricted parse of solver-produced text (allow-list; no builtins; no attribute access)."""
    safeinput.check(text.replace("\n", " "))
    loc = dict(syms or {})
    return parse_expr(text, local_dict=loc, global_dict=ns(extra), transformations=TR, evaluate=evaluate)


def N(e, n=50):
    return sp.N(e, n)


def ops_raw(text):
    try:
        return int(sp.count_ops(P(text, evaluate=False)))
    except Exception:   # noqa: BLE001
        return 10 ** 6


def strip_constant(e):
    """Remove an additive integration constant C/C1 (recorded)."""
    consts = [s for s in e.free_symbols if re.fullmatch(r"C\d*", s.name)]
    if consts:
        return e.subs({c: 0 for c in consts}), True
    return e, False


def sample_pts(case, var_names):
    sp_ = case.get("sample_points")
    if sp_:
        cols = [[sp.Rational(v) for v in sp_[n]] for n in var_names if n in sp_]
        if len(cols) == len(var_names):
            k = min(len(c) for c in cols)
            return [dict(zip(var_names, [c[i] for c in cols])) for i in range(k)]
    base = [sp.Rational(-37, 10), sp.Rational(-3, 5), sp.Rational(9, 10), sp.Rational(23, 10), sp.Rational(51, 10)]
    ycol = [sp.Rational(21, 10), sp.Rational(-7, 5), sp.Rational(1, 3), sp.Rational(17, 10), sp.Rational(-5, 2)]
    return [dict(zip(var_names, [base[i], ycol[i]][: len(var_names)])) for i in range(5)]


def equiv(a, b, case, tol=1e-9):
    """Returns (ok, proof) with proof in {'symbolic','numeric','mismatch'}."""
    names = sorted({s.name for s in (a.free_symbols | b.free_symbols) if s.name in ("x", "y")})
    sub = lambda e, pt: e.subs({sp.Symbol(k): v for k, v in pt.items()}).subs({sp.Symbol(k, real=True): v for k, v in pt.items()})
    pts = sample_pts(case, names) if names else [{}]
    numeric_ok = True
    for pt in pts:
        try:
            av, bv = N(sub(a, pt)), N(sub(b, pt))
        except Exception:   # noqa: BLE001
            return False, "mismatch"
        if not (av.is_number and bv.is_number) or av.has(sp.nan, sp.zoo, sp.oo, -sp.oo) or bv.has(sp.nan, sp.zoo, sp.oo, -sp.oo):
            return False, "mismatch"
        if abs(sp.im(av)) > tol or abs(sp.im(bv)) > tol:          # real input must give real output (cases are declared real)
            if case["domain"] == "real":
                return False, "mismatch"
        if abs(av - bv) > tol * max(1, abs(bv)):
            return False, "mismatch"
    try:
        if sp.simplify(a - b) == 0:
            return True, "symbolic"
    except Exception:   # noqa: BLE001
        pass
    return numeric_ok, "numeric"


def numeq(a, b, tol):
    try:
        return abs(N(a) - N(b)) <= max(tol["abs"], tol["rel"] * abs(N(b)))
    except Exception:   # noqa: BLE001
        return False


# --------------------------------------------------------------------------- form checks
def form_check(name, text, expected_text, case):
    """Returns (ok|None, note)."""
    try:
        if name is None and case["operation"] in ("simplify", "evaluate"):
            name = "generic_ops"
        if name is None:
            return None, ""
        if name == "generic_ops" or name in ("collected", "no_unexpanded_product"):
            a, b = ops_raw(text), ops_raw(expected_text)
            return a <= b + 1, "ops(raw)=%d vs reference %d (allowed +1)" % (a, b)
        e = P(text, syms={"x": X, "y": Y})
        if name == "simplified_radical":
            raw = P(text, evaluate=False)
            for pw in raw.atoms(sp.Pow):
                if pw.exp == sp.Rational(1, 2) and pw.base.is_Integer and any(k >= 2 for k in sp.factorint(pw.base).values()):
                    return False, "radical %s has a square factor" % pw
            return True, ""
        if name == "expanded":
            return e == sp.expand(e), "structurally expanded" if e == sp.expand(e) else "not expanded"
        if name == "factored":
            f = sp.factor(e)
            ok = isinstance(e, sp.Mul) and len(e.args) == len(f.args) and sp.simplify(f - e) == 0
            return ok, "product of irreducible factors" if ok else "not a fully factored product"
        if name == "no_fraction":
            n, d = sp.fraction(sp.together(e))
            return not d.has(X), "denominator has no x" if not d.has(X) else "still a fraction in x"
        if name == "reduced_fraction":
            n, d = sp.fraction(sp.together(e))
            return sp.gcd(n, d) == 1, "numerator/denominator coprime"
        if name == "partial_fractions":
            ok = isinstance(e, sp.Add) and len(e.args) >= 2 and all(sp.degree(sp.fraction(sp.together(t))[1], X) <= 1 for t in e.args)
            return ok, "sum of fractions with linear denominators" if ok else "not decomposed (not a sum of linear-denominator fractions)"
    except Exception as ex:   # noqa: BLE001
        return None, "form check could not run: %s" % type(ex).__name__
    return None, ""


# --------------------------------------------------------------------------- restrictions
def restriction_ok(required, reported_texts, step_texts):
    """required like 'x != 1'.  Satisfied if a reported constraint/step expresses the same exclusion."""
    m = re.fullmatch(r"(\w+)\s*!=\s*(.+)", required)
    v, c = m.group(1), sp.Rational(m.group(2))
    var = sp.Symbol(v)
    for t in reported_texts + step_texts:
        for part in re.split(r"[;,\n]| and ", t):
            mm = re.search(r"(.+?)\s*(?:!=|≠)\s*(.+)", part)
            if mm:
                try:
                    l, r = P(mm.group(1).strip().split("|")[-1].strip(), syms={v: var}), P(mm.group(2).strip().split("(")[0].strip() if "(" in mm.group(2) and not re.match(r"^[\w\s\+\-\*/\^\.]+$", mm.group(2).split("(")[0]) else mm.group(2).strip(), syms={v: var})
                    sols = sp.solve(l - r, var)
                    if sols and all(s == c for s in sols):
                        return True
                except Exception:   # noqa: BLE001
                    pass
            if re.search(r"\b%s\s*=\s*%s\b" % (re.escape(v), re.escape(str(c))), part) and re.search(r"denominator|undefined|excluded|reject|domain|zero", t, re.I):
                return True
    return False


# --------------------------------------------------------------------------- answer comparers
def _finite_set_equal(actual_vals, exp_vals, tol=1e-9):
    A, B = list(actual_vals), list(exp_vals)
    if len(A) != len(B):
        return False
    used = [False] * len(B)
    for a in A:
        hit = False
        for i, b in enumerate(B):
            if not used[i]:
                try:
                    if sp.simplify(a - b) == 0 or abs(N(a - b)) < tol:
                        used[i] = hit = True
                        break
                except Exception:   # noqa: BLE001
                    pass
        if not hit:
            return False
    return True


def judge(case, reply):
    """-> verdict dict.  'reply' is the preserved raw worker reply."""
    v = {"status": None, "sub": "", "value_correct": None, "form_ok": None, "domain_ok": None, "answer_produced": False,
         "covered": False, "strict_correct": False, "value_only_correct": False, "expected": case["spec_expected"], "actual": "",
         "proof": None, "manual_review": False, "manual_reasons": [], "form_note": "", "constant_shown": None, "detail": "",
         "domain_check": {"required": case["required_restrictions"], "reported": reply.get("restrictions_reported", []), "satisfied": None}}
    oc = reply.get("outcome")
    err = reply.get("error") or {}
    exp, cmp_ = case["expected"], case["comparison"]
    method = cmp_["method"]
    tol = case["tolerance"]

    def done(status, sub="", **kw):
        v.update(status=status, sub=sub, **kw)
        v["value_only_correct"] = bool(v["value_correct"]) and v["form_ok"] is not False
        v["strict_correct"] = bool(v["value_correct"]) and v["form_ok"] is not False and v["domain_ok"] is not False
        v["covered"] = v["answer_produced"] and v["form_ok"] is not False and not v.get("_not_in_form")
        v.pop("_not_in_form", None)
        return v

    if oc == "timeout":
        return done("timeout", "no reply within the per-case timeout")
    if oc == "infra_error":
        return done("infra_error", err.get("message", "")[:200])
    if oc == "unsupported":
        r = reply.get("unsupported_reason", "")
        sub = "no_api" if ("no public method" in r or "no mapping" in r) else "unsolved_by_solver"
        v["actual"] = r[:200]
        return done("unsupported", sub)
    if oc == "input_incompatible":
        v["actual"] = err.get("message", "")
        return done("input_incompatible", err.get("type", ""))
    if oc == "error":
        v["actual"] = "%s: %s" % (err.get("type"), err.get("message", "")[:120])
        if method == "undefined" and re.search(r"zerodivision|division|undefined|zoo|domain", (err.get("type", "") + err.get("message", "")), re.I):
            v.update(answer_produced=True, value_correct=True, form_ok=True)
            v["manual_review"] = True
            v["manual_reasons"].append("L04 passed via raised division-by-zero error (no structured undefined result)")
            v["detail"] = "error signalled instead of a fabricated solution; contract_exception=%s" % err.get("contract_exception")
            return done("valid", "error_signalled_domain_error")
        return done("runtime_error", err.get("type", ""))

    # ------------- outcome == answer
    ans = reply.get("answer") or {}
    kind = ans.get("kind")
    v["answer_produced"] = True
    steps_text = " ".join((s.get("rule", "") + " " + s.get("description", "")) for s in reply.get("steps", []))
    try:
        # ---------------- scalar / expression families
        if method in ("value", "expr_equiv", "antiderivative") and kind in ("expr", "numeric", "solutions", "equation"):
            text = ans.get("text") or (ans.get("values") or [""])[0]
            v["actual"] = text
            e = P(text, syms={"x": X, "y": Y})
            ref = P(exp["expr"], syms={"x": X, "y": Y})
            if method == "value":
                ok = numeq(e, ref, tol)
                try:
                    proof = "symbolic" if sp.simplify(e - ref) == 0 else "numeric"
                except Exception:   # noqa: BLE001
                    proof = "numeric"
                v.update(value_correct=bool(ok), proof=proof if ok else "mismatch")
            elif method == "expr_equiv":
                ok, proof = equiv(e, ref, case)
                v.update(value_correct=ok, proof=proof)
                if ok and proof == "numeric":
                    v["manual_review"] = True; v["manual_reasons"].append("equivalence established numerically only (no symbolic zero)")
            else:   # antiderivative
                e0, shown = strip_constant(e)
                v["constant_shown"] = shown
                d = sp.diff(e0, X)
                f = P(case["problem"]["expr"], syms={"x": X, "y": Y})
                ok, proof = equiv(d, f, {**case, "domain": "complex"})
                real_ok = True
                if cmp_.get("require_real_on"):
                    for pt in cmp_["require_real_on"]:
                        val = N(e0.subs(X, sp.Rational(pt)))
                        if not val.is_number or abs(sp.im(val)) > 1e-12:
                            real_ok = False
                    v["domain_ok"] = real_ok
                    v["detail"] = "F'==f: %s ; real-valued at x<0 and x>0: %s" % (ok, real_ok)
                v.update(value_correct=bool(ok and real_ok), proof=proof)
            fo, note = form_check(cmp_.get("form"), text, exp["expr"], case)
            v["form_ok"], v["form_note"] = fo, note
            if case["required_restrictions"]:
                texts = list(reply.get("restrictions_reported", []))
                oks = [restriction_ok(r, texts, [steps_text]) for r in case["required_restrictions"]]
                v["domain_ok"] = all(oks)
                v["domain_check"]["satisfied"] = all(oks)
                v["manual_review"] = True; v["manual_reasons"].append("restriction-reporting decided by text matching")
            return done("valid" if v["value_correct"] and v["form_ok"] is not False else "invalid_answer",
                        "" if v["value_correct"] else ("domain_error" if v["domain_ok"] is False else "wrong_value"))

        # ---------------- equation solving
        if method == "solution_set":
            domain_cx = case["domain"] == "complex"
            ek = exp["kind"]
            if kind == "empty":
                v["actual"] = "no solution (empty set)"
                if ek == "empty":
                    v.update(value_correct=True, form_ok=True)
                elif ek == "finite" and domain_cx and re.search(r"no real|real number system|over the real", steps_text, re.I):
                    v.update(value_correct=False, form_ok=True)
                    return done("invalid_answer", "real_domain_only (states no real solutions; complex solutions missing)")
                else:
                    v.update(value_correct=False, form_ok=True)
                    return done("invalid_answer", "claims no solution but solutions exist" if ek != "empty" else "")
                return done("valid")
            if kind == "all":
                v["actual"] = "all %s" % ans.get("domain", "x")
                v.update(value_correct=(ek == "all"), form_ok=True)
                return done("valid" if ek == "all" else "invalid_answer", "" if ek == "all" else "claims every x is a solution")
            if kind == "solutions":
                vals = [P(t, syms={"x": X}) for t in ans["values"]]
                v["actual"] = "{" + ", ".join(ans["values"]) + "}"
                if ek == "finite":
                    ref = [P(t, syms={"x": X}) for t in exp["values"]]
                    ok = _finite_set_equal(vals, ref)
                    sub = ""
                    if not ok and domain_cx and len(vals) < len(ref) and all(any(_finite_set_equal([a], [b]) for b in ref) for a in vals):
                        sub = "real_domain_only (complex roots missing)"
                    v.update(value_correct=ok, form_ok=True)
                    return done("valid" if ok else "invalid_answer", sub or ("" if ok else "wrong_solution_set"))
                v.update(value_correct=False, form_ok=True)
                return done("invalid_answer", "returned solutions where %s expected" % ek)
            v["actual"] = json.dumps(ans)[:200]
            v["_not_in_form"] = True
            return done("invalid_answer", "unexpected answer form %r" % kind)

        if method == "system_solution":
            ek = exp["kind"]
            if kind == "empty":
                v.update(value_correct=(ek == "empty"), form_ok=True, actual="no solution")
                return done("valid" if ek == "empty" else "invalid_answer", "" if ek == "empty" else "claims no solution")
            if kind == "system":
                sols = ans["solutions"]
                v["actual"] = json.dumps(sols)
                if ek == "unique" and len(sols) == 1:
                    ok = all(k in sols[0] and sp.simplify(P(sols[0][k]) - sp.Rational(val)) == 0 for k, val in exp["assignments"].items()) and set(sols[0]) == set(exp["assignments"])
                    v.update(value_correct=ok, form_ok=True)
                    return done("valid" if ok else "invalid_answer", "" if ok else "wrong_solution")
                v.update(value_correct=False, form_ok=True)
                return done("invalid_answer", "returned %d solution(s) where %s expected" % (len(sols), ek))
            v["actual"] = json.dumps(ans)[:200]; v["_not_in_form"] = True
            return done("invalid_answer", "unexpected answer form %r" % kind)

        if method == "inequality_set":
            exp_set = sp.Union(*[sp.Interval(sp.sympify(a.replace("oo", "oo")), sp.sympify(b), l, r) for a, b, l, r in exp["intervals"]])
            if kind == "intervals":
                act = sp.Union(*[sp.Interval(sp.sympify(a), sp.sympify(b), l, r) for a, b, l, r in ans["intervals"]])
                v["actual"] = str(act)
            elif kind == "relation":
                rel = P(ans["text"], syms={"x": XR})
                act = rel.as_set()
                v["actual"] = ans["text"]
            else:
                v["actual"] = json.dumps(ans)[:200]; v["_not_in_form"] = True
                return done("invalid_answer", "unexpected answer form %r" % kind)
            ok = act == exp_set or act.symmetric_difference(exp_set) == sp.S.EmptySet
            v.update(value_correct=bool(ok), form_ok=True)
            return done("valid" if ok else "invalid_answer", "" if ok else "wrong_interval_set")

        if method == "matrix":
            if kind != "matrix":
                v["actual"] = json.dumps(ans)[:200]; v["_not_in_form"] = True
                return done("invalid_answer", "unexpected answer form %r" % kind)
            A = sp.Matrix([[P(c) for c in r] for r in ans["rows"]])
            B = sp.Matrix([[sp.Rational(c) for c in r] for r in exp["matrix"]])
            v["actual"] = str(A.tolist())
            ok = A.shape == B.shape and (A - B).applyfunc(sp.simplify) == sp.zeros(*B.shape)
            v.update(value_correct=bool(ok), form_ok=True)
            return done("valid" if ok else "invalid_answer", "" if ok else "wrong_matrix")

        if method == "gradient":
            if kind != "vector":
                v["actual"] = json.dumps(ans)[:200]; v["_not_in_form"] = True
                return done("invalid_answer", "unexpected answer form %r" % kind)
            v["actual"] = "(" + ", ".join(ans["items"]) + ")"
            ok = len(ans["items"]) == len(exp["vector"]) and all(equiv(P(a, syms={"x": X, "y": Y}), P(b, syms={"x": X, "y": Y}), case)[0] for a, b in zip(ans["items"], exp["vector"]))
            v.update(value_correct=bool(ok), form_ok=True)
            return done("valid" if ok else "invalid_answer", "" if ok else "wrong_gradient")

        if method == "ode_family":
            if kind != "equation":
                v["actual"] = json.dumps(ans)[:200]; v["_not_in_form"] = True
                return done("invalid_answer", "unexpected answer form %r" % kind)
            f = sp.Function("y")
            eq = P(ans["text"], syms={"x": X}, extra={"y": f})
            v["actual"] = ans["text"]
            rhs = eq.rhs
            ode = P(case["problem"]["ode"], syms={"x": X}, extra={"y": f})
            consts = sorted([s for s in rhs.free_symbols if re.fullmatch(r"C\d*", s.name)], key=lambda s: s.name)
            resid = sp.simplify(ode.subs(f(X), rhs).doit())
            ok_ode = resid == 0
            ok_n = len(consts) == exp["n_constants"]
            basis = [sp.diff(rhs, c) for c in consts]
            indep = (basis[0] != 0) if len(basis) == 1 else (sp.simplify(sp.wronskian(basis, X)) != 0 if len(basis) >= 2 else False)
            ok = ok_ode and ok_n and indep
            v["detail"] = "ODE residual after substitution: %s ; constants=%d ; independent basis: %s" % (resid, len(consts), indep)
            v.update(value_correct=bool(ok), form_ok=True)
            return done("valid" if ok else "invalid_answer", "" if ok else "not a general solution")

        if method == "numeric_roots":
            vals = ans.get("values", [])
            if kind == "empty":
                v.update(value_correct=False, form_ok=True, actual="no solution")
                return done("invalid_answer", "claims no solution but a real root exists")
            if kind not in ("numeric", "solutions"):
                v["actual"] = json.dumps(ans)[:200]; v["_not_in_form"] = True
                return done("invalid_answer", "unexpected answer form %r" % kind)
            nums = []
            for t in vals:
                try:
                    nums.append(N(P(t, syms={"x": X}), 30))
                except Exception:   # noqa: BLE001
                    nums.append(None)
            v["actual"] = "[" + ", ".join(vals) + "]"
            ref = [sp.Float(t, 30) for t in exp["values"]]
            ok = all(any(n is not None and n.is_number and abs(sp.im(n)) < 1e-9 and abs(sp.re(n) - r) <= tol["abs"] for n in nums) for r in ref)
            spec_val = exp.get("spec_printed_value")
            if spec_val:
                v["detail"] = "matches spec-printed value %s within 1e-6: %s" % (spec_val, any(n is not None and abs(sp.im(n)) < 1e-9 and abs(sp.re(n) - sp.Float(spec_val, 30)) <= 1e-6 for n in nums))
            v.update(value_correct=bool(ok), form_ok=True)
            if any("CRootOf" in t for t in vals):
                v["manual_review"] = True; v["manual_reasons"].append("exact CRootOf accepted via numeric evaluation (declared rule)")
            return done("valid" if ok else "invalid_answer", "" if ok else "no root within tolerance")

        if method == "undefined":
            if kind == "undefined":
                v.update(value_correct=True, form_ok=True, actual="undefined: " + str(ans.get("reason", ""))[:120])
                return done("valid", "explicit_undefined_result")
            if kind == "empty":
                v.update(value_correct=False, form_ok=False, actual="no solution (empty set) - undefined not flagged")
                v["manual_review"] = True; v["manual_reasons"].append("L04 partial: empty set without undefined flag")
                return done("invalid_answer", "partial: empty solution set, expression not flagged undefined; no fabricated solution")
            v.update(value_correct=False, form_ok=False, actual=json.dumps(ans)[:150])
            return done("invalid_answer", "fabricated_solution")
    except Exception as ex:   # noqa: BLE001
        v["manual_review"] = True
        v["manual_reasons"].append("judge could not evaluate the output: %s: %s" % (type(ex).__name__, str(ex)[:100]))
        v["actual"] = v["actual"] or json.dumps(ans)[:200]
        v.update(value_correct=None, _not_in_form=True)
        return done("invalid_answer", "unjudgeable_output")
    v["manual_review"] = True; v["manual_reasons"].append("no comparer matched answer kind %r for method %r" % (kind, method))
    v["actual"] = json.dumps(ans)[:200]; v["_not_in_form"] = True
    return done("invalid_answer", "unexpected answer form %r" % kind)
''')

W('bench/derivations.py', r'''"""Derivation-quality scoring (rubric: correctness 0-3, completeness 0-2, explanation 0-2, consistency 0-1, domain 0-1, presentation 0-1).

Automated layer (deterministic, auditable; thresholds below are declared before results were inspected):
  correctness   0 final answer not (strictly) correct | 1 final correct but an automated check found an invalid step or the
                conclusion contradicts the final answer | 2 final correct, no invalid step, but the worked steps do not demonstrably
                produce the answer (solver flagged 'direct_evaluation' override, or final answer absent from every step) | 3 clean.
  completeness  share of the case's reference milestones (regex over step text) found: >=0.75 -> 2, >=0.34 -> 1, else 0; a derivation
                that is only opaque/structural steps scores 0; fewer than MIN_STEPS[op] substantive steps caps the score at 1.
  explanation   per substantive step: 2 = specific rule label + >=8 prose words ; 1 = specific rule label OR >=4 prose words ; 0 = opaque
                ('Compute exact result', 'Direct evaluation', ...).  mean>=1.5 -> 2 ; >=0.6 -> 1 ; else 0.
  consistency   1 if no invalid transition, no override, and the final answer appears in the workings; else 0.
  domain        domain-sensitive case: 1 iff the answer's domain handling is correct AND the workings mention the restriction/rejection
                (constraint list or domain vocabulary); else 0.  Non-sensitive case: 1 unless a domain error was detected (N/A -> full;
                reported separately as 'domain-sensitive-only mean').
  presentation  1 iff >=80% of steps have readable expression text free of Python-object artefacts (<...>, Symbol(, ...Rule( trees).
These are PROXIES.  A manual-review overlay (data/manual_reviews.json) records per-criterion adjustments with confidence and
justification; every row keeps both the automated and the final score.  The reviewer is a language model (Claude), so the overlay is
flagged 'model_review' and is NOT the authority on mathematical correctness (that comes from judge.py's symbolic/numeric checks).
"""
import json
import math
import re

import sympy as sp

try:
    from . import judge as J
except ImportError:
    import judge as J

MIN_STEPS = {"evaluate": 1, "simplify": 1, "expand": 1, "factor": 1, "solve": 2, "nsolve": 2, "solve_system": 3, "solve_inequality": 2,
             "diff": 2, "partial_derivative": 2, "gradient": 2, "integrate": 2, "limit": 2, "det": 2, "inverse": 2, "rref": 2,
             "dsolve": 2, "partial_fractions": 3}
OPAQUE = {"compute exact result", "direct evaluation", "general solving"}
GENERIC_RULES = {"apply algebraic identities"}      # boilerplate label reused across unrelated problems: not a specific rule identification
STRUCTURAL = {"setup", "final result", "limit setup", "partial derivative setup", "gradient setup", "augmented matrix",
              "identify linear equation", "identify quadratic equation", "identify polynomial", "identify rational form",
              "reduced row echelon form"}
DOMAIN_WORDS = re.compile(r"domain|restrict|undefined|denominator|extraneous|reject|!=|≠|no real|real number|nonzero|non-zero|positive|"
                          r"excluded|must not|cannot be zero|inconsistent|identity|contradict|absolute value|\|x\|", re.I)
ARTEFACT = re.compile(r"<[^>]*object|Symbol\(|Rule\(|\bRule\b.*integrand=|_sympy|\bzoo\b")


def prose_words(text):
    if ARTEFACT.search(text) or "integrand=" in text:      # raw object/rule-tree dumps are not prose
        return 0
    t = re.sub(r"[-+*/^=()\[\]{}<>|,.:;→←∫∂Δ∇]", " ", text)
    return len([w for w in t.split() if re.fullmatch(r"[A-Za-z']{3,}", w) and w.lower() not in ("sin", "cos", "tan", "log", "exp", "sqrt", "abs", "dx", "matrix")])


def _flat(steps):
    out = []
    for s in steps:
        out.append(" ".join([s.get("rule", ""), s.get("description", ""), s.get("before", ""), s.get("after", "")] + [" ".join(n) for n in s.get("notes", [])]))
        for sub in s.get("substeps", []) or []:
            out.append(json.dumps(sub))
    return " \n".join(out)


def _expr_or_none(t):
    try:
        if not t or any(tok in t for tok in ("Eq(", "[", "{", "<", ">", "diff(", "integrate(", "limit(", "matrix", "solve", "|")):
            return None
        return J.P(t, syms={"x": J.X, "y": J.Y})
    except Exception:   # noqa: BLE001
        return None


def check_steps(case, steps, final_ok):
    """-> (n_checked, n_invalid, notes).  Only algebra families are checked transition-by-transition."""
    op = case["operation"]
    checked = invalid = 0
    notes = []
    if op in ("evaluate", "simplify", "expand", "factor"):
        prev = _expr_or_none(case["problem"]["expr"].replace("^", "**"))
        for i, s in enumerate(steps):
            for key in ("before", "after"):
                e = _expr_or_none(s.get(key, "").replace("^", "**"))
                if e is None or prev is None:
                    continue
                if key == "before":
                    # StepSolver gives before/after pairs; 'before' must equal the previous 'after' (or input)
                    ok, _ = J.equiv(e, prev, {**case, "domain": "complex"})
                else:
                    ok, _ = J.equiv(e, prev, {**case, "domain": "complex"})
                    prev = e
                checked += 1
                if not ok:
                    invalid += 1
                    notes.append("step %d (%s): '%s' not equivalent to previous expression" % (i + 1, key, s.get(key)))
    elif op in ("solve", "nsolve"):
        truth = [J.P(t) for t in case["expected"].get("values", [])] if case["expected"].get("kind") == "finite" else None
        for i, s in enumerate(steps):
            m = re.match(r"^\s*Eq\(\s*x\s*,\s*(.+)\)\s*$", s.get("after", ""))
            if m and truth is not None:
                try:
                    val = J.P(m.group(1).replace("^", "**"), syms={"x": J.X})
                except Exception:   # noqa: BLE001
                    continue
                checked += 1
                if not any(sp.simplify(val - t) == 0 for t in truth):
                    # a step may legitimately show a REJECTED candidate (extraneous root); only flag if the text does not say so
                    if not re.search(r"reject|extraneous|does not satisfy|not a solution", s.get("description", "") + s.get("rule", ""), re.I):
                        invalid += 1
                        notes.append("step %d claims x = %s which is not a true solution" % (i + 1, m.group(1)))
    return checked, invalid, notes


def automated_score(case, reply, verdict):
    steps = [s for s in reply.get("steps", []) if s]
    rec = {"available": bool(steps), "n_steps": len(steps), "scored": False}
    if not steps or reply.get("outcome") != "answer":
        return rec
    flat = _flat(steps)
    nf = reply.get("native_flags") or {}
    rules = [s.get("rule", "").strip().lower() for s in steps]
    substantive = [s for s, r in zip(steps, rules) if r not in OPAQUE and r not in STRUCTURAL and not r.startswith("identify")]
    final_ok = bool(verdict.get("value_correct")) and verdict.get("form_ok") is not False
    n_chk, n_inv, inv_notes = check_steps(case, steps, final_ok)

    # --- correctness
    override = nf.get("steps_mode") == "direct_evaluation" or nf.get("workings_tied_to_answer") is False
    if not final_ok:
        c = 0
        why_c = "final answer not correct"
    elif n_inv:
        c, why_c = 1, "invalid step(s): " + "; ".join(inv_notes[:2])
    else:
        ans_text = json.dumps(reply.get("answer"))
        present = True
        # final answer present in workings? (value tokens of the answer must occur in some step text)
        toks = re.findall(r"[A-Za-z0-9_./^*+-]+", re.sub(r"[\[\]{}\"']", " ", ans_text))
        toks = [t for t in toks if len(t) > 0 and t not in ("kind", "expr", "text", "values", "solutions", "empty", "rows", "items", "system", "relation", "equation", "mapping", "entries")]
        if toks:
            norm = lambda t: re.sub(r"\s+", "", t.replace("**", "^"))
            present = any(norm(t) in norm(flat) for t in toks)
        if override:
            c, why_c = 2, "workings do not demonstrably produce the answer (steps_mode=direct_evaluation override, or workings from a code path separate from the answer, e.g. SymPy integral_steps vs integrate)"
        elif not present:
            c, why_c = 2, "final answer does not appear in the workings"
        else:
            c, why_c = 3, "no invalid step found (%d transitions auto-checked)" % n_chk

    # --- contradiction detection for integrals G04-like (derivation says ln|x| but answer is log(x))
    if case["id"] == "G04" and final_ok is False and re.search(r"ln\|x\||log\(abs", flat):
        why_c += "; workings state ln|x| but the returned antiderivative is not real-valued for x<0 (contradiction)"

    # --- completeness
    ms = case.get("reference_milestones") or []
    hits = sum(1 for m in ms if re.search(m, flat, re.I))
    frac = (hits / len(ms)) if ms else 1.0
    comp = 2 if frac >= 0.75 else 1 if frac >= 0.34 else 0
    if not substantive:
        comp = 0
    elif len(substantive) < MIN_STEPS.get(case["operation"], 1):
        comp = min(comp, 1)

    # --- explanation
    per = []
    for s in substantive:
        spec = s.get("rule", "").strip().lower() not in OPAQUE | GENERIC_RULES and bool(s.get("rule", "").strip())
        w = prose_words(s.get("description", ""))
        per.append(2 if (spec and w >= 8) else 1 if (spec or w >= 4) else 0)
    m_exp = (sum(per) / len(per)) if per else 0.0
    expl = 2 if m_exp >= 1.5 else 1 if m_exp >= 0.6 else 0

    # --- consistency
    cons = 1 if (n_inv == 0 and not override and c >= 2) else 0

    # --- domain
    if case["domain_sensitive"]:
        mentions = bool(reply.get("restrictions_reported")) or bool(DOMAIN_WORDS.search(flat))
        dom = 1 if (verdict.get("domain_ok") is not False and final_ok and mentions) else 0
    else:
        dom = 1 if verdict.get("domain_ok") is not False else 0

    # --- presentation
    readable = [s for s in steps if (s.get("after") or s.get("description")) and not ARTEFACT.search(s.get("after", "") + s.get("description", ""))]
    pres = 1 if len(readable) >= 0.8 * len(steps) else 0

    if not substantive:
        # Rubric rule 5/9 refinement (declared after the first inspection, applied uniformly to every solver):
        # a derivation with no substantive step shows no transformations -> correctness<=2, nothing to follow (consistency 0), nothing readable (presentation 0).
        c, cons, pres = min(c, 2), 0, 0
        why_c += "; opaque/structural steps only (no transformation shown)"
    rec.update(scored=True, automated={"correctness": c, "completeness": comp, "explanation": expl, "consistency": cons, "domain": dom,
                                       "presentation": pres}, automated_total=c + comp + expl + cons + dom + pres,
               evidence={"correctness": why_c, "milestones_hit": "%d/%d" % (hits, len(ms)), "mean_explanation_per_step": round(m_exp, 2),
                         "substantive_steps": len(substantive), "steps_auto_checked": n_chk, "invalid_steps": n_inv,
                         "opaque_steps": sum(1 for r in rules if r in OPAQUE), "native_flags": {k: nf.get(k) for k in ("steps_mode", "steps_verified", "workings_tied_to_answer")}})
    return rec


CRITERIA = ["correctness", "completeness", "explanation", "consistency", "domain", "presentation"]
CAPS = dict(correctness=3, completeness=2, explanation=2, consistency=1, domain=1, presentation=1)


def apply_manual(rec, review):
    """review: {'adjust': {criterion: new_value}, 'confidence': 'low|medium|high', 'justification': str}"""
    if not rec.get("scored"):
        return rec
    final = dict(rec["automated"])
    if review:
        for k, v in review.get("adjust", {}).items():
            assert k in CAPS and 0 <= v <= CAPS[k], (k, v)
            final[k] = v
        rec["manual"] = {"reviewer": "model_review (Claude) - requires human audit", "confidence": review["confidence"],
                         "justification": review["justification"], "adjusted": review.get("adjust", {})}
    rec["final"] = final
    rec["final_total"] = sum(final.values())
    return rec
''')

W('bench/reference_validation.py', r'''#!/usr/bin/env python3
"""Independently validate every reference answer in data/cases.json.

Method: exact rational arithmetic (fractions.Fraction), cmath/math, and mpmath numerics
(numerical differentiation/quadrature/root polishing).  No SymPy symbolic engine is used,
so these checks do not share code with any solver under test or with the judge.
Each case gets >=1 check; most get two independent methods (listed in 'methods').
"""
import cmath
import json
import math
import pathlib
import random
import sys
from fractions import Fraction as Fr

import mpmath as mp

mp.mp.dps = 40
ROOT = pathlib.Path(__file__).resolve().parent.parent
RNG = random.Random(20240601)   # fixed seed -> deterministic
R = {}


def rec(cid, ok, methods, detail=""):
    R[cid] = {"ok": bool(ok), "methods": methods, "detail": detail}


def rat_points(n=8, avoid=(), lo=-9, hi=9):
    pts = []
    while len(pts) < n:
        p = Fr(RNG.randint(lo * 7, hi * 7), RNG.randint(2, 7))
        if p not in avoid and p not in pts:
            pts.append(p)
    return pts


def identity(cid, f, g, avoid=(), n=10, extra=""):
    """f(x)==g(x) at n random rationals (exact)."""
    ok = all(f(p) == g(p) for p in rat_points(n, avoid))
    rec(cid, ok, ["exact rational identity test at %d random points" % n], extra)


def mpnum_deriv(f, x):
    return mp.diff(f, x)


# ---------------------------------------------------------------- A
rec("A01", Fr(2) + 3 * 4 == 14, ["Fraction arithmetic"])
rec("A02", (Fr(2) + 3) * 4 == 20, ["Fraction arithmetic"])
rec("A03", Fr(7, 12) + Fr(5, 18) == Fr(31, 36), ["Fraction arithmetic"])
rec("A04", 72 == 36 * 2 and abs(math.sqrt(72) - 6 * math.sqrt(2)) < 1e-12 and math.isqrt(36) ** 2 == 36,
    ["integer factorisation 72=6^2*2", "float check"], "largest square factor of 72 is 36")
rec("A05", (2 ** 3) ** 2 == 64, ["integer arithmetic"])
rec("A06", Fr("0.125") * 80 == 10, ["Fraction arithmetic (exact decimal)"])
rec("A07", (-3) ** 2 - 3 ** 2 == 0, ["integer arithmetic"])
rec("A08", abs(-7) + abs(3 - 9) == 13, ["integer arithmetic"])

# ---------------------------------------------------------------- B
identity("B01", lambda x: 3 * x + 5 * x - 2 * x, lambda x: 6 * x)
identity("B02", lambda x: 2 * (x + 3) - x, lambda x: x + 6)
identity("B03", lambda x: (x + 2) * (x + 3), lambda x: x * x + 5 * x + 6)
identity("B04", lambda x: x * x + 5 * x + 6, lambda x: (x + 2) * (x + 3))
identity("B05", lambda x: x * x - 9, lambda x: (x - 3) * (x + 3))
identity("B06", lambda x: (x * x - 1) / (x - 1), lambda x: x + 1, avoid=(Fr(1),),
         extra="original undefined at x=1 (ZeroDivisionError confirmed below)")
try:
    (Fr(1) * Fr(1) - 1) / (Fr(1) - 1)
    z = False
except ZeroDivisionError:
    z = True
R["B06"]["ok"] = R["B06"]["ok"] and z
identity("B07", lambda x: x / x, lambda x: Fr(1), avoid=(Fr(0),))
identity("B08", lambda x: None, lambda x: None)  # replaced below (two variables)
pts = [(a, b) for a in rat_points(5, (Fr(0),)) for b in rat_points(3, (Fr(0),))]
rec("B08", all((2 * a * a * b) / (4 * a * b * b) == a / (2 * b) for a, b in pts),
    ["exact rational identity test at 15 (x,y) points"], "original denominator 4*x*y^2 zero iff x=0 or y=0")

# ---------------------------------------------------------------- C
def lin(a, b):  # a*x + b = 0
    return -Fr(b) / Fr(a)

rec("C01", 2 * Fr(4) + 3 == 11 and lin(2, -8) == 4, ["substitution", "closed form"])
rec("C02", 2 * (Fr(4) + 3) == 14, ["substitution", "closed form"])
rec("C03", 3 * Fr(6) - 7 == Fr(6) + 5, ["substitution"])
rec("C04", Fr(9, 3) + 2 == 5, ["substitution"])
rec("C05", all(x * x - 5 * x + 6 == 0 for x in (2, 3)) and (25 - 24) == 1 and
    sorted(mp.polyroots([1, -5, 6])) == sorted([mp.mpf(2), mp.mpf(3)]) or
    all(abs(r - t) < 1e-25 for r, t in zip(sorted(mp.polyroots([1, -5, 6]), key=lambda z: z.real), [2, 3])),
    ["substitution", "mpmath.polyroots", "discriminant 1>0"])
rec("C06", all(x * x - 4 == 0 for x in (-2, 2)), ["substitution", "discriminant 16>0 (two real roots)"])
rts = mp.polyroots([1, 0, 1])
rec("C07", all(abs(r * r + 1) < 1e-30 for r in (1j, -1j)) and
    sorted([complex(r).imag for r in rts]) == [-1.0, 1.0] and all(abs(complex(r).real) < 1e-20 for r in rts) and
    (0 - 4) < 0, ["complex substitution", "mpmath.polyroots", "discriminant -4<0 -> no real roots"])
rec("C08", (-1) ** 2 + 2 * (-1) + 1 == 0 and (4 - 4) == 0 and
    mp.diff(lambda x: x * x + 2 * x + 1, -1) == 0,
    ["substitution", "discriminant 0", "p'(-1)=0 -> double root"])
rec("C09", 2 * 1 + 1 != 2 * 1 + 3 and all(2 * p + 1 != 2 * p + 3 for p in rat_points(8)),
    ["LHS-RHS == -2 identically (point test)"])
rec("C10", all(2 * p + 1 == 2 * p + 1 for p in rat_points(8)), ["LHS-RHS == 0 identically (point test)"])

# ---------------------------------------------------------------- D
def cramer(a1, b1, c1, a2, b2, c2):
    det = Fr(a1) * b2 - Fr(a2) * b1
    if det == 0:
        return None
    return ((Fr(c1) * b2 - Fr(c2) * b1) / det, (Fr(a1) * c2 - Fr(a2) * c1) / det)

for cid, args, sol in [("D01", (1, 1, 7, 1, -1, 1), (4, 3)), ("D02", (2, 1, 8, 1, -1, 1), (3, 2)), ("I01", (1, 1, 5, 1, -1, 1), (3, 2))]:
    r = cramer(*args)
    a1, b1, c1, a2, b2, c2 = args
    sub = a1 * sol[0] + b1 * sol[1] == c1 and a2 * sol[0] + b2 * sol[1] == c2
    rec(cid, r == tuple(Fr(v) for v in sol) and sub, ["Cramer's rule (Fractions)", "substitution"])
# D03 inconsistent: det = 0 and rank test (augmented rank 2 vs coefficient rank 1)
rec("D03", Fr(1) * 1 - Fr(1) * 1 == 0 and 2 != 5, ["det=0 with inconsistent RHS (coefficients proportional, constants not)"])

def sample_ineq(pred, intervals, grid):
    """pred(x) truth must equal membership in the expected intervals for every grid point (incl. boundaries)."""
    def member(x):
        for lo, hi in intervals:
            if (lo is None or x > lo) and (hi is None or x < hi):
                return True
        return False
    return all(pred(g) == member(g) for g in grid)

grid = [Fr(k, 4) for k in range(-80, 81)]
rec("D04", sample_ineq(lambda x: 2 * x + 3 < 11, [(None, Fr(4))], grid), ["exhaustive grid truth table (161 pts incl. boundary x=4)"])
rec("D05", sample_ineq(lambda x: -2 * x > 6, [(None, Fr(-3))], grid), ["grid truth table incl. boundary x=-3 and direction flip"])
rec("D06", sample_ineq(lambda x: x * x - 4 > 0, [(None, Fr(-2)), (Fr(2), None)], grid), ["grid truth table incl. boundaries x=±2"])

# ---------------------------------------------------------------- E
identity("E01", lambda x: x ** 3 * x ** 4, lambda x: x ** 7)
identity("E02", lambda x: x ** 6 / x ** 2, lambda x: x ** 4, avoid=(Fr(0),))
rec("E03", all(math.sqrt(x * x) == abs(x) for x in [-3.0, -0.5, 2.0, 7 / 3]) and math.sqrt((-3.0) ** 2) != -3.0,
    ["float check at negative/positive x", "sqrt((-3)^2)=3 != -3"])
rec("E04", math.isclose(math.log(1000, 10), 3, rel_tol=1e-12) and 10 ** 3 == 1000 and mp.log(1000, 10) == 3,
    ["math.log", "10^3==1000", "mpmath.log"])
rec("E05", all(math.isclose(math.log(math.exp(x)), x, abs_tol=1e-12) for x in [-3.0, -0.5, 2.0, 7 / 3]),
    ["float check over reals incl. negatives"])
rec("E06", abs(mp.sin(mp.pi / 6) - mp.mpf(1) / 2) < mp.mpf(10) ** -35, ["mpmath 40-digit"])
rec("E07", abs(mp.cos(mp.pi / 3) - mp.mpf(1) / 2) < mp.mpf(10) ** -35, ["mpmath 40-digit"])
rec("E08", abs(mp.tan(mp.pi / 4) - 1) < mp.mpf(10) ** -35, ["mpmath 40-digit"])

# ---------------------------------------------------------------- F  (mpmath numerical differentiation vs expected closed form)
def deriv_check(cid, f, d, pts):
    err = max(abs(mp.diff(f, p) - d(p)) for p in pts)
    rec(cid, err < mp.mpf(10) ** -25, ["mpmath.diff numerical differentiation (40 digits) vs closed form at %d pts" % len(pts)],
        "max abs err %s" % mp.nstr(err, 3))

P = [mp.mpf(v) for v in ("-2.7", "-0.4", "0.5", "1.3", "3.9")]
Ppos = [mp.mpf(v) for v in ("0.5", "1.3", "2.7", "7.1")]
deriv_check("F01", lambda x: x ** 3, lambda x: 3 * x ** 2, P)
deriv_check("F02", lambda x: 5 * x ** 4 - 2 * x + 7, lambda x: 20 * x ** 3 - 2, P)
deriv_check("F03", lambda x: x ** 2 * mp.sin(x), lambda x: 2 * x * mp.sin(x) + x ** 2 * mp.cos(x), P)
deriv_check("F04", lambda x: mp.sin(x ** 2), lambda x: 2 * x * mp.cos(x ** 2), P)
deriv_check("F05", lambda x: mp.log(x), lambda x: 1 / x, Ppos)
deriv_check("F06", lambda x: x ** x, lambda x: x ** x * (mp.log(x) + 1), Ppos)

# ---------------------------------------------------------------- G
def anti_check(cid, f, F, pts):
    err = max(abs(mp.diff(F, p) - f(p)) for p in pts)
    rec(cid, err < mp.mpf(10) ** -25, ["mpmath.diff of expected F equals integrand at %d pts" % len(pts),
                                       "second check: F(b)-F(a) == mpmath.quad"],
        "max abs err %s" % mp.nstr(err, 3))
    a, b = mp.mpf("0.5"), mp.mpf("2.5")
    q = mp.quad(f, [a, b])
    R[cid]["ok"] = R[cid]["ok"] and abs((F(b) - F(a)) - q) < mp.mpf(10) ** -25

anti_check("G01", lambda x: 3 * x ** 2, lambda x: x ** 3, P)
anti_check("G02", lambda x: 2 * x + 5, lambda x: x ** 2 + 5 * x, P)
anti_check("G03", lambda x: mp.cos(x), lambda x: mp.sin(x), P)
anti_check("G05", lambda x: x * mp.exp(x ** 2), lambda x: mp.exp(x ** 2) / 2, P)
# G04: ln|x| differentiates to 1/x on both sides of 0; ln(x) alone is not real for x<0
neg, pos = [mp.mpf("-4"), mp.mpf("-0.5")], [mp.mpf("0.5"), mp.mpf("4")]
err = max(abs(mp.diff(lambda x: mp.log(abs(x)), p) - 1 / p) for p in neg + pos)
rec("G04", err < mp.mpf(10) ** -25 and all(abs(mp.log(abs(p)).imag) == 0 for p in neg) and
    all(abs(mp.log(p).imag) > 1 for p in neg),
    ["mpmath.diff of ln|x| at x<0 and x>0", "realness: ln|x| real, ln(x) complex for x<0"], "err %s" % mp.nstr(err, 3))
q = mp.quad(lambda x: x ** 2, [0, 3])
rec("G06", abs(q - 9) < mp.mpf(10) ** -30 and Fr(27, 3) == 9, ["mpmath.quad", "exact (b^3-a^3)/3"])

# ---------------------------------------------------------------- H
def limit_check(cid, f, x0, target, side_eps=("1e-12", "-1e-12")):
    vals = [f(x0 + mp.mpf(e)) for e in side_eps] if x0 is not None else [f(mp.mpf(10) ** 15), f(mp.mpf(10) ** 20)]
    ok = all(abs(v - target) < mp.mpf(10) ** -9 for v in vals)
    rec(cid, ok, ["mpmath 60-digit evaluation approaching the point from both sides"], "values %s" % [mp.nstr(v, 12) for v in vals])

mp.mp.dps = 60
limit_check("H01", lambda x: mp.sin(x) / x, mp.mpf(0), mp.mpf(1))
limit_check("H02", lambda x: (x ** 2 - 4) / (x - 2), mp.mpf(2), mp.mpf(4))
limit_check("H03", lambda x: (3 * x ** 2 + 1) / (x ** 2 - 5), None, mp.mpf(3))
limit_check("H04", lambda x: (1 - mp.cos(x)) / x ** 2, mp.mpf(0), mp.mpf(1) / 2)
lim = mp.limit(lambda x: (1 - mp.cos(x)) / x ** 2, 0)
R["H04"]["ok"] = R["H04"]["ok"] and abs(lim - mp.mpf(1) / 2) < mp.mpf(10) ** -15
R["H04"]["methods"].append("mpmath.limit")
mp.mp.dps = 40

# ---------------------------------------------------------------- I
A = [[Fr(1), Fr(2)], [Fr(3), Fr(4)]]
det = A[0][0] * A[1][1] - A[0][1] * A[1][0]
inv = [[Fr(-2), Fr(1)], [Fr(3, 2), Fr(-1, 2)]]
prod = [[sum(A[i][k] * inv[k][j] for k in range(2)) for j in range(2)] for i in range(2)]
rec("I02", det == -2, ["2x2 formula ad-bc", "Leibniz/cofactor by hand in Fractions"])
rec("I03", prod == [[1, 0], [0, 1]] and all(inv[i][j] * det == [[4, -2], [-3, 1]][i][j] * 1 for i in range(2) for j in range(2)) is False or prod == [[1, 0], [0, 1]],
    ["A*A^-1 == I (Fractions)"])
R["I03"]["ok"] = prod == [[1, 0], [0, 1]] and [[inv[i][j] for j in range(2)] for i in range(2)] == [[Fr(-2), Fr(1)], [Fr(3, 2), Fr(-1, 2)]]

def rref_fr(m):
    m = [row[:] for row in m]
    r = 0
    for c in range(len(m[0])):
        p = next((i for i in range(r, len(m)) if m[i][c] != 0), None)
        if p is None:
            continue
        m[r], m[p] = m[p], m[r]
        m[r] = [v / m[r][c] for v in m[r]]
        for i in range(len(m)):
            if i != r and m[i][c] != 0:
                f = m[i][c]
                m[i] = [a - f * b for a, b in zip(m[i], m[r])]
        r += 1
    return m

exp_rref = [[Fr(1), Fr(2), Fr(3)], [Fr(0), Fr(0), Fr(0)]]
rec("I04", rref_fr([[Fr(1), Fr(2), Fr(3)], [Fr(2), Fr(4), Fr(6)]]) == exp_rref and
    (Fr(1) * Fr(4) - Fr(2) * Fr(2)) == 0, ["independent Gauss-Jordan in Fractions", "rank-1 check: 2x2 minor determinant 0"])

# ---------------------------------------------------------------- J
def ode_check(cid, residual, constants_sets, methods):
    err = 0
    for cs in constants_sets:
        for p in [mp.mpf("-1.3"), mp.mpf("0.4"), mp.mpf("1.7")]:
            err = max(err, abs(residual(p, *cs)))
    rec(cid, err < mp.mpf(10) ** -20, methods, "max residual %s" % mp.nstr(err, 3))

ode_check("J01", lambda x, C: mp.diff(lambda t: C * mp.exp(3 * t), x) - 3 * C * mp.exp(3 * x),
          [(mp.mpf(c),) for c in ("1", "-2.5", "0.3")], ["substitute y=C*exp(3x) into y'-3y=0 (mpmath.diff), 3 constants"])
ode_check("J02", lambda x, C1, C2: mp.diff(lambda t: C1 * mp.cos(t) + C2 * mp.sin(t), x, 2) + (C1 * mp.cos(x) + C2 * mp.sin(x)),
          [(mp.mpf(a), mp.mpf(b)) for a, b in (("1", "0"), ("0", "1"), ("2.2", "-0.7"))],
          ["substitute y=C1 cos x + C2 sin x into y''+y=0 (mpmath.diff order 2)"])
R["J02"]["detail"] += "; Wronskian(cos,sin)=cos^2+sin^2=1 != 0 -> 2 independent solutions (second-order => general solution)"
d = max(abs(mp.diff(lambda x, y=y0: mp.exp(x * y), x0) - y0 * mp.exp(x0 * y0)) for x0 in (mp.mpf("-1"), mp.mpf("0.5"), mp.mpf(2)) for y0 in (mp.mpf("-1.5"), mp.mpf(1), mp.mpf(2)))
rec("J03", d < mp.mpf(10) ** -20, ["mpmath.diff in x with y fixed"], "err %s" % mp.nstr(d, 3))
f2 = lambda x, y: x ** 2 + x * y + y ** 2
g = 0
for x0 in (mp.mpf("-1"), mp.mpf("0.5"), mp.mpf(2)):
    for y0 in (mp.mpf("-1.5"), mp.mpf(1), mp.mpf(2)):
        g = max(g, abs(mp.diff(lambda t: f2(t, y0), x0) - (2 * x0 + y0)), abs(mp.diff(lambda t: f2(x0, t), y0) - (x0 + 2 * y0)))
rec("J04", g < mp.mpf(10) ** -20, ["mpmath.diff of each partial vs expected components"], "err %s" % mp.nstr(g, 3))

# ---------------------------------------------------------------- K
identity("K01", lambda x: 1 / (x * x - 1), lambda x: Fr(1, 2) / (x - 1) - Fr(1, 2) / (x + 1), avoid=(Fr(1), Fr(-1)))
roots = mp.polyroots([1, 0, 0, -1])
expected_k02 = [mp.mpf(1), (-1 + 1j * mp.sqrt(3)) / 2, (-1 - 1j * mp.sqrt(3)) / 2]
rec("K02", all(min(abs(r - e) for e in expected_k02) < mp.mpf(10) ** -30 for r in roots) and
    all(abs(e ** 3 - 1) < mp.mpf(10) ** -30 for e in expected_k02) and len(roots) == 3,
    ["mpmath.polyroots", "complex substitution of each expected root", "|r|=1 and 3 distinct cube roots of unity"])
r3 = mp.findroot(lambda x: mp.cos(x) - x, 0.7)
# independent bisection with plain floats
lo, hi = 0.0, 1.0
for _ in range(200):
    mid = (lo + hi) / 2
    if (math.cos(lo) - lo) * (math.cos(mid) - mid) <= 0:
        hi = mid
    else:
        lo = mid
rec("K03", abs(r3 - mp.mpf("0.7390851332")) < 1e-10 and abs(lo - 0.7390851332) < 1e-10,
    ["mpmath.findroot", "float bisection on [0,1]"], "root %s ; bisection %.12f" % (mp.nstr(r3, 15), lo))
r4 = mp.findroot(lambda x: x ** 5 - x - 1, 1.2)
lo, hi = 1.0, 2.0
for _ in range(200):
    mid = (lo + hi) / 2
    if (lo ** 5 - lo - 1) * (mid ** 5 - mid - 1) <= 0:
        hi = mid
    else:
        lo = mid
real_roots = [z for z in mp.polyroots([1, 0, 0, 0, -1, -1], maxsteps=200, extraprec=100) if abs(z.imag) < 1e-20]
rec("K04", abs(r4 - mp.mpf("1.167303978261")) < 1e-9 and abs(lo - 1.167303978261) < 1e-9 and len(real_roots) == 1,
    ["mpmath.findroot", "float bisection on [1,2]", "mpmath.polyroots: exactly one real root"],
    "root %s ; polyroots real count %d ; spec printed 1.167304015 differs by %s" % (mp.nstr(r4, 15), len(real_roots), mp.nstr(abs(r4 - mp.mpf("1.167304015")), 3)))

# ---------------------------------------------------------------- L
def L01_f(x):
    return (x * x - 1) / (x - 1)

try:
    L01_f(Fr(1)); undef = False
except ZeroDivisionError:
    undef = True
scan_roots = [Fr(k, 8) for k in range(-160, 161) if (Fr(k, 8) != 1 and L01_f(Fr(k, 8)) == 0)]
rec("L01", undef and scan_roots == [Fr(-1)], ["original expression undefined at x=1 (ZeroDivisionError)",
                                               "exhaustive scan of 321 rationals: only root is -1"])
cand = [x for x in [Fr(k, 8) for k in range(-8, 160)] if (x + 1) >= 0 and abs(math.sqrt(x + 1) - (x - 1)) < 1e-12]
rec("L02", cand == [Fr(3)] and math.isclose(math.sqrt(3 + 1), 3 - 1) and not math.isclose(math.sqrt(0 + 1), 0 - 1) and
    -1 - 1 < 0, ["scan of 168 rationals in the domain x>=-1: only x=3 satisfies the ORIGINAL equation",
                 "squaring candidate x=0 is extraneous (sqrt(1)=1 != -1)"],
    "x^2-3x=0 -> {0,3}; 0 rejected on substitution")
rec("L03", all(math.sqrt(x * x) == abs(x) for x in [-3.0, -0.5, 2.0, 7 / 3]), ["float check at negative/positive x"])
und = True
for p in rat_points(6):
    try:
        1 / (p - p); und = False
    except ZeroDivisionError:
        pass
rec("L04", und, ["denominator x-x is identically 0 -> expression undefined for every x (6 points, ZeroDivisionError)"])

# ----------------------------------------------------------------
ids = [c["id"] for c in json.load(open(ROOT / "data" / "cases.json"))["cases"]]
missing = [i for i in ids if i not in R]
out = {"validated": len(R), "all_ok": all(v["ok"] for v in R.values()) and not missing, "missing": missing,
       "failed": [k for k, v in R.items() if not v["ok"]], "checks": dict(sorted(R.items()))}
(ROOT / "results").mkdir(exist_ok=True)
(ROOT / "results" / "reference_validation.json").write_text(json.dumps(out, indent=1, default=str) + "\n")
print("validated", len(R), "cases | all_ok =", out["all_ok"], "| failed:", out["failed"], "| missing:", missing)
sys.exit(0 if out["all_ok"] else 1)
''')
print('wrote', 4, 'files')

In [ ]:
#@title 3d. Benchmark source: evaluation, engineering probes, scoring, reports { display-mode: "form" }
W('bench/evaluate.py', r'''#!/usr/bin/env python3
"""raw replies (results/raw/*.json) + data/cases.json  ->  results/raw_results.json / .csv  (one row per solver per case).

Re-runnable without re-running any solver.  Usage: python -m bench.evaluate [--raw results/raw] [--out results]
"""
import argparse
import csv
import hashlib
import json
import pathlib
import statistics
import subprocess

try:
    from . import derivations as D
    from . import judge as J
except ImportError:
    import derivations as D
    import judge as J

ROOT = pathlib.Path(__file__).resolve().parent.parent
ORDER = ["sympy-direct", "smartsolver", "stepsolver"]


def load_evidence():
    """results/evidence/candidates.json is written by the setup step (commit, url, ref, licence, dependency count, history)."""
    p = ROOT / "results/evidence/candidates.json"
    return json.load(open(p)) if p.exists() else {}


def git_commit(name, evidence):
    return (evidence.get(name) or {}).get("commit")


def fingerprint(reply):
    """Identity of the derivation + answer a manual review was written for (a changed engine output invalidates the review)."""
    keep = {"steps": reply.get("steps"), "answer": reply.get("answer"), "outcome": reply.get("outcome")}
    return hashlib.sha256(json.dumps(keep, sort_keys=True, default=str).encode()).hexdigest()[:16]


def pct(sorted_vals, q):
    if not sorted_vals:
        return None
    k = (len(sorted_vals) - 1) * q
    f, c = int(k), min(int(k) + 1, len(sorted_vals) - 1)
    return sorted_vals[f] + (sorted_vals[c] - sorted_vals[f]) * (k - f)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--raw", default=str(ROOT / "results/raw"))
    ap.add_argument("--out", default=str(ROOT / "results"))
    a = ap.parse_args()
    cases = json.load(open(ROOT / "data/cases.json"))["cases"]
    mr_path = ROOT / "data/manual_reviews.json"
    manual = json.load(open(mr_path)) if mr_path.exists() else {}
    rows = []
    meta = {}
    evidence = load_evidence()
    stale_reviews = []
    for name in ORDER:
        p = pathlib.Path(a.raw) / ("%s.json" % name)
        if not p.exists():
            continue
        raw = json.load(open(p))
        info = raw["info"] or {}
        meta[name] = {"version": info.get("version"), "commit": git_commit(name, evidence), "package": info.get("package"),
                      "python": info.get("python"), "sympy_version": info.get("sympy_version") or info.get("version") if name == "sympy-direct" else info.get("sympy_version"),
                      "cold_start": raw["cold_start"], "maxrss_kb": raw["maxrss_kb"], "timeouts": raw["timeouts"], "crashes": raw["crashes"],
                      "protocol": raw["protocol"], "public_methods": info.get("public_methods"), "operations": info.get("operations")}
        for c in cases:
            rr = raw["rows"].get(c["id"])
            if rr is None:
                rows.append({"solver": name, "case_id": c["id"], "status": "skipped", "skip_reason": "case not selected in this run"})
                continue
            reply = rr["reply"]
            verdict = J.judge(c, reply)
            deriv = D.automated_score(c, reply, verdict)
            review = manual.get("%s|%s" % (name, c["id"]))
            if review is not None and review.get("fingerprint") not in (None, fingerprint(reply)):
                stale_reviews.append("%s|%s" % (name, c["id"]))      # engine output changed since the review: NOT applied
                review = None
            deriv = D.apply_manual(deriv, review)
            warm = sorted(t for t in rr.get("warm_times_ms", []) if t is not None)
            rows.append({
                "solver": name, "solver_version": meta[name]["version"], "solver_commit": meta[name]["commit"],
                "case_id": c["id"], "category": c["category"], "operation": c["operation"], "display_input": c["display_input"],
                "normalised_operation": c["operation"], "problem": c["problem"], "domain": c["domain"], "domain_sensitive": c["domain_sensitive"],
                "adapter_api_call": reply.get("api_call"), "adapter_input": reply.get("input_sent"),
                "adapter_overrides": bool(reply.get("input_overrides_used")), "domain_assumption_passed": reply.get("domain_assumption_passed"),
                "raw_output": reply.get("raw_output"), "raw_answer_canonical": reply.get("answer"), "raw_steps": reply.get("steps"),
                "native_status": reply.get("native_status"), "native_flags": reply.get("native_flags"),
                "restrictions_reported": reply.get("restrictions_reported"),
                "expected_spec": c["spec_expected"], "expected_reference": c["expected"], "actual": verdict["actual"],
                "status": verdict["status"], "status_detail": verdict["sub"], "value_correct": verdict["value_correct"],
                "form_ok": verdict["form_ok"], "form_note": verdict["form_note"], "domain_ok": verdict["domain_ok"],
                "domain_check": verdict["domain_check"], "answer_produced": verdict["answer_produced"], "covered": verdict["covered"],
                "strict_correct": verdict["strict_correct"], "value_only_correct": verdict["value_only_correct"],
                "equivalence_proof": verdict["proof"], "judge_detail": verdict["detail"], "constant_shown": verdict["constant_shown"],
                "derivation": deriv, "unsupported_reason": reply.get("unsupported_reason"), "error": reply.get("error"),
                "time_first_call_ms": reply.get("time_ms"), "wall_ms": rr.get("wall_ms"), "warm_times_ms": warm,
                "warm_median_ms": statistics.median(warm) if warm else None, "repeatable": rr.get("repeatable"),
                "timeout": reply.get("outcome") == "timeout", "manual_review_flag": verdict["manual_review"],
                "manual_review_reasons": verdict["manual_reasons"],
            })
    out = pathlib.Path(a.out)
    out.mkdir(exist_ok=True)
    (out / "raw_results.json").write_text(json.dumps({"meta": meta, "rows": rows, "stale_manual_reviews": stale_reviews,
                                                       "manual_reviews_total": len(manual)}, indent=1, default=str))
    if stale_reviews:
        print("manual reviews NOT applied (engine output changed since review): %d" % len(stale_reviews))
    cols = ["solver", "solver_version", "solver_commit", "case_id", "category", "operation", "display_input", "status", "status_detail",
            "strict_correct", "value_only_correct", "covered", "form_ok", "domain_ok", "expected_spec", "actual", "derivation_available",
            "derivation_score_auto", "derivation_score_final", "time_first_call_ms", "warm_median_ms", "timeout", "error_type",
            "manual_review_flag"]
    with open(out / "raw_results.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=cols)
        w.writeheader()
        for r in rows:
            d = r.get("derivation") or {}
            w.writerow({**{k: r.get(k) for k in cols if k in r}, "derivation_available": d.get("available"),
                        "derivation_score_auto": d.get("automated_total"), "derivation_score_final": d.get("final_total"),
                        "error_type": (r.get("error") or {}).get("type")})
    print("rows:", len(rows), "->", out / "raw_results.json")
    for name in ORDER:
        rs = [r for r in rows if r["solver"] == name]
        if rs:
            from collections import Counter
            print("%-13s" % name, dict(Counter(r["status"] for r in rs)), "strict_correct=%d covered=%d" % (sum(1 for r in rs if r.get("strict_correct")), sum(1 for r in rs if r.get("covered"))))


if __name__ == "__main__":
    main()
''')

W('bench/engineering.py', r'''#!/usr/bin/env python3
"""Engineering / reliability probes (kept separate from mathematical failures).  Each probe goes through the same isolated worker
protocol (own venv, killed on timeout).  Benign canary only: math.factorial(4) -- proves evaluation of input as code without side effects.
Usage: python -m bench.engineering [--out results/engineering_results.json]"""
import argparse, json, pathlib, re, time
try:
    from .runner import Worker, WORKERS, ROOT
except ImportError:
    from runner import Worker, WORKERS, ROOT

CANARY = "x - __import__('math').factorial(4) = 0"


def probes(solver):
    """-> {probe_id: (description, request)}; None = not applicable to this solver's interface."""
    def sym(kind, **k): return {"cmd": "raw", "probe": {"kind": kind, **k}}
    def smart(method, *args): return {"cmd": "raw", "probe": {"kind": "call", "method": method, "args": list(args)}}
    def step(q): return {"cmd": "raw", "probe": {"kind": "query", "query": q}}
    P = {}
    if solver == "sympy-direct":
        P["malformed_syntax"] = sym("parse", text="2 +* 3")
        P["unbalanced_parens"] = sym("parse", text="(x + 1")
        P["empty_input"] = sym("eval", text="")
        P["zero_denominator"] = sym("eval", text="1/0")
        P["missing_variable"] = sym("solve_for", equation="x + 1 = 0", var="z")
        P["two_variable_parametric"] = sym("solve_for", equation="x*y = 6", var="x")
        P["unsupported_operation"] = None
        P["heavy_equation"] = sym("solve_for", equation="x^25 - x - 1 = 0", var="x")
        P["large_polynomial_equation"] = sym("solve_for", equation="(x + 1)^60 = 0", var="x")
        P["untrusted_input_canary"] = sym("canary", text="__import__('math').factorial(4)")
    elif solver == "smartsolver":
        P["malformed_syntax"] = smart("solve_equation", "2 +* 3 = x", "x")
        P["unbalanced_parens"] = smart("solve_equation", "(x + 1 = 2", "x")
        P["empty_input"] = smart("solve_equation", "", "x")
        P["zero_denominator"] = smart("solve_equation", "1/0 = x", "x")
        P["missing_variable"] = smart("solve_equation", "x + 1 = 0", "z")
        P["two_variable_parametric"] = smart("solve_equation", "x*y = 6", "x")
        P["unsupported_operation"] = None
        P["heavy_equation"] = smart("solve_equation", "x**25 - x - 1 = 0", "x")
        P["large_polynomial_equation"] = smart("solve_equation", "(x + 1)**60 = 0", "x")
        P["untrusted_input_canary"] = smart("solve_equation", CANARY, "x")
    else:
        P["malformed_syntax"] = step("solve(2 +* 3 = x, x)")
        P["unbalanced_parens"] = step("solve((x + 1 = 2, x)")
        P["empty_input"] = step("")
        P["zero_denominator"] = step("simplify(1/0)")
        P["missing_variable"] = step("solve(x + 1 = 0, z)")
        P["two_variable_parametric"] = step("solve(x*y = 6, x)")
        P["unsupported_operation"] = step("frobnicate(x)")
        P["heavy_equation"] = step("solve(x^25 - x - 1 = 0, x)")
        P["large_polynomial_equation"] = step("solve((x + 1)^60 = 0, x)")
        P["untrusted_input_canary"] = step("solve(" + CANARY + ", x)")
    return P


def score(pid, solver, r, status):
    """-> (score in {0,0.5,1,None}, note).  None = not applicable."""
    err = r.get("error") or {}
    oc = r.get("outcome")
    txt = json.dumps(r.get("raw_output", "")) if r.get("raw_output") is not None else ""   # full text: matches must not miss a truncated tail
    if status != "ok":
        return 0.0, "worker %s (hang/crash) - needed external kill" % status
    if pid in ("malformed_syntax", "unbalanced_parens", "empty_input"):
        if oc == "answer":
            return 0.0, "returned an answer for malformed input: %s" % txt[:100]
        return (1.0, "library-typed error %s" % err.get("type")) if err.get("contract_exception") else (0.5, "raw Python exception %s (caught by adapter)" % err.get("type"))
    ns_ = r.get("native_status")
    if pid == "zero_denominator":
        if ns_ == "UndefinedResult":
            return 1.0, "explicit UndefinedResult"
        if ns_ == "UnsolvedResult":
            return 0.5, "honest decline (UnsolvedResult), no fabricated value"
        if oc == "error" and re.search(r"zero|division|undefined|zoo|domain", err.get("type", "") + err.get("message", ""), re.I):
            return 1.0, "explicit error: %s" % err.get("type")
        if oc == "answer" and re.search(r"undefined", txt, re.I):
            return 1.0, "explicit undefined result"
        if oc == "answer" and "zoo" in txt:
            return 0.5, "returns 'zoo' (complex infinity) without an undefined/domain message"
        if oc == "answer" and re.search(r"no solution|\[\]|EmptySet|empty", txt, re.I):
            return 0.5, "silently reports no solution"
        return 0.0, "outcome=%s %s" % (oc, (err.get("message") or txt)[:100])
    if pid == "missing_variable":
        if oc == "error":
            return 1.0, "error: %s" % err.get("type")
        if oc == "answer" and not re.search(r"-1", txt):
            return 0.5, "answer without fabricating a root: %s" % txt[:100]
        return 0.0, "fabricated/unclear result: %s" % txt[:100]
    if pid == "two_variable_parametric":
        if oc == "answer" and re.search(r"6\s*/\s*y|6 / y|\{6/y\}|y\*\*\(-1\)", txt.replace(" ", "")) or (oc == "answer" and "6/y" in txt.replace(" ", "")):
            return 1.0, "parametric solution 6/y"
        if oc == "answer" and "6 / y" in txt:
            return 1.0, "parametric solution 6/y"
        return (0.5, "honest error/unsolved: %s" % (err.get("type") or txt[:80])) if oc in ("error", "unsupported") else (0.0, "wrong/absent parametric solution: %s" % txt[:100])
    if pid == "unsupported_operation":
        if ns_ == "ExactResult":
            return 0.0, "unknown call 'frobnicate(x)' echoed back as a Verified ExactResult (silent acceptance of unknown functions)"
        if ns_ == "UnsolvedResult":
            return 0.5, "honest decline (UnsolvedResult)"
        return (1.0, "typed error %s" % err.get("type")) if err.get("contract_exception") else (0.5, "raw exception %s" % err.get("type"))
    if pid == "large_polynomial_equation":
        ok = oc == "answer" and re.search(r"-1\b", txt) is not None
        return (1.0, "correct root -1 in %.0f ms" % r.get("time_ms", -1)) if ok else (0.0, "outcome=%s %s" % (oc, (err.get("message") or txt)[:100]))
    if pid == "heavy_equation":
        return 1.0, "completed in %.0f ms (%s)" % (r.get("time_ms", -1), r.get("native_status") or oc)
    if pid == "untrusted_input_canary":
        executed = bool(re.search(r"\b24\b", txt)) or (oc == "error" and re.search(r"unsupported operand.*str|'str'", err.get("message", "")))
        return (0.0, "INPUT EVALUATED AS PYTHON (math.factorial(4)=24 reached the solver): %s" % (txt[:80] or err.get("message", "")[:80])) if executed else (1.0, "payload rejected: %s" % (err.get("type") or txt[:60]))
    return None, ""


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default=str(ROOT / "results/engineering_results.json"))
    a = ap.parse_args()
    out = {}
    ev_path = ROOT / "results/evidence/candidates.json"
    evidence = json.loads(ev_path.read_text()) if ev_path.exists() else {}
    present = [s for s in WORKERS if (ROOT / "envs" / WORKERS[s][0] / "bin" / "python").exists() and (evidence.get(s, {}).get("installed", True))]
    for s in present:
        res = {}
        w = Worker(s)
        for pid, req in probes(s).items():
            if req is None:
                res[pid] = {"score": None, "note": "not applicable: interface has no operation-name dispatch", "reply": None}
                continue
            timeout = 15 if pid in ("heavy_equation", "large_polynomial_equation") else 20
            t = time.perf_counter()
            r, st = w.call({**req, "req_id": pid}, timeout)
            wall = (time.perf_counter() - t) * 1000
            if st != "ok":
                sc, note = 0.0, "worker %s after %ss - external kill needed" % (st, timeout)
                w.kill(); w = Worker(s); r = {"outcome": st}
            else:
                sc, note = score(pid, s, r, st)
            res[pid] = {"score": sc, "note": note, "wall_ms": wall, "reply": r}
        w.close()
        out[s] = res
    # static evidence: network/process imports in candidate library code (offline claim)
    import subprocess
    ev = {}
    for name in [n for n in present if n != "sympy-direct"]:
        path = evidence.get(name, {}).get("source_dir")
        if not path:
            continue
        g = subprocess.run("grep -rnE '^\\s*(import|from)\\s+(requests|urllib|socket|httpx|http\\.client|aiohttp|subprocess|os\\.system)' %s --include=*.py | grep -v '/tests/' | head -20" % path,
                           shell=True, capture_output=True, text=True).stdout.strip().splitlines()
        ev[name] = g
    out["_static_network_imports"] = ev
    pathlib.Path(a.out).write_text(json.dumps(out, indent=1, default=str))
    for s in present:
        print(s, {k: v["score"] for k, v in out[s].items()})


if __name__ == "__main__":
    main()
''')

W('bench/score.py', r'''#!/usr/bin/env python3
"""results/raw_results.json + engineering_results.json -> results/scores.json   (all rules documented in docs/SCORING_RULES.md)"""
import json
import math
import pathlib
import statistics
from collections import Counter, defaultdict

ROOT = pathlib.Path(__file__).resolve().parent.parent
ALL_SOLVERS = ["sympy-direct", "smartsolver", "stepsolver"]
W = dict(correctness=0.30, derivations=0.30, coverage=0.20, reliability=0.10, integration=0.05, performance=0.05)

# ---- Integration & maintainability rubric (14 points). Every score is computed from evidence collected by the setup step
# (results/evidence/candidates.json) or measured in this run; the two fixed assumptions are labelled as such.
def integration_rubric(solver, ev, override_share):
    ev = ev or {}
    crit = {}
    deps = ev.get("runtime_deps")
    if not ev.get("installed", False):
        crit["env_compat"] = (0, "install failed: %s" % ev.get("install_error"))
        crit["install"] = (0, "install failed")
    else:
        crit["env_compat"] = ((2 if deps <= 4 else 1 if deps <= 30 else 0),
                              "%s third-party packages installed besides the engine (2 if <=4, 1 if <=30, else 0); requires-python %s" % (deps, ev.get("python_requires")))
        crit["install"] = ((1 if ev.get("sympy_pin_fallback") else 2), "single install command succeeded%s" % ("; needed an unpinned-SymPy fallback" if ev.get("sympy_pin_fallback") else ""))
    crit["adaptation"] = ((2 if override_share <= 0.02 else 1 if override_share <= 0.10 else 0),
                          "%.1f%% of cases needed per-case adapter handling (2 if <=2%%, 1 if <=10%%, else 0)" % (100 * override_share))
    if solver == "sympy-direct":
        crit["testability_extensibility"] = (0, "FIXED ASSUMPTION: bare engine has no derivation/step framework to test or extend")
    else:
        n = (ev.get("tests") or {}).get("static_test_functions", 0)
        own = (ev.get("tests") or {}).get("own_suite") or {}
        pts = 2 if n >= 100 else 1 if n >= 1 else 0
        note = "%d test functions found statically (2 if >=100, 1 if >=1)" % n
        if own.get("run"):
            note += "; own suite run: %s passed / %s failed" % (own.get("passed"), own.get("failed"))
            if own.get("failed"):
                pts = max(0, pts - 1)
        crit["testability_extensibility"] = (pts, note)
    cls = ev.get("licence_class", "unknown")
    crit["licence"] = ({"permissive": 2, "unknown": 1, "copyleft": 0}[cls], "licence %s (%s)" % (ev.get("licence"), cls))
    if solver == "sympy-direct":
        crit["maintenance"] = (2, "FIXED ASSUMPTION: mature, widely maintained project (not measurable from a PyPI install)")
    else:
        h = ev.get("history") or {}
        if h.get("age_days", 0) >= 730 and h.get("authors", 0) >= 5:
            m = 2
        elif h.get("commits", 0) >= 5 and h.get("days_since_last_commit", 10 ** 6) <= 365:
            m = 1
        else:
            m = 0
        crit["maintenance"] = (m, "history: %s commits, %s authors, age %s days, last commit %s days ago, %s tags (2 if >=2y and >=5 authors; 1 if >=5 commits and active within 1y)" %
                               (h.get("commits"), h.get("authors"), h.get("age_days"), h.get("days_since_last_commit"), h.get("tags")))
    crit["ui_independent"] = (1, "FIXED ASSUMPTION: used headless as a library in this benchmark")
    crit["no_rewrite"] = (1, "FIXED ASSUMPTION: plain library call; adapter-sized integration")
    return crit


def lat_score(ms):
    """100 at <=10 ms, 0 at >=2000 ms, log-linear in between."""
    if ms is None:
        return 0.0
    if ms <= 10:
        return 100.0
    if ms >= 2000:
        return 0.0
    return 100.0 * math.log10(2000.0 / ms) / math.log10(200.0)


def pctl(vals, q):
    v = sorted(vals)
    k = (len(v) - 1) * q
    f, c = int(k), min(int(k) + 1, len(v) - 1)
    return v[f] + (v[c] - v[f]) * (k - f)


def main():
    d = json.load(open(ROOT / "results/raw_results.json"))
    eng = json.load(open(ROOT / "results/engineering_results.json"))
    rows, meta = d["rows"], d["meta"]
    SOLVERS = [s for s in ALL_SOLVERS if s in meta]
    evidence = json.load(open(ROOT / "results/evidence/candidates.json")) if (ROOT / "results/evidence/candidates.json").exists() else {}
    by = {s: [r for r in rows if r["solver"] == s] for s in SOLVERS}
    n_app = len(by[SOLVERS[0]])
    out = {"applicable_cases": n_app, "weights": W, "solvers": {}, "solver_order": SOLVERS}

    # common set for fair latency comparison: cases where ALL solvers produced an answer
    answered = {s: {r["case_id"] for r in by[s] if r["answer_produced"] and r["warm_times_ms"]} for s in SOLVERS}
    common = sorted(set.intersection(*answered.values()))

    for s in SOLVERS:
        rs = by[s]
        cnt = Counter(r["status"] for r in rs)
        sub = Counter((r["status"], r["status_detail"].split(" ")[0] if r["status"] in ("unsupported", "invalid_answer") else "") for r in rs)
        strict = sum(1 for r in rs if r["strict_correct"])
        value_only = sum(1 for r in rs if r["value_only_correct"])
        value_nofrm = sum(1 for r in rs if r["value_correct"])
        covered = sum(1 for r in rs if r["covered"])
        attempted = [r for r in rs if not (r["status"] == "unsupported" and r["status_detail"] == "no_api")]
        crashes = sum(1 for r in attempted if r["status"] in ("runtime_error", "timeout", "infra_error"))
        dsens = [r for r in rs if r["domain_sensitive"]]
        dsens_ok = sum(1 for r in dsens if r["strict_correct"])
        rep = [r["repeatable"] for r in rs if r["repeatable"] is not None]
        eng_scores = [v["score"] for k, v in eng[s].items() if v["score"] is not None]
        canary = eng[s]["untrusted_input_canary"]["score"]
        R = dict(R1_repeatability=(sum(1 for x in rep if x) / len(rep)) if rep else 0.0,
                 R2_engineering_probes=statistics.mean(eng_scores),
                 R3_stability=1 - crashes / max(1, len(attempted)),
                 R4_domain_edge_cases=dsens_ok / len(dsens),
                 R5_untrusted_input_safety=canary)
        reliability = 100 * statistics.mean(R.values())
        # derivations
        scored = [r for r in rs if r["derivation"].get("scored")]
        avail = [r for r in rs if r["derivation"].get("available")]
        mean_final = statistics.mean(r["derivation"]["final_total"] for r in scored) if scored else 0.0
        mean_auto = statistics.mean(r["derivation"]["automated_total"] for r in scored) if scored else 0.0
        deriv_primary = mean_final * 10
        deriv_penalised = 10 * sum(r["derivation"]["final_total"] for r in scored) / n_app          # unavailable = 0 over ALL applicable cases
        dsens_scored = [r for r in scored if r["domain_sensitive"]]
        crit = defaultdict(list)
        for r in scored:
            for k, v in r["derivation"]["final"].items():
                crit[k].append(v)
        # performance
        warm_by_case = {r["case_id"]: r["warm_median_ms"] for r in rs if r["case_id"] in common}
        all_warm = [t for r in rs if r["case_id"] in common for t in r["warm_times_ms"]]
        med = statistics.median(warm_by_case.values()) if warm_by_case else None
        p95 = pctl(all_warm, 0.95) if all_warm else None
        own_med = statistics.median([r["warm_median_ms"] for r in rs if r["warm_median_ms"] is not None]) if any(r["warm_median_ms"] for r in rs) else None
        own_all = [t for r in rs for t in r["warm_times_ms"]]
        own_p95 = pctl(own_all, 0.95) if own_all else None
        fail_rate = (meta[s]["timeouts"] + meta[s]["crashes"]) / max(1, len(attempted))
        perf = 0.4 * lat_score(med) + 0.4 * lat_score(p95) + 0.2 * 100 * (1 - fail_rate)
        # integration
        override_share = sum(1 for r in rs if r["adapter_overrides"]) / n_app
        integ = integration_rubric(s, evidence.get(s), override_share)
        integ_pts = sum(v[0] for v in integ.values())
        integration = 100 * integ_pts / 14
        comp = dict(correctness=100 * strict / n_app, derivations=deriv_primary, coverage=100 * covered / n_app, reliability=reliability,
                    integration=integration, performance=perf)
        overall = sum(W[k] * comp[k] for k in W)
        # sensitivity variants
        def total(c):
            return sum(W[k] * c[k] for k in W)
        variants = {
            "primary (spec formula)": overall,
            "value-only correctness (no domain-reporting requirement)": total({**comp, "correctness": 100 * value_only / n_app}),
            "value correctness ignoring form rules too": total({**comp, "correctness": 100 * value_nofrm / n_app}),
            "availability-penalised derivations (unavailable=0 over all 72)": total({**comp, "derivations": deriv_penalised}),
            "automated-only derivation scores (no manual overlay)": total({**comp, "derivations": mean_auto * 10}),
            "latency on each solver's own supported cases": total({**comp, "performance": 0.4 * lat_score(own_med) + 0.4 * lat_score(own_p95) + 0.2 * 100 * (1 - fail_rate)}),
        }
        # per-category
        cats = {}
        for c in sorted({r["category"] for r in rs}):
            cr = [r for r in rs if r["category"] == c]
            cats[c] = {"n": len(cr), "strict_correct": sum(1 for r in cr if r["strict_correct"]), "covered": sum(1 for r in cr if r["covered"]),
                       "unsupported": sum(1 for r in cr if r["status"] == "unsupported"), "invalid": sum(1 for r in cr if r["status"] == "invalid_answer"),
                       "errors": sum(1 for r in cr if r["status"] in ("runtime_error", "timeout", "infra_error", "input_incompatible"))}
        # per-operation timing (common set)
        ops = defaultdict(list)
        for r in rs:
            if r["case_id"] in common and r["warm_median_ms"] is not None:
                ops[r["operation"]].append(r["warm_median_ms"])
        cs = meta[s]["cold_start"]
        out["solvers"][s] = {
            "version": meta[s]["version"], "commit": meta[s]["commit"], "status_counts": dict(cnt),
            "defined": n_app, "executed": n_app, "skipped": cnt.get("skipped", 0), "passed(strict_correct)": strict, "failed(invalid+error)": cnt["invalid_answer"] + cnt["runtime_error"] + cnt["timeout"] + cnt["infra_error"] + cnt["input_incompatible"],
            "unsupported": cnt["unsupported"], "unsupported_no_api": sum(1 for r in rs if r["status"] == "unsupported" and r["status_detail"] == "no_api"),
            "unsolved_by_solver": sum(1 for r in rs if r["status"] == "unsupported" and r["status_detail"] != "no_api"),
            "value_only_correct": value_only, "covered": covered, "attempted": len(attempted), "crashes_timeouts": crashes,
            "reliability_metrics": R, "domain_sensitive_cases": len(dsens), "domain_sensitive_strict_correct": dsens_ok,
            "derivation": {"available": len(avail), "available_pct_of_applicable": 100 * len(avail) / n_app, "scored": len(scored),
                           "mean_automated": mean_auto, "mean_final": mean_final, "mean_final_domain_sensitive": statistics.mean(r["derivation"]["final_total"] for r in dsens_scored) if dsens_scored else None,
                           "criterion_means": {k: statistics.mean(v) for k, v in crit.items()}, "manual_reviewed": sum(1 for r in scored if r["derivation"].get("manual")),
                           "distribution": dict(Counter(int(r["derivation"]["final_total"]) for r in scored))},
            "performance": {"cold_start_ready_ms_median": statistics.median(c["spawn_to_ready_ms"] for c in cs) if cs else None,
                            "cold_first_solve_ms_median": statistics.median(c["first_solve_wall_ms"] for c in cs) if cs else None,
                            "warm_median_common_ms": med, "warm_p95_common_ms": p95, "warm_median_own_ms": own_med, "warm_p95_own_ms": own_p95,
                            "common_cases": len(common), "timeouts": meta[s]["timeouts"], "crashes": meta[s]["crashes"], "peak_rss_mb": meta[s]["maxrss_kb"] / 1024,
                            "per_operation_median_ms": {k: statistics.median(v) for k, v in sorted(ops.items())}},
            "integration": {"points": integ_pts, "max": 14, "criteria": {k: {"score": v[0], "evidence": v[1]} for k, v in integ.items()}},
            "components": comp, "overall": overall, "sensitivity": variants, "by_category": cats,
        }
    # ---- hard gates
    gates = {}
    for s in SOLVERS:
        rs = by[s]
        critical = [r for r in rs if r["status"] == "invalid_answer" and r["status_detail"].startswith(("claims no solution", "claims every x", "fabricated_solution", "wrong_", "returned solutions", "returned "))]
        majors = [r for r in rs if r["status"] == "invalid_answer" and r not in critical]
        g2 = []
        for r in rs:
            dd = r["derivation"]
            if not dd.get("scored"):
                continue
            nf = r.get("native_flags") or {}
            if nf.get("steps_mode") == "direct_evaluation":
                g2.append((r["case_id"], "steps_mode=direct_evaluation: worked steps replaced by direct solve"))
            ev = dd["evidence"]
            if ev["invalid_steps"] > 0:
                g2.append((r["case_id"], "automated invalid-step detection: " + ev["correctness"]))
            elif dd.get("manual") and dd["manual"]["adjusted"].get("correctness") is not None and "CONTRADICTION" in dd["manual"]["justification"]:
                g2.append((r["case_id"], "manual: contradiction"))
        g2_ids = sorted({c for c, _ in g2})
        # K01 final step shows the unchanged input (manual finding) counts as inconsistency
        for r in rs:
            dd = r["derivation"]
            if dd.get("manual") and "ORIGINAL expression" in dd["manual"]["justification"] and r["case_id"] not in g2_ids:
                g2_ids.append(r["case_id"]); g2.append((r["case_id"], "manual: final step shows the unchanged input"))
        c3 = eng[s]["untrusted_input_canary"]["score"]
        gates[s] = {
            "G1_critical_correctness": {"status": "FAIL" if critical else "PASS", "critical_cases": [(r["case_id"], r["status_detail"]) for r in critical],
                                         "major_defects(not gating)": [(r["case_id"], r["status_detail"][:70]) for r in majors]},
            "G2_fabricated_or_invalid_derivations": {"status": "FAIL" if len(g2_ids) >= 2 else "CONDITIONAL" if len(g2_ids) == 1 else "PASS", "cases": sorted(g2_ids), "evidence": g2},
            "G3_unsafe_execution": {"status": "PASS" if c3 == 1 else ("CONDITIONAL" if s == "sympy-direct" else "FAIL"), "canary_score": c3,
                                    "note": eng[s]["untrusted_input_canary"]["note"]},
            "G4_licence": {"status": ("PASS" if (evidence.get(s) or {}).get("licence_class") == "permissive" else "CONDITIONAL"), "licence": (evidence.get(s) or {}).get("licence", "unknown")},
        }
        st = [g["status"] for g in gates[s].values()]
        gates[s]["eligible_for_production_recommendation"] = "NO (as-is)" if "FAIL" in st else ("CONDITIONAL" if "CONDITIONAL" in st else "YES")
    out["hard_gates"] = gates
    out["common_latency_cases"] = common
    (ROOT / "results/scores.json").write_text(json.dumps(out, indent=1, default=str))
    for s in SOLVERS:
        o = out["solvers"][s]
        print("%-13s overall=%.1f  comps=%s  gates=%s" % (s, o["overall"], {k: round(v, 1) for k, v in o["components"].items()}, gates[s]["eligible_for_production_recommendation"]))
        print("      variants:", {k[:28]: round(v, 1) for k, v in o["sensitivity"].items()})


if __name__ == "__main__":
    main()
''')

W('bench/compare_runs.py', r'''#!/usr/bin/env python3
"""Reproducibility check: compare two full runs (raw replies and judged verdicts). Timings are expected to differ; everything else must not."""
import json, pathlib, statistics, subprocess, sys
ROOT = pathlib.Path(__file__).resolve().parent.parent
a, b = sys.argv[1], sys.argv[2]
out = pathlib.Path(sys.argv[3]); out.mkdir(exist_ok=True, parents=True)
subprocess.run([sys.executable, str(ROOT / "bench/evaluate.py"), "--raw", b, "--out", str(out)], check=True, capture_output=True)
A = json.load(open(ROOT / "results/raw_results.json"))["rows"]; B = json.load(open(out / "raw_results.json"))["rows"]
Bi = {(r["solver"], r["case_id"]): r for r in B}
diff_hash = diff_status = diff_score = n = 0; details = []
ra = {s: json.load(open(pathlib.Path(a) / f"{s}.json")) for s in ("sympy-direct", "smartsolver", "stepsolver") if (pathlib.Path(a) / f"{s}.json").exists()}
rb = {s: json.load(open(pathlib.Path(b) / f"{s}.json")) for s in ra}
for r in A:
    n += 1
    o = Bi[(r["solver"], r["case_id"])]
    h1, h2 = ra[r["solver"]]["rows"][r["case_id"]].get("hash"), rb[r["solver"]]["rows"][r["case_id"]].get("hash")
    if h1 != h2: diff_hash += 1; details.append((r["solver"], r["case_id"], "raw canonical output differs"))
    if r["status"] != o["status"] or r["strict_correct"] != o["strict_correct"]: diff_status += 1; details.append((r["solver"], r["case_id"], "verdict differs"))
    if (r["derivation"].get("automated_total"), r["derivation"].get("final_total")) != (o["derivation"].get("automated_total"), o["derivation"].get("final_total")): diff_score += 1; details.append((r["solver"], r["case_id"], "derivation score differs"))
res = {"rows_compared": n, "raw_output_differences": diff_hash, "verdict_differences": diff_status, "derivation_score_differences": diff_score, "details": details[:20], "timing_median_ms_run1_vs_run2": {}}
for s in ra:
    m = lambda R: statistics.median([x["warm_median_ms"] for x in R if x["solver"] == s and x["warm_median_ms"] is not None])
    res["timing_median_ms_run1_vs_run2"][s] = [round(m(A), 2), round(m(B), 2)]
(ROOT / "results/reproducibility_check.json").write_text(json.dumps(res, indent=1))
print(json.dumps(res, indent=1))
''')

W('bench/report.py', r'''#!/usr/bin/env python3
"""results/*.json -> results/summary.md, results/failure_analysis.md (and results/changes_vs_baseline.md if data/baseline.json exists).

EVERYTHING in these reports is computed from the preserved results. There is no run-specific narrative: strongest/weakest areas and the
recommendation are produced by documented rules from the measured data, so the reports stay valid for any engine version."""
import json
import pathlib
from collections import defaultdict

ROOT = pathlib.Path(__file__).resolve().parent.parent
RES = ROOT / "results"
NAMES = {"sympy-direct": "SymPy (direct)", "smartsolver": "SmartSolver", "stepsolver": "StepSolver"}
CRIT = ["correctness", "completeness", "explanation", "consistency", "domain", "presentation"]


def f(x, n=1):
    return "n/a" if x is None else ("%.*f" % (n, x))


def table(head, rows):
    return "\n".join(["| " + " | ".join(head) + " |", "|" + "|".join("---" for _ in head) + "|"] + ["| " + " | ".join(str(c) for c in r) + " |" for r in rows])


def cut(x, n):
    return str(x if x is not None else "").replace("\n", " ").replace("|", "/")[:n]


def main():
    d = json.load(open(RES / "raw_results.json"))
    sc = json.load(open(RES / "scores.json"))
    eng = json.load(open(RES / "engineering_results.json"))
    rep = json.load(open(RES / "reproducibility_check.json")) if (RES / "reproducibility_check.json").exists() else None
    ref = json.load(open(RES / "reference_validation.json"))
    evp = RES / "evidence/candidates.json"
    evidence = json.load(open(evp)) if evp.exists() else {}
    envtxt = (RES / "evidence/environment.txt").read_text().strip() if (RES / "evidence/environment.txt").exists() else ""
    cases = {c["id"]: c for c in json.load(open(ROOT / "data/cases.json"))["cases"]}
    cats = json.load(open(ROOT / "data/cases.json"))["categories"]
    rows, meta = d["rows"], d["meta"]
    S3 = sc["solver_order"]
    by = {s: [r for r in rows if r["solver"] == s] for s in S3}
    sv, g = sc["solvers"], sc["hard_gates"]
    pf = {s: sv[s]["performance"] for s in S3}
    n_app = sc["applicable_cases"]
    scored_total = sum(sv[s]["derivation"]["scored"] for s in S3)
    reviewed_total = sum(sv[s]["derivation"]["manual_reviewed"] for s in S3)
    stale = len(d.get("stale_manual_reviews", []))
    L = []
    A = L.append

    A("# Math-solver benchmark: " + " vs ".join(NAMES[s] for s in S3) + "\n")
    A("Generated from `results/raw_results.json`, `scores.json`, `engineering_results.json`. Every number and sentence is computed by code from preserved raw solver output; there is no hand-written, run-specific narrative.\n")
    A("> **Read this first.** (1) Mathematical correctness is decided by a symbolic/numeric judge (SymPy used only as a checker) and every reference answer is independently validated with mpmath/Fraction arithmetic. (2) **Derivation scores are proxies**: an automated rubric plus a manual overlay written by a language model for a specific earlier set of outputs. This run scored %d derivations; %d use that overlay; %d overlay entries were NOT applied because the engine's output changed since they were written (their automated score is used). The overlay needs human audit and never decides mathematical correctness. (3) One reference value in the task specification (K04) is slightly wrong; see section 6. (4) SymPy is evaluated as a **bare engine** (no step layer by design).\n" % (scored_total, reviewed_total, stale))

    # 1
    A("## 1. Environment and versions\n")
    A("```\n" + envtxt + "\n```")
    A(table(["Engine", "Version", "Commit / source", "Python", "SymPy in its env", "Isolated env"],
            [[NAMES[s], meta[s]["version"], (meta[s]["commit"] or (evidence.get(s, {}).get("source") or "PyPI release"))[:44], meta[s]["python"], meta[s].get("sympy_version") or "n/a",
              "`envs/%s`" % {"sympy-direct": "sympy", "smartsolver": "smartsolver", "stepsolver": "stepsolver"}[s]] for s in S3]))
    A("\nEach engine ran in its own virtualenv and worker process from an empty working directory. The judge runs in a separate pinned environment. Timings depend on the hardware above and are only comparable between runs on the same hardware.\n")

    # 2
    A("## 2. Installation and execution status\n")
    tr = []
    for s in S3:
        e = evidence.get(s, {})
        own = (e.get("tests") or {}).get("own_suite") or {}
        tr.append([NAMES[s], "yes" if e.get("installed", True) else "NO: " + cut(e.get("install_error"), 80),
                   "%d/%d attempted" % (sv[s]["executed"], sv[s]["defined"]),
                   ("%s passed / %s failed" % (own.get("passed"), own.get("failed"))) if own.get("run") else "not run (static test count: %s)" % ((e.get("tests") or {}).get("static_test_functions") if (e.get("tests") or {}).get("static_test_functions") is not None else "n/a: bare engine"),
                   e.get("runtime_deps", "n/a"), "%s (%s)" % (e.get("licence"), e.get("licence_class")), e.get("python_requires") or "n/a"])
    A(table(["Engine", "Installed", "Executed", "Own test-suite", "Third-party deps", "Licence", "requires-python"], tr))
    A("\nSkipped cases: %s. A case is only 'unsupported' when the engine has no method for it or declares it unsolved; nothing is substituted.\n" % ", ".join("%s %d" % (NAMES[s], sv[s]["skipped"]) for s in S3))

    # 3
    A("## 3. Results by category (strictly correct / cases)\n")
    A(table(["Category"] + [NAMES[s] for s in S3],
            [[c + " " + nm] + ["%d/%d" % (sv[s]["by_category"][c]["strict_correct"], sv[s]["by_category"][c]["n"]) for s in S3] for c, nm in cats.items()] +
            [["**Total**"] + ["**%d/%d**" % (sv[s]["passed(strict_correct)"], sv[s]["defined"]) for s in S3]]))
    A("\n*Strictly correct* = correct value AND required form AND (where required) the domain restriction reported. Value-only counts: " + ", ".join("%s %d" % (NAMES[s], sv[s]["value_only_correct"]) for s in S3) + ".\n")

    # 4
    A("## 4. Overall correctness and coverage\n")
    A(table(["Metric", *[NAMES[s] for s in S3]], [
        ["Cases defined / executed / skipped", *["%d / %d / %d" % (sv[s]["defined"], sv[s]["executed"], sv[s]["skipped"]) for s in S3]],
        ["Strictly correct (passed)", *[sv[s]["passed(strict_correct)"] for s in S3]],
        ["Invalid answers (produced, wrong)", *[sv[s]["status_counts"].get("invalid_answer", 0) for s in S3]],
        ["Unsupported: no API for the operation", *[sv[s]["unsupported_no_api"] for s in S3]],
        ["Unsupported: engine declared it unsolved", *[sv[s]["unsolved_by_solver"] for s in S3]],
        ["Runtime errors / timeouts / infra errors / input-incompatible", *["%d / %d / %d / %d" % (sv[s]["status_counts"].get("runtime_error", 0), sv[s]["status_counts"].get("timeout", 0), sv[s]["status_counts"].get("infra_error", 0), sv[s]["status_counts"].get("input_incompatible", 0)) for s in S3]],
        ["Coverage (result in the expected form)", *["%d (%.1f%%)" % (sv[s]["covered"], 100 * sv[s]["covered"] / sv[s]["defined"]) for s in S3]],
        ["Correctness (strict) %", *[f(sv[s]["components"]["correctness"]) for s in S3]]]))
    A("\nCoverage counts a result only if it is in the requested form; correctness is reported separately.\n")

    # 5
    A("## 5. Derivation availability and quality\n")
    A(table(["Metric", *[NAMES[s] for s in S3]], [
        ["Derivation available (cases / %d)" % n_app, *["%d (%.0f%%)" % (sv[s]["derivation"]["available"], sv[s]["derivation"]["available_pct_of_applicable"]) for s in S3]],
        ["Mean automated score /10", *[f(sv[s]["derivation"]["mean_automated"], 2) for s in S3]],
        ["Mean final score /10 (after any applied manual overlay)", *[f(sv[s]["derivation"]["mean_final"], 2) for s in S3]],
        ["Mean final, domain-sensitive cases only", *[f(sv[s]["derivation"]["mean_final_domain_sensitive"], 2) for s in S3]],
        ["Derivations with an applied manual review", *[sv[s]["derivation"]["manual_reviewed"] for s in S3]],
        *[["Mean " + k, *[f(sv[s]["derivation"]["criterion_means"].get(k), 2) for s in S3]] for k in CRIT]]))
    opaque = {s: [r["case_id"] for r in by[s] if r["derivation"].get("scored") and r["derivation"]["evidence"]["substantive_steps"] == 0] for s in S3}
    A("\nMax per criterion: correctness 3, completeness 2, explanation 2, consistency 1, domain 1, presentation 1. Opaque-only derivations (no transformation shown): " +
      "; ".join("%s %d%s" % (NAMES[s], len(opaque[s]), (" (" + ", ".join(opaque[s]) + ")") if opaque[s] else "") for s in S3) + ".\n")
    A("Score distribution (final /10 -> number of derivations): " + "; ".join("%s %s" % (NAMES[s], dict(sorted(sv[s]["derivation"]["distribution"].items()))) for s in S3) + ".\n")

    # 6
    A("## 6. Edge-case, domain and engineering behaviour\n")
    A("Engineering probes (1 = clean, 0.5 = acceptable but raw/implicit, 0 = failure, n/a = does not apply):\n")
    pids = list(eng[S3[0]].keys())
    A(table(["Probe", *[NAMES[s] for s in S3]], [[p, *[("n/a" if eng[s][p]["score"] is None else eng[s][p]["score"]) for s in S3]] for p in pids]))
    A("\nProbe outcomes that were not clean (from the raw probe notes):\n")
    for s in S3:
        for p in pids:
            if eng[s][p]["score"] is not None and eng[s][p]["score"] < 1:
                A("- **%s / %s** (%s): %s" % (NAMES[s], p, eng[s][p]["score"], cut(eng[s][p]["note"], 220)))
    A("\n`untrusted_input_canary` sends the benign payload `__import__('math').factorial(4)`; a score of 0 means the engine's own string interface evaluated it as Python. `heavy_equation` is `x^25 - x - 1 = 0` with a 15 s budget; a score of 0 means the engine did not finish and needed an external kill.\n")
    dsn = sv[S3[0]]["domain_sensitive_cases"]
    A("Domain-sensitive cases (%d): strictly correct %s.\n" % (dsn, ", ".join("%s %d" % (NAMES[s], sv[s]["domain_sensitive_strict_correct"]) for s in S3)))
    k4 = [(NAMES[r["solver"]], r["judge_detail"]) for s in S3 for r in by[s] if r["case_id"] == "K04" and r["judge_detail"]]
    A("**Specification discrepancy (K04):** the task prints \"approximately 1.167304015\", but the real root of x^5 - x - 1 is 1.1673039782614187 (mpmath 40 digits, bisection, polyroots agree); the printed value is off by 3.7e-8. The verbatim spec text is kept in `spec_expected`; the verified root is the reference (tolerance 1e-8). Per-engine: " + "; ".join("%s: %s" % (n, cut(t, 90)) for n, t in k4) + ".\n")
    A("Reference validation: %d/%d references independently validated by a second method, all_ok=%s.\n" % (ref["validated"] - len(ref["failed"]), ref["validated"], ref["all_ok"]))

    # 7
    A("## 7. Reliability and performance\n")
    A(table(["Reliability metric (0-1)", *[NAMES[s] for s in S3]], [[k, *[f(sv[s]["reliability_metrics"][k], 3) for s in S3]] for k in sv[S3[0]]["reliability_metrics"]] +
            [["**Reliability score (mean x100)**", *["**%s**" % f(sv[s]["components"]["reliability"]) for s in S3]]]))
    A("\nReliability = 100 x mean(R1 repeatability, R2 engineering probes, R3 = 1 - crashes/attempted, R4 strict-correct rate on domain-sensitive cases, R5 untrusted-input safety as shipped); see `docs/SCORING_RULES.md`.\n")
    A(table(["Performance (ms)", *[NAMES[s] for s in S3]], [
        ["Cold start: process spawn -> ready (median)", *[f(pf[s]["cold_start_ready_ms_median"], 0) for s in S3]],
        ["Cold first solve of C01 (median)", *[f(pf[s]["cold_first_solve_ms_median"], 0) for s in S3]],
        ["Warm median, %d common cases (cache-cleared repeats)" % pf[S3[0]]["common_cases"], *[f(pf[s]["warm_median_common_ms"], 2) for s in S3]],
        ["Warm p95, common cases", *[f(pf[s]["warm_p95_common_ms"], 2) for s in S3]],
        ["Warm median, own supported cases", *[f(pf[s]["warm_median_own_ms"], 2) for s in S3]],
        ["Warm p95, own supported cases", *[f(pf[s]["warm_p95_own_ms"], 2) for s in S3]],
        ["Timeouts / crashes (main pass)", *["%d / %d" % (pf[s]["timeouts"], pf[s]["crashes"]) for s in S3]],
        ["Peak RSS of worker (MB)", *[f(pf[s]["peak_rss_mb"], 0) for s in S3]],
        ["**Performance score**", *["**%s**" % f(sv[s]["components"]["performance"]) for s in S3]]]))
    A("\nLatency score: 100 at <=10 ms, 0 at >=2000 ms, log-linear between; Performance = 0.4 x score(median) + 0.4 x score(p95) + 0.2 x 100 x (1 - failure rate), on the cases ALL engines answered. Cold start is reported, not scored.\n")
    if rep:
        A("Reproducibility (run 2 vs run 1): **%d/%d** rows identical in raw output, **%d** verdict differences, **%d** derivation-score differences. Median warm latency run1 -> run2: %s.\n" %
          (rep["rows_compared"] - rep["raw_output_differences"], rep["rows_compared"], rep["verdict_differences"], rep["derivation_score_differences"],
           "; ".join("%s %s -> %s ms" % (NAMES[k], v[0], v[1]) for k, v in rep["timing_median_ms_run1_vs_run2"].items())))

    # 8
    A("## 8. Weighted scores and hard gates\n")
    A(table(["Component (weight)", *[NAMES[s] for s in S3]], [
        ["Correctness (30%)", *[f(sv[s]["components"]["correctness"]) for s in S3]],
        ["Derivation quality (30%)", *[f(sv[s]["components"]["derivations"]) for s in S3]],
        ["Coverage (20%)", *[f(sv[s]["components"]["coverage"]) for s in S3]],
        ["Reliability (10%)", *[f(sv[s]["components"]["reliability"]) for s in S3]],
        ["Integration/maintainability (5%)", *[f(sv[s]["components"]["integration"]) for s in S3]],
        ["Performance (5%)", *[f(sv[s]["components"]["performance"]) for s in S3]],
        ["**Overall /100**", *["**%s**" % f(sv[s]["overall"]) for s in S3]]]))
    A("\n`Overall = 0.30*Correctness + 0.30*Derivations + 0.20*Coverage + 0.10*Reliability + 0.05*Integration + 0.05*Performance`. The derivation component is the mean score over *available* derivations x 10, as specified; the availability-penalised variant is in the sensitivity table.\n")
    A("**Integration evidence (computed):**\n")
    for s in S3:
        A("- **%s** (%d/14): " % (NAMES[s], sv[s]["integration"]["points"]) + "; ".join("%s=%d (%s)" % (k, v["score"], cut(v["evidence"], 110)) for k, v in sv[s]["integration"]["criteria"].items()))
    A("\n**Sensitivity (ranking stability):**\n")
    vs = list(sv[S3[0]]["sensitivity"].keys())
    A(table(["Variant", *[NAMES[s] for s in S3], "Order"], [[v, *[f(sv[s]["sensitivity"][v]) for s in S3], " > ".join(NAMES[s] for s in sorted(S3, key=lambda s: -sv[s]["sensitivity"][v]))] for v in vs]))
    A("\n**Hard gates** (a weighted score never overrides these):\n")
    A(table(["Gate", *[NAMES[s] for s in S3]], [
        ["G1 critical correctness defect (confidently wrong answer)", *[g[s]["G1_critical_correctness"]["status"] + (" " + ", ".join(c for c, _ in g[s]["G1_critical_correctness"]["critical_cases"]) if g[s]["G1_critical_correctness"]["critical_cases"] else "") for s in S3]],
        ["G2 invalid / self-contradicting derivation (FAIL >=2 cases, CONDITIONAL 1)", *[g[s]["G2_fabricated_or_invalid_derivations"]["status"] + (" " + ", ".join(g[s]["G2_fabricated_or_invalid_derivations"]["cases"]) if g[s]["G2_fabricated_or_invalid_derivations"]["cases"] else "") for s in S3]],
        ["G3 untrusted input executed as code (as shipped)", *[g[s]["G3_unsafe_execution"]["status"] for s in S3]],
        ["G4 licence", *[g[s]["G4_licence"]["status"] + " (" + str(g[s]["G4_licence"]["licence"]) + ")" for s in S3]],
        ["**Eligible for production recommendation**", *["**" + g[s]["eligible_for_production_recommendation"] + "**" for s in S3]]]))
    A("\nGate evidence:\n")
    for s in S3:
        for k in ("G1_critical_correctness", "G2_fabricated_or_invalid_derivations", "G3_unsafe_execution"):
            gg = g[s][k]
            if gg["status"] == "PASS":
                continue
            if k.startswith("G1"):
                ev = "; ".join("%s: %s" % (c, cut(t, 80)) for c, t in gg["critical_cases"])
            elif k.startswith("G2"):
                ev = "; ".join("%s: %s" % (c, cut(t, 120)) for c, t in gg["evidence"][:6])
            else:
                ev = cut(gg["note"], 200)
            A("- **%s %s = %s**: %s" % (NAMES[s], k[:2], gg["status"], ev))
    A("\nG3 is CONDITIONAL for SymPy-direct by definition: only its *string* parser evaluates code, it offers programmatic construction, and this benchmark's allow-list adapter blocks the payload.\n")

    # 9
    A("## 9. Strongest and weakest areas (computed)\n")
    for s in S3:
        o, cs = sv[s], sv[s]["by_category"]
        full = [c for c, v in cs.items() if v["strict_correct"] == v["n"]]
        zero = [c for c, v in cs.items() if v["strict_correct"] == 0]
        partial = sorted([(v["strict_correct"] / v["n"], c) for c, v in cs.items() if 0 < v["strict_correct"] < v["n"]])
        fails = defaultdict(list)
        for r in by[s]:
            if r["status"] == "invalid_answer":
                fails["invalid answer"].append(r["case_id"])
            elif r["status"] == "runtime_error":
                fails["runtime error"].append(r["case_id"])
            elif r["status"] == "unsupported":
                fails["unsupported (%s)" % ("no API" if r["status_detail"] == "no_api" else "declared unsolved")].append(r["case_id"])
            elif r["status"] == "valid" and not r["strict_correct"]:
                fails["right value, domain/form requirement missed"].append(r["case_id"])
        dv = o["derivation"]
        low = [p for p in pids if eng[s][p]["score"] is not None and eng[s][p]["score"] < 1]
        A("**%s** (overall %.1f)\n" % (NAMES[s], o["overall"]))
        A("- Strongest: %s; strict %d/%d, coverage %d/%d, derivations available %d/%d (mean %.2f/10), warm median %.1f ms." % (
            ("all cases correct in categories " + ", ".join(full)) if full else "no category fully correct", o["passed(strict_correct)"], o["defined"], o["covered"], o["defined"],
            dv["available"], o["defined"], dv["mean_final"], pf[s]["warm_median_common_ms"] or 0))
        A("- Weakest: %s%s." % (("no correct answers in categories " + ", ".join(zero)) if zero else "no category entirely wrong",
                                ("; lowest partial categories: " + ", ".join("%s (%.0f%%)" % (c, 100 * p) for p, c in partial[:3])) if partial else ""))
        A("- Not-passed cases by type: " + ("; ".join("%s: %s" % (k, ", ".join(v)) for k, v in fails.items()) if fails else "none") + ".")
        A("- Derivation concerns: %d opaque-only, %d with auto-detected invalid steps, %d with override/untied workings." % (
            len(opaque[s]), sum(1 for r in by[s] if r["derivation"].get("scored") and r["derivation"]["evidence"]["invalid_steps"] > 0),
            sum(1 for r in by[s] if (r.get("native_flags") or {}).get("steps_mode") == "direct_evaluation" or (r.get("native_flags") or {}).get("workings_tied_to_answer") is False)))
        A("- Engineering probes below 1.0: %s; gates not PASS: %s.\n" % (", ".join("%s (%s)" % (p, eng[s][p]["score"]) for p in low) or "none",
                                                                        ", ".join("%s=%s" % (k[:2], v["status"]) for k, v in g[s].items() if k.startswith("G") and v["status"] != "PASS") or "none"))

    # 10
    A("## 10. Recommendation (rule-based, with limits)\n")
    elig = {s: g[s]["eligible_for_production_recommendation"] for s in S3}
    order = sorted(S3, key=lambda s: -sv[s]["overall"])
    A("Decision rules (fixed before any run): (1) an engine with any hard-gate FAIL is not recommended as-is; (2) among the rest, rank by weighted score; (3) scores within 5 points are treated as a tie to be broken by gates, derivation availability and the failures listed above; (4) a gate-failing engine can still be piloted behind a validation wrapper if the blocking defects are listed and fixed first.\n")
    A(table(["Rank", "Engine", "Overall", "Eligible as-is", "Blocking gates"], [[i + 1, NAMES[s], f(sv[s]["overall"]), elig[s], ", ".join("%s=%s" % (k[:2], v["status"]) for k, v in g[s].items() if k.startswith("G") and v["status"] in ("FAIL", "CONDITIONAL")) or "none"] for i, s in enumerate(order)]))
    ok = [s for s in order if elig[s] == "YES"]
    cond = [s for s in order if elig[s] == "CONDITIONAL"]
    top = order[0]
    A("")
    if ok:
        pick = ok[0]
        A("**Recommended as-is:** %s (highest-scoring engine with no failed or conditional gate; overall %.1f)." % (NAMES[pick], sv[pick]["overall"]))
    elif cond:
        pick = cond[0]
        A("**Recommended with conditions:** %s (highest-scoring engine with no FAIL gate; overall %.1f). Conditions: %s." % (NAMES[pick], sv[pick]["overall"],
          "; ".join("%s=%s" % (k[:2], v["status"]) for k, v in g[pick].items() if k.startswith("G") and v["status"] == "CONDITIONAL")))
    else:
        pick = None
        A("**No engine is eligible as-is.**")
    if top != pick:
        gap = (sv[top]["overall"] - sv[pick]["overall"]) if pick else None
        reasons = []
        for k, v in g[top].items():
            if k.startswith("G") and v["status"] in ("FAIL", "CONDITIONAL"):
                ids = v.get("cases") or [c for c, _ in v.get("critical_cases", [])]
                reasons.append("%s = %s%s" % (k[:2], v["status"], (" (" + ", ".join(ids) + ")") if ids else ""))
        A("")
        A("**Highest weighted score:** %s (%.1f)%s. It is not eligible as-is because: %s." % (NAMES[top], sv[top]["overall"], (", %.1f points above the recommended engine" % gap) if pick else "", "; ".join(reasons)))
        A("Piloting it behind a validation wrapper is reasonable only after the listed failures are fixed and this notebook is re-run to confirm the gates pass.")
    A("\nFrom the measured data, the practical split of responsibilities is: use the engine with the best strict correctness/coverage as the *answer* source, and treat an engine's derivation as shippable only where it is available, scored, and free of invalid/contradictory steps (see failure_analysis.md sections 2 and 3). Where derivations are opaque or unavailable, show the verified answer and label the derivation unavailable instead of showing a placeholder.\n")
    A("### Limitations of this benchmark\n")
    A("- 72 cases are one sample of school/early-university mathematics; this does **not** show that any engine supports all of mathematics or handles word problems, images, or free-form phrasing.\n"
      "- Derivation scores are proxy rubrics plus a model-assigned overlay (for earlier outputs only); human audit is required. Completeness uses regex milestones.\n"
      "- Latency is hardware-dependent and a small part of the score (5%); warm latency is measured cache-cleared in a warm process.\n"
      "- K03/K04 (numeric solving) use each engine's nearest documented API (SymPy `nsolve`/`nroots`; engines without a numeric root-finder use their generic equation solver). SmartSolver `limit` uses its documented one-sided default.\n"
      "- StepSolver `ln` is adapted to `log`; the as-specified result is preserved in `native_flags.as_specified_variant`.\n"
      "- Adapters map to the public API names that existed when the benchmark was written; a renamed/removed method is reported as unsupported (SmartSolver) or an infrastructure error (import failure), never silently substituted.\n"
      "- Offline operation was assessed only by static search for network imports in each library core (`engineering_results.json`).\n")
    A("### Artefacts\n")
    A("`results/raw_results.json|csv`, `results/raw/*.json` (verbatim worker replies), `results/scores.json`, `results/engineering_results.json`, `results/reference_validation.json`, `results/reproducibility_check.json`, `results/failure_analysis.md`, `results/changes_vs_baseline.md` (if a baseline is present), `data/cases.json`, `docs/`.\n")

    # 11 baseline
    base_path = ROOT / "data/baseline.json"
    if base_path.exists():
        base = json.load(open(base_path))
        B = []
        B.append("# Changes vs the baseline run\n")
        B.append("Baseline: **%s**.\n" % base["label"])
        B.append(table(["Engine", "Baseline version", "This run", "Baseline overall", "This overall", "Delta"], [
            [NAMES[s], "%s %s" % (base["engines"].get(s, {}).get("version"), (base["engines"].get(s, {}).get("commit") or "")[:8]), "%s %s" % (meta[s]["version"], (meta[s]["commit"] or "")[:8]),
             f(base["overall"].get(s)), f(sv[s]["overall"]), f(sv[s]["overall"] - base["overall"][s]) if s in base["overall"] else "n/a"] for s in S3]))
        B.append("\n" + table(["Component"] + ["%s (base -> now)" % NAMES[s] for s in S3], [[k] + ["%s -> %s" % (f(base["components"].get(s, {}).get(k)), f(sv[s]["components"][k])) for s in S3] for k in sv[S3[0]]["components"]]))
        reg, imp, other = [], [], []
        for r in rows:
            b = base["cases"].get("%s|%s" % (r["solver"], r["case_id"]))
            if not b:
                continue
            if b["strict_correct"] and not r["strict_correct"]:
                reg.append((r["solver"], r["case_id"], b["status"], r["status"], r["status_detail"]))
            elif (not b["strict_correct"]) and r["strict_correct"]:
                imp.append((r["solver"], r["case_id"], b["status"], r["status"]))
            elif b["status"] != r["status"]:
                other.append((r["solver"], r["case_id"], b["status"], r["status"]))
        B.append("\n## Regressions (strictly correct before, not now): %d\n" % len(reg))
        B.append(table(["Engine", "Case", "Before", "Now", "Detail"], [[NAMES[a], b, c, dd, cut(e, 90)] for a, b, c, dd, e in reg]) if reg else "None.")
        B.append("\n## Improvements (not strictly correct before, correct now): %d\n" % len(imp))
        B.append(table(["Engine", "Case", "Before", "Now"], [[NAMES[a], b, c, dd] for a, b, c, dd in imp]) if imp else "None.")
        B.append("\n## Other status changes: %d\n" % len(other))
        B.append(table(["Engine", "Case", "Before", "Now"], [[NAMES[a], b, c, dd] for a, b, c, dd in other]) if other else "None.")
        B.append("\n*Derivation scores that relied on the earlier manual overlay fall back to automated scores when output changed; compare derivation means with that in mind.*\n")
        (RES / "changes_vs_baseline.md").write_text("\n".join(B) + "\n")
        A("## 11. Changes vs the baseline\n")
        A("See `results/changes_vs_baseline.md`: %d regressions, %d improvements, %d other status changes versus *%s*.\n" % (len(reg), len(imp), len(other), base["label"]))
    (RES / "summary.md").write_text("\n".join(L) + "\n")

    # ------------------------------------------------------------------ failure analysis
    F = []
    Z = F.append
    Z("# Failure analysis (generated from this run's data)\n")
    Z("Reproduce any row after setup: `envs/judge/bin/python bench/runner.py --solver <name> --case <ID> --no-timing --out /tmp/x`, then inspect `/tmp/x/<name>.json` (raw reply: `api_call`, `input_sent`, `raw_output`, `steps`, traceback tail). Verdicts: `results/raw_results.json` (`status`, `status_detail`, `judge_detail`).\n")

    def rowline(r):
        c = cases[r["case_id"]]
        return "| %s | %s | `%s` | %s | `%s` | `%s` | %s |" % (r["case_id"], NAMES[r["solver"]], cut(r["display_input"], 44), cut(c["spec_expected"], 40), cut(r["actual"], 60), cut(r.get("adapter_api_call"), 70), cut(r["status_detail"] or r["status"], 80))

    hdr = "| Case | Engine | Input | Expected (spec) | Actual | Call | Verdict |\n|---|---|---|---|---|---|---|"
    Z("## 1. Incorrect answers (and values correct but not strictly correct)\n")
    inc = [r for s in S3 for r in by[s] if r["status"] == "invalid_answer" or (r["status"] == "valid" and not r["strict_correct"])]
    Z(hdr + "\n" + "\n".join(rowline(r) for r in inc) if inc else "None.")
    Z("\nRows with status `valid` but not strictly correct have the right value and miss a declared domain-reporting or form requirement.\n")
    Z("## 2. Missing or invalid intermediate steps\n")
    for s in S3:
        Z("- **%s** opaque/structural-only derivations (%d): %s" % (NAMES[s], len(opaque[s]), ", ".join(opaque[s]) or "none"))
        unavailable = [r["case_id"] for r in by[s] if r["answer_produced"] and not r["derivation"].get("available")]
        Z("  - answered but no derivation at all (%d): %s" % (len(unavailable), (", ".join(unavailable[:40]) + (" ..." if len(unavailable) > 40 else "")) or "none"))
    Z("")
    inv = [(r, r["derivation"]["evidence"]) for s in S3 for r in by[s] if r["derivation"].get("scored") and r["derivation"]["evidence"]["invalid_steps"] > 0]
    Z("**Automatically detected invalid/misrendered steps** (consecutive-step equivalence check; a displayed step that is not equivalent to the one before it):\n")
    Z(table(["Engine", "Case", "Evidence"], [[NAMES[r["solver"]], r["case_id"], cut(e["correctness"], 180)] for r, e in inv]) if inv else "None detected.")
    ov = [r for s in S3 for r in by[s] if (r.get("native_flags") or {}).get("steps_mode") == "direct_evaluation"]
    Z("\n**Derivations replaced by a direct-evaluation override (steps do not produce the answer):** " + (", ".join("%s %s" % (NAMES[r["solver"]], r["case_id"]) for r in ov) or "none") + ".")
    untied = [r for s in S3 for r in by[s] if (r.get("native_flags") or {}).get("workings_tied_to_answer") is False]
    Z("**Workings from a code path separate from the answer:** " + (", ".join("%s %s" % (NAMES[r["solver"]], r["case_id"]) for r in untied) or "none") + ".")
    for s in S3:
        flags = [r for r in by[s] if r["answer_produced"] and (r.get("native_flags") or {}).get("steps_verified") is not None]
        if flags and all(r["native_flags"]["steps_verified"] for r in flags) and any(r["status"] == "invalid_answer" for r in by[s]):
            Z("- **%s** reports `steps_verified=True` on all %d answered cases, including %d that are invalid: the flag carries no information." % (NAMES[s], len(flags), sum(1 for r in by[s] if r["status"] == "invalid_answer")))
    contra = [(r["solver"], r["case_id"], r["derivation"]["manual"]["justification"]) for s in S3 for r in by[s] if r["derivation"].get("manual") and any(w in r["derivation"]["manual"]["justification"] for w in ("CONTRADICTION", "ORIGINAL expression", "Displayed first step"))]
    if contra:
        Z("\n**Manual-review findings (overlay applied; model-assigned, needs audit):**\n")
        for a, b, c in contra:
            Z("- %s %s: %s" % (NAMES[a], b, cut(c, 400)))
    Z("")
    Z("## 3. Domain and extraneous-solution errors\n")
    dom = [r for s in S3 for r in by[s] if r["status"] in ("invalid_answer", "valid") and ((r["domain_ok"] is False) or any(w in (r["status_detail"] or "") for w in ("domain", "real_domain_only", "partial", "claims no solution", "claims every x")))]
    Z(hdr + "\n" + "\n".join(rowline(r) for r in dom) if dom else "None detected.")
    Z("")
    Z("## 4. Unsupported operations\n")
    for s in S3:
        na = defaultdict(list)
        un = []
        for r in by[s]:
            if r["status"] == "unsupported":
                if r["status_detail"] == "no_api":
                    na[r["category"]].append(r["case_id"])
                else:
                    un.append("%s (%s)" % (r["case_id"], cut(r["unsupported_reason"], 70)))
        Z("- **%s** no API: %s" % (NAMES[s], "; ".join("%s: %s" % (k, ", ".join(v)) for k, v in sorted(na.items())) or "none"))
        Z("  - declared unsolved by the engine: %s" % (", ".join(un) or "none"))
    Z("")
    Z("## 5. Parsing / input-format incompatibilities\n")
    pe = [r for s in S3 for r in by[s] if r["status"] == "input_incompatible"]
    Z("Input-incompatible results (engine rejected the adapted syntax): " + (", ".join("%s %s (%s)" % (NAMES[r["solver"]], r["case_id"], cut((r.get("error") or {}).get("message"), 60)) for r in pe) or "none") + ".\n")
    adp = [r for s in S3 for r in by[s] if r["adapter_overrides"]]
    Z("Cases that needed a documented adapter override/normalisation/composition: " + (", ".join("%s %s" % (NAMES[r["solver"]], r["case_id"]) for r in adp) or "none") + ".")
    for r in adp:
        v = (r.get("native_flags") or {}).get("as_specified_variant")
        if v:
            Z("- %s %s as specified (`%s`) -> %s: %s" % (NAMES[r["solver"]], r["case_id"], v["query"], v["result_type"], cut(v["raw_output"], 160)))
    Z("")
    for s in S3:
        for p in ("malformed_syntax", "unbalanced_parens", "empty_input", "unsupported_operation"):
            if eng[s][p]["score"] == 0:
                Z("- **%s / %s**: %s" % (NAMES[s], p, cut(eng[s][p]["note"], 200)))
    Z("")
    Z("## 6. Runtime and dependency failures\n")
    for s in S3:
        e = evidence.get(s, {})
        if not e.get("installed", True):
            Z("- **%s INSTALL FAILED**: %s" % (NAMES[s], cut(e.get("install_error"), 300)))
    rt = [r for s in S3 for r in by[s] if r["status"] in ("runtime_error", "timeout", "infra_error")]
    for r in rt:
        er = r.get("error") or {}
        Z("- **%s %s** %s: `%s: %s`  (call: `%s`)" % (NAMES[r["solver"]], r["case_id"], r["status"], er.get("type"), cut(er.get("message"), 140), cut(r.get("adapter_api_call"), 80)))
    if not rt:
        Z("- No runtime errors, timeouts or infrastructure failures in the main pass.")
    for s in S3:
        for p in ("heavy_equation", "large_polynomial_equation", "untrusted_input_canary", "zero_denominator"):
            if eng[s][p]["score"] is not None and eng[s][p]["score"] == 0:
                Z("- **%s / %s**: %s" % (NAMES[s], p, cut(eng[s][p]["note"], 220)))
    Z("")
    Z("## 7. Cases the evaluator cannot judge reliably (manual-review flags)\n")
    fl = [r for s in S3 for r in by[s] if r["manual_review_flag"]]
    Z(table(["Engine", "Case", "Reason"], [[NAMES[r["solver"]], r["case_id"], cut("; ".join(r["manual_review_reasons"]), 150)] for r in fl]) if fl else "None.")
    Z("\n- Restriction reporting is decided by text matching of reported constraints.\n- K03/K04 use the closest documented API; K04's reference differs from the specification's printed digits (specification typo).\n- Derivation completeness/explanation are regex/word-count proxies; a manual overlay applies only to the exact output it reviewed (%d applied, %d stale and ignored).\n- Expression equivalence is symbolic where possible, otherwise numeric at exact rational sample points (flagged when numeric-only).\n" % (reviewed_total, stale))
    (RES / "failure_analysis.md").write_text("\n".join(F) + "\n")
    print("wrote summary.md, failure_analysis.md" + (", changes_vs_baseline.md" if base_path.exists() else ""))


if __name__ == "__main__":
    main()
''')
print('wrote', 5, 'files')

In [ ]:
#@title 3e. README { display-mode: "form" }
W("README.md", """# Math-solver benchmark (generated by the Colab notebook)

Layout: data/cases.json (72 cases, generated by tools/make_cases.py), data/manual_reviews.json (model-assigned derivation overlay, fingerprinted),
data/baseline.json (original run, for the change report), bench/ (workers, runner, judge, derivations, engineering, score, report), docs/, pins/, results/.
Start with results/summary.md and results/failure_analysis.md (and results/changes_vs_baseline.md).
Re-run: open the notebook, set the engine versions in the configuration cell, Run all.
""")
print("ok")

In [ ]:
#@title 4. Clone, install (isolated environments) and record evidence { display-mode: "form" }
# ---- clone candidates, build isolated environments, record evidence ---------------------------------------------
import tomllib
(ROOT / "results/evidence").mkdir(parents=True, exist_ok=True)
(ROOT / "pins").mkdir(exist_ok=True)

def clone(name, url, ref):
    d = ROOT / "candidates" / name
    if not (d / ".git").exists():
        sh(["git", "clone", "--quiet", url, str(d)])
    sh(["git", "-C", str(d), "fetch", "--quiet", "--tags", "origin"], check=False)
    if ref and ref != "HEAD":
        sh(["git", "-C", str(d), "checkout", "--quiet", ref])
    else:
        sh(["git", "-C", str(d), "checkout", "--quiet", "-"], check=False)
        sh(["git", "-C", str(d), "pull", "--quiet", "--ff-only"], check=False)
    return d

def mkenv(name, packages, pin_note=None):
    env = ROOT / "envs" / name
    if env.exists():
        shutil.rmtree(env)
    sh([UV, "venv", "--quiet", "--python", PYTHON_VERSION, str(env)])
    py = str(env / "bin" / "python")
    t = time.time()
    r = sh([UV, "pip", "install", "--quiet", "--python", py] + packages, check=False)
    return py, r, time.time() - t

def licence_of(d):
    for f in sorted(d.glob("LICENSE*")) + sorted(d.glob("LICENCE*")) + sorted(d.glob("COPYING*")):
        t = f.read_text(errors="ignore")
        for key, name, cls in [(r"MIT License|Permission is hereby granted, free of charge", "MIT", "permissive"), (r"Apache License", "Apache-2.0", "permissive"),
                               (r"BSD|Redistribution and use in source and binary forms", "BSD", "permissive"), (r"\bISC License", "ISC", "permissive"),
                               (r"GNU AFFERO", "AGPL", "copyleft"), (r"GNU LESSER", "LGPL", "copyleft"), (r"GNU GENERAL PUBLIC", "GPL", "copyleft"), (r"Mozilla Public License", "MPL", "copyleft")]:
            if re.search(key, t):
                return name, cls
    return "unknown (no recognised LICENSE file)", "unknown"

def history_of(d):
    def g(*a):
        return sh(["git", "-C", str(d)] + list(a), check=False).stdout.strip()
    try:
        first = int(g("log", "--reverse", "--format=%ct").splitlines()[0]); last = int(g("log", "-1", "--format=%ct"))
        return {"commits": int(g("rev-list", "--count", "HEAD")), "authors": len(set(g("log", "--format=%an").splitlines())), "tags": len(g("tag").split()),
                "age_days": int((time.time() - first) / 86400), "days_since_last_commit": int((time.time() - last) / 86400)}
    except Exception as e:
        return {"error": str(e)}

def deps_count(py):
    r = sh([UV, "pip", "list", "--python", py, "--format", "json"], check=False)
    try:
        return len(json.loads(r.stdout)) - 1
    except Exception:
        return None

def static_tests(d):
    n = 0
    for tdir in list(d.rglob("tests")) + list(d.rglob("test")):
        if tdir.is_dir() and ".git" not in tdir.parts:
            for f in tdir.rglob("*.py"):
                n += len(re.findall(r"^\s*(?:async\s+)?def test_", f.read_text(errors="ignore"), re.M))
    return n

def pyproject_info(d):
    p = d / "pyproject.toml"
    if not p.exists():
        return {}
    try:
        return tomllib.loads(p.read_text())
    except Exception:
        return {}

evidence = {}
SY = [SYMPY_SPEC, "mpmath"]

# judge env (fixed SymPy, used by the judge and the reference validation only)
jpy, r, _ = mkenv("judge", [JUDGE_SYMPY_SPEC, "mpmath"])
assert r.returncode == 0, "judge env failed: " + r.stderr[-500:]

# standalone SymPy
py, r, secs = mkenv("sympy", SY)
if r.returncode == 0:
    md_ = sh([py, "-c", "import importlib.metadata as m; x=m.metadata('sympy'); print((x.get('License-Expression') or x.get('License') or '').strip()); print(x.get('Requires-Python') or '')"]).stdout.splitlines()
    lic = md_[0] if md_ else ""
    cls = "permissive" if re.search(r"BSD|MIT|Apache", lic) else "unknown"
    evidence["sympy-direct"] = {"installed": True, "install_seconds": secs, "runtime_deps": deps_count(py), "python_requires": md_[1] if len(md_) > 1 else None,
                                "licence": lic or "BSD-3-Clause (not in metadata)", "licence_class": cls if lic else "permissive", "source": "PyPI: " + SYMPY_SPEC, "commit": None,
                                "tests": {"static_test_functions": None, "own_suite": {"run": False}}}
else:
    evidence["sympy-direct"] = {"installed": False, "install_error": r.stderr[-600:]}

def candidate(key, name, url, ref):
    try:
        d = clone(name, url, ref)
    except Exception as e:
        evidence[key] = {"installed": False, "install_error": "clone/checkout failed: %s" % e, "source": "%s @ %s" % (url, ref)}
        return
    commit = sh(["git", "-C", str(d), "rev-parse", "HEAD"]).stdout.strip()
    pin = [SYMPY_SPEC, "mpmath"] if PIN_SYMPY_IN_CANDIDATES else []
    py, r, secs = mkenv(key, [str(d)] + pin)
    fallback = False
    if r.returncode != 0 and pin:
        py, r, secs = mkenv(key, [str(d)])
        fallback = r.returncode == 0
    lic, cls = licence_of(d)
    pp = pyproject_info(d)
    proj = pp.get("project", {})
    src = d / "src" if (d / "src").exists() else d
    evidence[key] = {"installed": r.returncode == 0, "install_error": None if r.returncode == 0 else r.stderr[-800:], "sympy_pin_fallback": fallback,
                     "install_seconds": secs, "commit": commit, "url": url, "ref": ref, "source": "%s @ %s" % (url, commit[:12]), "licence": lic, "licence_class": cls,
                     "python_requires": proj.get("requires-python"), "runtime_deps": deps_count(py) if r.returncode == 0 else None, "history": history_of(d),
                     "tests": {"static_test_functions": static_tests(d), "own_suite": {"run": False}}, "source_dir": str(src), "checkout_dir": str(d)}

candidate("smartsolver", "smartsolver", SMARTSOLVER_URL, SMARTSOLVER_REF)
candidate("stepsolver", "stepsolver", STEPSOLVER_URL, STEPSOLVER_REF)

for k in ("sympy", "smartsolver", "stepsolver", "judge"):
    p = ROOT / "envs" / k / "bin" / "python"
    if p.exists():
        (ROOT / "pins" / ("requirements-%s.txt" % k)).write_text(sh([UV, "pip", "freeze", "--python", str(p)], check=False).stdout)

# optional: candidates' own test suites (separate throw-away venvs; never mixed into the benchmark envs)
if RUN_OWN_TEST_SUITES:
    for key in ("smartsolver", "stepsolver"):
        ev = evidence[key]
        if not ev.get("installed"):
            continue
        d = pathlib.Path(ev["checkout_dir"])
        pp = pyproject_info(d)
        extra = ["pytest"]
        for item in (pp.get("dependency-groups", {}) or {}).get("dev", []):
            name = re.split(r"[ (<>=!~\[]", str(item).strip(), 1)[0]
            if name.startswith("pytest") or name.startswith("httpx") or name == "hypothesis":
                extra.append(re.sub(r"[ ()]", "", str(item)))
        py, r, _ = mkenv("dev-" + key, [str(d)] + sorted(set(extra)))
        if r.returncode:
            ev["tests"]["own_suite"] = {"run": True, "passed": 0, "failed": "n/a", "note": "dev env failed: " + r.stderr[-300:]}
            continue
        t = sh([py, "-m", "pytest", "tests", "-q", "-p", "no:cacheprovider", "-o", "addopts="], check=False, timeout=1500, cwd=str(d))
        out = t.stdout + t.stderr
        m_pass = re.search(r"(\d+) passed", out); m_fail = re.search(r"(\d+) (?:failed|error)", out)
        ev["tests"]["own_suite"] = {"run": True, "passed": int(m_pass.group(1)) if m_pass else 0, "failed": int(m_fail.group(1)) if m_fail else 0, "tail": out[-300:]}
        print(key, "own suite:", ev["tests"]["own_suite"]["passed"], "passed /", ev["tests"]["own_suite"]["failed"], "failed")

(ROOT / "results/evidence/candidates.json").write_text(json.dumps(evidence, indent=1))
cpu = ""
try:
    cpu = [l for l in open("/proc/cpuinfo") if l.startswith("model name")][0].split(":", 1)[1].strip()
except Exception:
    pass
env_txt = "date: %s\nos: %s\ncpu: %d core(s), %s\nmem: %s\npython (notebook): %s  uv: %s\nengine python: %s\nexperiment config: SYMPY_SPEC=%s SMARTSOLVER_REF=%s STEPSOLVER_REF=%s JUDGE=%s" % (
    time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), platform.platform(), os.cpu_count(), cpu,
    sh("free -m | awk '/Mem/{print $2\" MB\"}'", check=False).stdout.strip(), platform.python_version(), sh([UV, "--version"]).stdout.strip(), PYTHON_VERSION,
    SYMPY_SPEC, SMARTSOLVER_REF, STEPSOLVER_REF, JUDGE_SYMPY_SPEC)
(ROOT / "results/evidence/environment.txt").write_text(env_txt + "\n")
print(env_txt, "\n")
for k, v in evidence.items():
    print("%-13s installed=%s commit=%s deps=%s licence=%s" % (k, v.get("installed"), (v.get("commit") or "PyPI")[:10], v.get("runtime_deps"), v.get("licence")))
    if not v.get("installed"):
        print("   INSTALL FAILED:", (v.get("install_error") or "")[-300:])

In [ ]:
#@title 5. Dataset (72 cases) and reference validation { display-mode: "form" }
# ---- dataset + independent validation of every reference answer -------------------------------------------------
JPY = str(ROOT / "envs/judge/bin/python")
run_stream([sys.executable, "tools/make_cases.py"])
run_stream([JPY, "bench/reference_validation.py"], check=False)
ref = json.load(open(ROOT / "results/reference_validation.json"))
print("references validated: %d/%d  all_ok=%s  failed=%s" % (ref["validated"] - len(ref["failed"]), ref["validated"], ref["all_ok"], ref["failed"]))

In [ ]:
#@title 6. Run the benchmark (run 1) { display-mode: "form" }
# ---- run 1 (functional pass, cold-start trials, warm repeats); raw replies are preserved verbatim -------------------
WORKER_ENV = {"sympy-direct": "sympy", "smartsolver": "smartsolver", "stepsolver": "stepsolver"}
installed = [s for s, e in WORKER_ENV.items() if json.load(open(ROOT / "results/evidence/candidates.json")).get(s, {}).get("installed")]
print("engines to run:", installed)
if (ROOT / "results/raw").exists():
    shutil.rmtree(ROOT / "results/raw")
run_stream([JPY, "bench/runner.py", "--solver", *installed, "--repeats", str(REPEATS), "--timeout", str(TIMEOUT_S), "--out", "results/raw"])

In [ ]:
#@title 7. Evaluate, engineering probes, scoring { display-mode: "form" }
# ---- judge + derivation rubric -> raw_results; engineering probes; weighted scores and hard gates ------------------
run_stream([JPY, "bench/evaluate.py"])
run_stream([JPY, "bench/engineering.py"])
run_stream([JPY, "bench/score.py"])

In [ ]:
#@title 8. Reproducibility run (run 2) { display-mode: "form" }
# ---- second full run -> reproducibility check ------------------------------------------------------------------
rp = ROOT / "results/reproducibility_check.json"
if rp.exists():
    rp.unlink()
if RUN_TWICE:
    if (ROOT / "results/raw_run2").exists():
        shutil.rmtree(ROOT / "results/raw_run2")
    run_stream([JPY, "bench/runner.py", "--solver", *installed, "--repeats", str(REPEATS), "--timeout", str(TIMEOUT_S), "--out", "results/raw_run2"])
    run_stream([JPY, "bench/compare_runs.py", "results/raw", "results/raw_run2", "results/run2_eval"])
else:
    print("RUN_TWICE is off: skipping the reproducibility check")

In [ ]:
#@title 9. Generate summary.md, failure_analysis.md, changes_vs_baseline.md { display-mode: "form" }
# ---- reports ---------------------------------------------------------------------------------------------------
if not COMPARE_WITH_BASELINE and (ROOT / "data/baseline.json").exists():
    (ROOT / "data/baseline.json").rename(ROOT / "data/baseline.json.off")
run_stream([JPY, "bench/report.py"])
if (ROOT / "data/baseline.json.off").exists():
    (ROOT / "data/baseline.json.off").rename(ROOT / "data/baseline.json")
print("\nwrote:", ", ".join(sorted(p.name for p in (ROOT / "results").glob("*.md"))))

In [ ]:
#@title 10. Show the results { display-mode: "form" }
# ---- show the results ------------------------------------------------------------------------------------------
try:
    from IPython.display import Markdown, display
    show = lambda t: display(Markdown(t))
except Exception:
    show = print
sc = json.load(open(ROOT / "results/scores.json"))
NAMES = {"sympy-direct": "SymPy (direct)", "smartsolver": "SmartSolver", "stepsolver": "StepSolver"}
rows_ = []
for s in sc["solver_order"]:
    o = sc["solvers"][s]; gt = sc["hard_gates"][s]
    rows_.append("| %s | %s | **%.1f** | %d/%d | %d | %.1f | %s |" % (NAMES[s], o["version"], o["overall"], o["passed(strict_correct)"], o["defined"], o["covered"], o["derivation"]["mean_final"], gt["eligible_for_production_recommendation"]))
show("## Headline\n\n| Engine | Version | Overall /100 | Strictly correct | Covered | Mean derivation /10 | Eligible as-is |\n|---|---|---|---|---|---|---|\n" + "\n".join(rows_))
for name in ("changes_vs_baseline.md", "summary.md"):
    p = ROOT / "results" / name
    if p.exists():
        show(p.read_text())

In [ ]:
#@title 11. Download everything as a zip { display-mode: "form" }
# ---- package everything and download -----------------------------------------------------------------------------
import zipfile
out = pathlib.Path("/content") / "math_solver_benchmark_results.zip" if pathlib.Path("/content").exists() else ROOT.parent / "math_solver_benchmark_results.zip"
skip = {"envs", "candidates", "__pycache__", ".git"}
n = 0
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(ROOT.rglob("*")):
        rel = p.relative_to(ROOT)
        if p.is_dir() or any(part in skip for part in rel.parts) or p.suffix == ".pyc":
            continue
        z.write(p, "math_solver_benchmark/" + str(rel)); n += 1
print("zip:", out, "|", n, "files |", round(out.stat().st_size / 1024), "KB")
try:
    from google.colab import files
    files.download(str(out))
except ImportError:
    print("(not running in Colab: copy the zip from the path above)")